# Monte Carlo Milestone Forecasting

**Cleaned forecasting notebook.** Original 106-cell development notebook retained separately. Forecast cutoff: 2026-08-28 (978 goals). 

In [3]:
# Establish the historical state from which future simulation begins

import json
from pathlib import Path

import numpy as np
import pandas as pd


DATA_PATH = Path(
    "../data/engineered/"
    "ronaldo_1000_ml_engineered.csv"
)

ARTIFACTS_DIR = Path(
    "../artifacts"
)


# ---------------------------------------------------------
# Load historical data
# ---------------------------------------------------------

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"],
)

df = (
    df.sort_values("date")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Load locked model/evaluation contracts
# ---------------------------------------------------------

with open(
    ARTIFACTS_DIR / "feature_spec.json",
    "r",
    encoding="utf-8",
) as f:
    feature_spec = json.load(f)


with open(
    ARTIFACTS_DIR / "preprocessing_spec.json",
    "r",
    encoding="utf-8",
) as f:
    preprocessing_spec = json.load(f)


with open(
    ARTIFACTS_DIR / "main_model_spec.json",
    "r",
    encoding="utf-8",
) as f:
    main_model_spec = json.load(f)


with open(
    ARTIFACTS_DIR / "evaluation_spec.json",
    "r",
    encoding="utf-8",
) as f:
    evaluation_spec = json.load(f)


# ---------------------------------------------------------
# Restore feature contracts
# ---------------------------------------------------------

TARGET = feature_spec[
    "primary_target"
]

FULL_FEATURES = feature_spec[
    "full_features"
]

NUMERIC_FEATURES = feature_spec[
    "numeric_features"
]

CATEGORICAL_FEATURES = feature_spec[
    "categorical_features"
]

FEATURE_SETS = preprocessing_spec[
    "feature_sets"
]


XGB_FEATURES = FEATURE_SETS[
    main_model_spec[
        "xgboost"
    ]["feature_set"]
]

CATBOOST_FEATURES = FEATURE_SETS[
    main_model_spec[
        "catboost"
    ]["feature_set"]
]


# ---------------------------------------------------------
# Final historical appearance
# ---------------------------------------------------------

last_match = (
    df.iloc[-1]
)

last_date = pd.Timestamp(
    last_match["date"]
)


# State AFTER the final historical match.
career_appearances_after = (
    int(
        last_match[
            "career_appearances_pre"
        ]
    )
    + 1
)

career_goals_after = (
    int(
        last_match[
            "career_goals_pre"
        ]
    )
    + int(
        last_match[
            "target_goals"
        ]
    )
)

goals_remaining = (
    1000
    - career_goals_after
)


# ---------------------------------------------------------
# Basic current scoring history
# ---------------------------------------------------------

career_goal_history = (
    df["target_goals"]
    .astype(int)
    .tolist()
)

last_3_goals = (
    career_goal_history[-3:]
)

last_5_goals = (
    career_goal_history[-5:]
)

last_10_goals = (
    career_goal_history[-10:]
)

last_20_goals = (
    career_goal_history[-20:]
)


# ---------------------------------------------------------
# Current streak state
# ---------------------------------------------------------

goalless_streak = 0

for goals in reversed(
    career_goal_history
):

    if goals == 0:
        goalless_streak += 1
    else:
        break


scoring_streak = 0

for goals in reversed(
    career_goal_history
):

    if goals > 0:
        scoring_streak += 1
    else:
        break


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert len(df) == 1333

assert (
    df["match_id"].nunique()
    == 1333
)

assert (
    career_appearances_after
    == 1333
)

assert (
    career_goals_after
    == 978
)

assert (
    goals_remaining
    == 22
)

assert (
    sum(
        career_goal_history
    )
    == 978
)

assert (
    evaluation_spec[
        "distribution_evaluation"
    ][
        "simulation_distribution"
    ]
    == "Poisson"
)


# ---------------------------------------------------------
# Report starting state
# ---------------------------------------------------------

print(
    "PHASE 9 — MONTE CARLO MILESTONE FORECAST"
)

print("=" * 85)

print(
    "Historical cutoff:",
    last_date.date()
)

print(
    "Last appearance:",
    last_match["team"],
    "vs",
    last_match["opponent"],
)

print(
    "Competition:",
    last_match["competition"]
)

print(
    "Goals in last appearance:",
    int(
        last_match[
            "target_goals"
        ]
    )
)


print("\nMILESTONE STATE")
print("=" * 55)

print(
    "Career appearances:",
    career_appearances_after
)

print(
    "Career goals:",
    career_goals_after
)

print(
    "Goals remaining:",
    goals_remaining
)


print("\nRECENT SCORING STATE")
print("=" * 55)

print(
    "Last 3 :",
    last_3_goals,
    "→",
    sum(last_3_goals),
    "goals"
)

print(
    "Last 5 :",
    last_5_goals,
    "→",
    sum(last_5_goals),
    "goals"
)

print(
    "Last 10:",
    last_10_goals,
    "→",
    sum(last_10_goals),
    "goals"
)

print(
    "Last 20:",
    last_20_goals,
    "→",
    sum(last_20_goals),
    "goals"
)

print(
    "Current scoring streak:",
    scoring_streak
)

print(
    "Current goalless streak:",
    goalless_streak
)


print("\nLOCKED FORECAST SYSTEM")
print("=" * 55)

print(
    "Model:",
    evaluation_spec[
        "selected_model"
    ]["name"]
)

print(
    "Distribution:",
    evaluation_spec[
        "distribution_evaluation"
    ][
        "simulation_distribution"
    ]
)

print(
    "\nSimulation starting-state checks: PASSED"
)

PHASE 9 — MONTE CARLO MILESTONE FORECAST
Historical cutoff: 2026-08-28
Last appearance: Al-Nassr vs Al Taawon
Competition: Pro League
Goals in last appearance: 1

MILESTONE STATE
Career appearances: 1333
Career goals: 978
Goals remaining: 22

RECENT SCORING STATE
Last 3 : [1, 0, 1] → 2 goals
Last 5 : [1, 0, 1, 0, 1] → 3 goals
Last 10: [0, 0, 0, 2, 0, 1, 0, 1, 0, 1] → 5 goals
Last 20: [1, 0, 1, 0, 1, 0, 1, 0, 0, 2, 0, 0, 0, 2, 0, 1, 0, 1, 0, 1] → 11 goals
Current scoring streak: 1
Current goalless streak: 0

LOCKED FORECAST SYSTEM
Model: 60% XGBoost + 40% CatBoost ensemble
Distribution: Poisson

Simulation starting-state checks: PASSED


## Build the confirmed fixture calendar

In [5]:
# Build confirmed future fixtures through the end of May 2027.
# Historical/model state remains frozen at 2026-08-28 / 978 goals.

future_fixtures = pd.DataFrame(
    [
        # =================================================
        # AL-NASSR — SAUDI PRO LEAGUE
        # =================================================
        ("2026-09-05", "Al-Nassr", "Al Ittihad", "Pro League", "domestic_league", "A"),
        ("2026-09-09", "Al-Nassr", "Abha", "Pro League", "domestic_league", "H"),
        ("2026-09-12", "Al-Nassr", "Al Khaleej", "Pro League", "domestic_league", "A"),

        ("2026-10-09", "Al-Nassr", "Diriyah Club", "Pro League", "domestic_league", "H"),
        ("2026-10-15", "Al-Nassr", "Al Ahli", "Pro League", "domestic_league", "A"),
        ("2026-10-20", "Al-Nassr", "NEOM", "Pro League", "domestic_league", "H"),
        ("2026-10-23", "Al-Nassr", "Al Faisaly", "Pro League", "domestic_league", "A"),
        ("2026-10-29", "Al-Nassr", "Al Kholood", "Pro League", "domestic_league", "H"),

        ("2026-11-05", "Al-Nassr", "Al Hazem", "Pro League", "domestic_league", "A"),
        ("2026-11-20", "Al-Nassr", "Al Qadsiah", "Pro League", "domestic_league", "H"),
        ("2026-11-26", "Al-Nassr", "Al Shabab", "Pro League", "domestic_league", "H"),

        ("2026-12-03", "Al-Nassr", "Al Fayha", "Pro League", "domestic_league", "A"),
        ("2026-12-10", "Al-Nassr", "Al Hilal", "Pro League", "domestic_league", "H"),
        ("2026-12-14", "Al-Nassr", "Al Fateh", "Pro League", "domestic_league", "A"),

        ("2027-02-11", "Al-Nassr", "Al Riyadh", "Pro League", "domestic_league", "H"),
        ("2027-02-18", "Al-Nassr", "Al-Ettifaq", "Pro League", "domestic_league", "H"),
        ("2027-02-25", "Al-Nassr", "Al Taawoun", "Pro League", "domestic_league", "A"),

        ("2027-03-11", "Al-Nassr", "Al Ittihad", "Pro League", "domestic_league", "H"),
        ("2027-03-18", "Al-Nassr", "Abha", "Pro League", "domestic_league", "A"),

        ("2027-04-02", "Al-Nassr", "Al Khaleej", "Pro League", "domestic_league", "H"),
        ("2027-04-08", "Al-Nassr", "Diriyah Club", "Pro League", "domestic_league", "A"),
        ("2027-04-15", "Al-Nassr", "Al Ahli", "Pro League", "domestic_league", "H"),
        ("2027-04-18", "Al-Nassr", "NEOM", "Pro League", "domestic_league", "A"),
        ("2027-04-22", "Al-Nassr", "Al Faisaly", "Pro League", "domestic_league", "H"),
        ("2027-04-29", "Al-Nassr", "Al Kholood", "Pro League", "domestic_league", "A"),

        ("2027-05-06", "Al-Nassr", "Al Hazem", "Pro League", "domestic_league", "H"),
        ("2027-05-12", "Al-Nassr", "Al Qadsiah", "Pro League", "domestic_league", "A"),
        ("2027-05-20", "Al-Nassr", "Al Shabab", "Pro League", "domestic_league", "A"),
        ("2027-05-24", "Al-Nassr", "Al Fayha", "Pro League", "domestic_league", "H"),
        ("2027-05-28", "Al-Nassr", "Al Hilal", "Pro League", "domestic_league", "A"),

        # =================================================
        # AL-NASSR — AFC CHAMPIONS LEAGUE ELITE
        # =================================================
        ("2026-09-15", "Al-Nassr", "Al Ain", "AFC Champions League Elite", "continental_club", "A"),
        ("2026-10-12", "Al-Nassr", "Neftchi", "AFC Champions League Elite", "continental_club", "H"),
        ("2026-10-26", "Al-Nassr", "Al Wasl", "AFC Champions League Elite", "continental_club", "H"),
        ("2026-11-02", "Al-Nassr", "Pakhtakor", "AFC Champions League Elite", "continental_club", "A"),
        ("2026-11-24", "Al-Nassr", "Al Sadd", "AFC Champions League Elite", "continental_club", "H"),
        ("2026-12-08", "Al-Nassr", "Al Gharafa", "AFC Champions League Elite", "continental_club", "A"),
        ("2027-02-08", "Al-Nassr", "Al Shamal", "AFC Champions League Elite", "continental_club", "H"),
        ("2027-02-15", "Al-Nassr", "Shabab Al-Ahli", "AFC Champions League Elite", "continental_club", "A"),

        # =================================================
        # AL-NASSR — KING CUP
        # =================================================
        ("2026-12-01", "Al-Nassr", "Al Kholood", "King Cup", "domestic_or_other_cup", "H"),

        # =================================================
        # PORTUGAL — UEFA NATIONS LEAGUE
        # =================================================
        ("2026-09-24", "Portugal", "Wales", "Nations League", "international", "H"),
        ("2026-09-27", "Portugal", "Norway", "Nations League", "international", "A"),
        ("2026-10-01", "Portugal", "Denmark", "Nations League", "international", "A"),
        ("2026-10-04", "Portugal", "Norway", "Nations League", "international", "H"),
        ("2026-11-14", "Portugal", "Denmark", "Nations League", "international", "H"),
        ("2026-11-17", "Portugal", "Wales", "Nations League", "international", "A"),
    ],
    columns=[
        "date",
        "team",
        "opponent",
        "competition",
        "competition_type",
        "venue",
    ],
)


# ---------------------------------------------------------
# Clean and order
# ---------------------------------------------------------

future_fixtures["date"] = pd.to_datetime(
    future_fixtures["date"]
)

future_fixtures = (
    future_fixtures
    .sort_values("date")
    .reset_index(drop=True)
)

future_fixtures.insert(
    0,
    "fixture_id",
    [
        f"future_{i:03d}"
        for i in range(
            1,
            len(future_fixtures) + 1
        )
    ],
)


# ---------------------------------------------------------
# Forecast metadata
# ---------------------------------------------------------

future_fixtures["forecast_cutoff"] = pd.Timestamp(
    "2026-08-28"
)

future_fixtures["fixture_status"] = "confirmed"

future_fixtures["result_used"] = False


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert (
    future_fixtures["date"].min()
    > pd.Timestamp("2026-08-28")
)

assert (
    future_fixtures["date"]
    .is_monotonic_increasing
)

assert (
    future_fixtures[
        "fixture_id"
    ].is_unique
)

assert (
    future_fixtures[
        [
            "date",
            "team",
            "opponent",
            "competition",
            "competition_type",
            "venue",
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert set(
    future_fixtures["venue"]
) <= {"H", "A", "N"}


# ---------------------------------------------------------
# Audit output
# ---------------------------------------------------------

print(
    "PHASE 9 — EXTENDED FUTURE FIXTURE CALENDAR"
)

print("=" * 100)

print(
    "Forecast cutoff:",
    future_fixtures[
        "forecast_cutoff"
    ].iloc[0].date()
)

print(
    "Confirmed fixtures:",
    len(future_fixtures)
)

print(
    "Calendar:",
    future_fixtures["date"].min().date(),
    "→",
    future_fixtures["date"].max().date(),
)


print("\nFIXTURES BY TEAM")
print("=" * 50)

print(
    future_fixtures[
        "team"
    ].value_counts()
)


print("\nFIXTURES BY COMPETITION")
print("=" * 50)

print(
    future_fixtures[
        "competition"
    ].value_counts()
)


print("\nFIXTURES BY MONTH")
print("=" * 50)

print(
    future_fixtures[
        "date"
    ]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)


print("\nFULL CALENDAR")
print("=" * 100)

print(
    future_fixtures[
        [
            "fixture_id",
            "date",
            "team",
            "opponent",
            "competition",
            "venue",
        ]
    ]
    .to_string(index=False)
)


print(
    "\nExtended future fixture audit: PASSED"
)

PHASE 9 — EXTENDED FUTURE FIXTURE CALENDAR
Forecast cutoff: 2026-08-28
Confirmed fixtures: 45
Calendar: 2026-09-05 → 2027-05-28

FIXTURES BY TEAM
team
Al-Nassr    39
Portugal     6
Name: count, dtype: int64

FIXTURES BY COMPETITION
competition
Pro League                    30
AFC Champions League Elite     8
Nations League                 6
King Cup                       1
Name: count, dtype: int64

FIXTURES BY MONTH
date
2026-09    6
2026-10    9
2026-11    7
2026-12    5
2027-02    5
2027-03    2
2027-04    6
2027-05    5
Freq: M, Name: count, dtype: int64

FULL CALENDAR
fixture_id       date     team       opponent                competition venue
future_001 2026-09-05 Al-Nassr     Al Ittihad                 Pro League     A
future_002 2026-09-09 Al-Nassr           Abha                 Pro League     H
future_003 2026-09-12 Al-Nassr     Al Khaleej                 Pro League     A
future_004 2026-09-15 Al-Nassr         Al Ain AFC Champions League Elite     A
future_005 2026-09-24 Por

## Audit recursive forecast feature dependencies

In [7]:
# Audit which locked-model features can be updated recursively

# ---------------------------------------------------------
# Fixture-known features
# ---------------------------------------------------------

fixture_known_features = {
    "season",
    "team",
    "opponent",
    "competition",
    "competition_type",
    "venue",
    "is_national_team",
    "competition_importance",
    "age_years",
    "days_rest",
    "matches_last_7d",
    "matches_last_14d",
    "matches_last_30d",
    "month",
    "day_of_week",
    "season_progress_pct",
    "is_season_opening_phase",
    "is_season_run_in",
    "is_dec_jan_congestion",
}


# ---------------------------------------------------------
# Stateful features that can be updated from:
# - appearance
# - simulated Ronaldo goals
# - fixture/opponent identity
# ---------------------------------------------------------

ronaldo_recursive_features = {
    "career_appearances_pre",
    "career_goals_pre",
    "career_goals_per_game_pre",

    "goals_per_game_last_5",
    "goals_per_game_last_10",
    "goals_per_game_last_20",
    "goals_ewma_pre",

    "scoring_rate_last_5",
    "scoring_rate_last_10",
    "scoring_rate_last_20",

    "consecutive_scoring_games_pre",
    "goalless_streak_games_pre",
    "days_since_last_goal",

    "season_appearances_pre",
    "season_goals_pre",
    "season_goals_per_game_pre",

    "team_appearances_with_ronaldo_pre",
    "team_goals_by_ronaldo_pre",
    "team_ronaldo_goals_per_game_pre",

    "opponent_prior_meetings_pre",
    "ronaldo_goals_vs_opponent_pre",
    "ronaldo_goals_per_game_vs_opponent_pre",
    "ronaldo_scoring_rate_vs_opponent_pre",
    "ronaldo_goals_last_3_vs_opponent_pre",

    "first_time_opponent_pre",
    "opponent_meetings_log_pre",

    "no_prior_career_goal_pre",
    "first_career_appearance",
}


# ---------------------------------------------------------
# Features requiring future information NOT determined
# by Ronaldo goal count alone
# ---------------------------------------------------------

external_state_features = {
    # Ronaldo assists
    "career_assists_pre",
    "ronaldo_assists_vs_opponent_pre",

    # Team recent results/form
    "team_points_last_5_ronaldo_apps_pre",
    "team_points_last_10_ronaldo_apps_pre",
    "team_win_rate_last_5_ronaldo_apps_pre",
    "team_unbeaten_rate_last_5_ronaldo_apps_pre",

    # Team score history
    "team_goals_for_last_5_scored_matches_pre",
    "team_goals_against_last_5_scored_matches_pre",
    "team_score_coverage_last_5_pre",
    "no_team_score_history_pre",

    # Team-v-opponent H2H results
    "team_h2h_wins_pre",
    "team_h2h_draws_pre",
    "team_h2h_losses_pre",
    "opponent_h2h_points_per_game_pre",
    "opponent_h2h_unbeaten_rate_pre",

    # Team-v-opponent H2H scores
    "opponent_goals_for_per_h2h_scored_match_pre",
    "opponent_goals_against_per_h2h_scored_match_pre",
    "opponent_clean_sheet_rate_h2h_scored_match_pre",
    "opponent_score_data_h2h_matches_pre",
    "no_h2h_score_history_pre",

    # Matchup Elo state
    "team_matchup_elo_pre",
    "opponent_matchup_elo_pre",
    "matchup_elo_diff_pre",
    "opponent_matchup_elo_games_pre",
    "opponent_elo_unseen_pre",
    "opponent_elo_experience_log_pre",
}


# ---------------------------------------------------------
# Audit both locked models
# ---------------------------------------------------------

def audit_feature_dependencies(
    model_name,
    features,
):
    features = set(features)

    known = sorted(
        features
        & fixture_known_features
    )

    recursive = sorted(
        features
        & ronaldo_recursive_features
    )

    external = sorted(
        features
        & external_state_features
    )

    unclassified = sorted(
        features
        - fixture_known_features
        - ronaldo_recursive_features
        - external_state_features
    )

    return {
        "model": model_name,
        "total": len(features),
        "fixture_known": known,
        "ronaldo_recursive": recursive,
        "external_state": external,
        "unclassified": unclassified,
    }


xgb_audit = audit_feature_dependencies(
    "XGBoost",
    XGB_FEATURES,
)

cat_audit = audit_feature_dependencies(
    "CatBoost",
    CATBOOST_FEATURES,
)


# ---------------------------------------------------------
# Print summary
# ---------------------------------------------------------

for audit in [
    xgb_audit,
    cat_audit,
]:

    print(
        "\n",
        audit["model"],
        sep="",
    )

    print("=" * 90)

    print(
        "Total features:",
        audit["total"],
    )

    print(
        "Fixture-known:",
        len(
            audit["fixture_known"]
        ),
    )

    print(
        "Ronaldo-recursive:",
        len(
            audit["ronaldo_recursive"]
        ),
    )

    print(
        "External-state:",
        len(
            audit["external_state"]
        ),
    )

    print(
        "Unclassified:",
        len(
            audit["unclassified"]
        ),
    )

    print(
        "\nEXTERNAL-STATE FEATURES"
    )

    for feature in (
        audit["external_state"]
    ):
        print(
            " -",
            feature,
        )

    print(
        "\nUNCLASSIFIED FEATURES"
    )

    if (
        len(
            audit["unclassified"]
        )
        == 0
    ):
        print(
            " None"
        )
    else:
        for feature in (
            audit["unclassified"]
        ):
            print(
                " -",
                feature,
            )


# ---------------------------------------------------------
# Combined unresolved dependency set
# ---------------------------------------------------------

combined_external = sorted(
    set(
        xgb_audit[
            "external_state"
        ]
    )
    |
    set(
        cat_audit[
            "external_state"
        ]
    )
)

combined_unclassified = sorted(
    set(
        xgb_audit[
            "unclassified"
        ]
    )
    |
    set(
        cat_audit[
            "unclassified"
        ]
    )
)


print(
    "\nCOMBINED FORECAST DEPENDENCY AUDIT"
)

print("=" * 90)

print(
    "External-state features requiring a policy:",
    len(
        combined_external
    ),
)

print(
    "Still unclassified:",
    len(
        combined_unclassified
    ),
)

print(
    "\nRecursive-feature audit: COMPLETE"
)


XGBoost
Total features: 80
Fixture-known: 15
Ronaldo-recursive: 27
External-state: 26
Unclassified: 12

EXTERNAL-STATE FEATURES
 - career_assists_pre
 - matchup_elo_diff_pre
 - no_h2h_score_history_pre
 - no_team_score_history_pre
 - opponent_clean_sheet_rate_h2h_scored_match_pre
 - opponent_elo_experience_log_pre
 - opponent_elo_unseen_pre
 - opponent_goals_against_per_h2h_scored_match_pre
 - opponent_goals_for_per_h2h_scored_match_pre
 - opponent_h2h_points_per_game_pre
 - opponent_h2h_unbeaten_rate_pre
 - opponent_matchup_elo_games_pre
 - opponent_matchup_elo_pre
 - opponent_score_data_h2h_matches_pre
 - ronaldo_assists_vs_opponent_pre
 - team_goals_against_last_5_scored_matches_pre
 - team_goals_for_last_5_scored_matches_pre
 - team_h2h_draws_pre
 - team_h2h_losses_pre
 - team_h2h_wins_pre
 - team_matchup_elo_pre
 - team_points_last_10_ronaldo_apps_pre
 - team_points_last_5_ronaldo_apps_pre
 - team_score_coverage_last_5_pre
 - team_unbeaten_rate_last_5_ronaldo_apps_pre
 - team_win

In [9]:
# Refine recursive/external feature classification to correct the 12 unclassified

# ---------------------------------------------------------
# Additional features recursively determined by
# Ronaldo's simulated appearances and goals
# ---------------------------------------------------------

additional_recursive_features = {
    # Recent scoring
    "goals_last_3",
    "multi_goal_rate_last_10",

    # Competition history
    "competition_appearances_pre",
    "competition_goals_pre",
    "competition_goals_per_game_pre",

    # Venue history
    "venue_appearances_pre",
    "venue_goals_pre",
    "venue_goals_per_game_pre",
}


# ---------------------------------------------------------
# Assist features require an additional future process
# ---------------------------------------------------------

additional_external_features = {
    "career_assists_pre",
    "assists_last_5",
    "assists_last_10",
    "assists_ewma_pre",
    "season_assists_pre",
    "ronaldo_assists_vs_opponent_pre",
}


# Update the classifications
ronaldo_recursive_features = (
    ronaldo_recursive_features
    | additional_recursive_features
)

external_state_features = (
    external_state_features
    | additional_external_features
)


# ---------------------------------------------------------
# Re-run dependency audit
# ---------------------------------------------------------

xgb_audit = audit_feature_dependencies(
    "XGBoost",
    XGB_FEATURES,
)

cat_audit = audit_feature_dependencies(
    "CatBoost",
    CATBOOST_FEATURES,
)


for audit in [
    xgb_audit,
    cat_audit,
]:

    print(
        "\n",
        audit["model"],
        sep="",
    )

    print("=" * 90)

    print(
        "Total features:",
        audit["total"],
    )

    print(
        "Fixture-known:",
        len(
            audit["fixture_known"]
        ),
    )

    print(
        "Ronaldo-recursive:",
        len(
            audit["ronaldo_recursive"]
        ),
    )

    print(
        "External-state:",
        len(
            audit["external_state"]
        ),
    )

    print(
        "Unclassified:",
        len(
            audit["unclassified"]
        ),
    )

    if audit["unclassified"]:
        print(
            "\nUNCLASSIFIED FEATURES"
        )

        for feature in (
            audit["unclassified"]
        ):
            print(
                " -",
                feature,
            )


# ---------------------------------------------------------
# Combined external dependencies
# ---------------------------------------------------------

combined_external = sorted(
    set(
        xgb_audit[
            "external_state"
        ]
    )
    |
    set(
        cat_audit[
            "external_state"
        ]
    )
)

combined_unclassified = sorted(
    set(
        xgb_audit[
            "unclassified"
        ]
    )
    |
    set(
        cat_audit[
            "unclassified"
        ]
    )
)


print(
    "\nTRUE EXTERNAL-STATE DEPENDENCIES"
)

print("=" * 90)

for feature in combined_external:
    print(
        " -",
        feature,
    )


print(
    "\nSUMMARY"
)

print("=" * 90)

print(
    "External-state features:",
    len(
        combined_external
    ),
)

print(
    "Unclassified features:",
    len(
        combined_unclassified
    ),
)


# ---------------------------------------------------------
# Integrity
# ---------------------------------------------------------

assert (
    len(
        combined_unclassified
    )
    == 0
)

for audit in [
    xgb_audit,
    cat_audit,
]:

    assert (
        audit["total"]
        ==
        len(
            audit["fixture_known"]
        )
        +
        len(
            audit["ronaldo_recursive"]
        )
        +
        len(
            audit["external_state"]
        )
    )


print(
    "\nRefined dependency classification: PASSED"
)


XGBoost
Total features: 80
Fixture-known: 15
Ronaldo-recursive: 35
External-state: 30
Unclassified: 0

CatBoost
Total features: 84
Fixture-known: 19
Ronaldo-recursive: 35
External-state: 30
Unclassified: 0

TRUE EXTERNAL-STATE DEPENDENCIES
 - assists_ewma_pre
 - assists_last_10
 - assists_last_5
 - career_assists_pre
 - matchup_elo_diff_pre
 - no_h2h_score_history_pre
 - no_team_score_history_pre
 - opponent_clean_sheet_rate_h2h_scored_match_pre
 - opponent_elo_experience_log_pre
 - opponent_elo_unseen_pre
 - opponent_goals_against_per_h2h_scored_match_pre
 - opponent_goals_for_per_h2h_scored_match_pre
 - opponent_h2h_points_per_game_pre
 - opponent_h2h_unbeaten_rate_pre
 - opponent_matchup_elo_games_pre
 - opponent_matchup_elo_pre
 - opponent_score_data_h2h_matches_pre
 - ronaldo_assists_vs_opponent_pre
 - season_assists_pre
 - team_goals_against_last_5_scored_matches_pre
 - team_goals_for_last_5_scored_matches_pre
 - team_h2h_draws_pre
 - team_h2h_losses_pre
 - team_h2h_wins_pre
 - 

## Audit future-opponent historical coverage

In [11]:
# Audit historical coverage for every future opponent

historical_opponents = set(
    df["opponent"]
    .dropna()
    .astype(str)
)

future_opponents = (
    future_fixtures["opponent"]
    .drop_duplicates()
    .tolist()
)


# ---------------------------------------------------------
# Historical rows for each future opponent
# ---------------------------------------------------------

coverage_rows = []

for opponent in future_opponents:

    opponent_history = (
        df[
            df["opponent"] == opponent
        ]
        .sort_values("date")
    )

    meetings = len(
        opponent_history
    )

    if meetings > 0:

        latest = (
            opponent_history
            .iloc[-1]
        )

        coverage_rows.append(
            {
                "opponent":
                    opponent,

                "historical_meetings":
                    meetings,

                "historical_goals":
                    int(
                        opponent_history[
                            "target_goals"
                        ].sum()
                    ),

                "last_meeting":
                    latest["date"],

                "last_team":
                    latest["team"],

                "last_competition":
                    latest["competition"],

                "has_history":
                    True,
            }
        )

    else:

        coverage_rows.append(
            {
                "opponent":
                    opponent,

                "historical_meetings":
                    0,

                "historical_goals":
                    0,

                "last_meeting":
                    pd.NaT,

                "last_team":
                    None,

                "last_competition":
                    None,

                "has_history":
                    False,
            }
        )


opponent_coverage = pd.DataFrame(
    coverage_rows
)


# ---------------------------------------------------------
# Attach coverage to all 45 fixtures
# ---------------------------------------------------------

fixture_coverage = (
    future_fixtures[
        [
            "fixture_id",
            "date",
            "team",
            "opponent",
            "competition",
            "venue",
        ]
    ]
    .merge(
        opponent_coverage,
        on="opponent",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

unique_seen = int(
    opponent_coverage[
        "has_history"
    ].sum()
)

unique_unseen = int(
    (
        ~opponent_coverage[
            "has_history"
        ]
    ).sum()
)

fixtures_seen = int(
    fixture_coverage[
        "has_history"
    ].sum()
)

fixtures_unseen = int(
    (
        ~fixture_coverage[
            "has_history"
        ]
    ).sum()
)


print(
    "FUTURE-OPPONENT HISTORICAL COVERAGE"
)

print("=" * 100)

print(
    "Unique future opponents:",
    len(
        opponent_coverage
    )
)

print(
    "Previously faced:",
    unique_seen
)

print(
    "Never faced:",
    unique_unseen
)

print(
    "\nFuture fixtures:",
    len(
        fixture_coverage
    )
)

print(
    "Fixtures with opponent history:",
    fixtures_seen
)

print(
    "Fixtures against unseen opponents:",
    fixtures_unseen
)


print(
    "\nOPPONENT COVERAGE"
)

print("=" * 100)

print(
    opponent_coverage
    .sort_values(
        [
            "has_history",
            "historical_meetings",
            "opponent",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .to_string(
        index=False
    )
)


print(
    "\nUNSEEN-OPPONENT FIXTURES"
)

print("=" * 100)

unseen_fixtures = (
    fixture_coverage[
        ~fixture_coverage[
            "has_history"
        ]
    ]
)

if len(
    unseen_fixtures
) == 0:

    print(
        "None"
    )

else:

    print(
        unseen_fixtures[
            [
                "fixture_id",
                "date",
                "team",
                "opponent",
                "competition",
                "venue",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ---------------------------------------------------------
# Integrity
# ---------------------------------------------------------

assert (
    len(
        fixture_coverage
    )
    ==
    len(
        future_fixtures
    )
)

assert (
    fixture_coverage[
        "has_history"
    ]
    .notna()
    .all()
)

assert (
    fixtures_seen
    +
    fixtures_unseen
    ==
    len(
        future_fixtures
    )
)


print(
    "\nFuture-opponent coverage audit: PASSED"
)

FUTURE-OPPONENT HISTORICAL COVERAGE
Unique future opponents: 28
Previously faced: 10
Never faced: 18

Future fixtures: 45
Fixtures with opponent history: 18
Fixtures against unseen opponents: 27

OPPONENT COVERAGE
      opponent  historical_meetings  historical_goals last_meeting last_team           last_competition  has_history
     Al Shabab                    9                 8   2026-05-07  Al-Nassr                 Pro League         True
    Al-Ettifaq                    9                 3   2026-08-25  Al-Nassr                 Pro League         True
       Denmark                    9                 4   2025-03-23  Portugal        UEFA Nations League         True
     Al Riyadh                    6                 6   2026-08-21  Al-Nassr                 Pro League         True
          Abha                    4                 4   2024-04-02  Al-Nassr                 Pro League         True
        Al Ain                    3                 2   2024-11-05  Al-Nassr AFC Cha

In [13]:
# Detect likely opponent-name mismatches

import re
from difflib import SequenceMatcher


def normalize_name(name):
    """
    Lightweight normalization used only for alias discovery.
    Does not modify the dataset yet.
    """
    name = str(name).lower().strip()

    # Standardize punctuation/separators
    name = re.sub(r"[-_.]", " ", name)

    # Remove repeated whitespace
    name = re.sub(r"\s+", " ", name)

    return name


historical_unique = sorted(
    df["opponent"]
    .dropna()
    .astype(str)
    .unique()
)


unseen_future_opponents = (
    opponent_coverage.loc[
        ~opponent_coverage["has_history"],
        "opponent",
    ]
    .tolist()
)


matches = []

for future_name in unseen_future_opponents:

    future_norm = normalize_name(
        future_name
    )

    scored_candidates = []

    for historical_name in historical_unique:

        historical_norm = normalize_name(
            historical_name
        )

        similarity = SequenceMatcher(
            None,
            future_norm,
            historical_norm,
        ).ratio()

        scored_candidates.append(
            (
                historical_name,
                similarity,
            )
        )

    scored_candidates = sorted(
        scored_candidates,
        key=lambda x: x[1],
        reverse=True,
    )[:5]

    for rank, (
        historical_name,
        similarity,
    ) in enumerate(
        scored_candidates,
        start=1,
    ):

        matches.append(
            {
                "future_opponent":
                    future_name,

                "rank":
                    rank,

                "historical_candidate":
                    historical_name,

                "similarity":
                    similarity,
            }
        )


alias_candidates = pd.DataFrame(
    matches
)


print(
    "LIKELY FUTURE/HISTORICAL OPPONENT ALIASES"
)

print("=" * 110)

for future_name in unseen_future_opponents:

    print(
        f"\nFuture opponent: {future_name}"
    )

    print("-" * 110)

    subset = (
        alias_candidates[
            alias_candidates[
                "future_opponent"
            ]
            == future_name
        ]
    )

    print(
        subset[
            [
                "rank",
                "historical_candidate",
                "similarity",
            ]
        ]
        .to_string(
            index=False,
            formatters={
                "similarity":
                    lambda x: f"{x:.3f}"
            },
        )
    )


# ---------------------------------------------------------
# Also search historical names containing
# common Saudi/AFC club fragments
# ---------------------------------------------------------

print(
    "\n\nHISTORICAL OPPONENTS CONTAINING 'AL'"
)

print("=" * 110)

historical_al_names = [
    name
    for name in historical_unique
    if "al" in normalize_name(name).split()
]

print(
    "\n".join(
        historical_al_names
    )
)

LIKELY FUTURE/HISTORICAL OPPONENT ALIASES

Future opponent: Al Ittihad
--------------------------------------------------------------------------------------------------------------
 rank historical_candidate similarity
    1        Al-Ittihad FC      0.870
    2           Al-Ettifaq      0.700
    3            Al Riyadh      0.632
    4              Al Taee      0.588
    5              Al-Hazm      0.588

Future opponent: Al Khaleej
--------------------------------------------------------------------------------------------------------------
 rank historical_candidate similarity
    1    Al Khaleej Saihat      0.741
    2              Al Taee      0.706
    3             Al-Hilal      0.667
    4           Al Kholood      0.600
    5              Al-Hazm      0.588

Future opponent: Diriyah Club
--------------------------------------------------------------------------------------------------------------
 rank historical_candidate similarity
    1      Al-Qadisiyah FC      0.593
    

In [15]:
# Canonicalize future opponent names using verified historical aliases

future_opponent_alias_map = {
    "Al Ittihad": "Al-Ittihad FC",
    "Al Khaleej": "Al Khaleej Saihat",
    "Al Ahli": "Al-Ahli Jeddah",
    "Al Wasl": "Al-Wasl FC",
    "Al Hazem": "Al-Hazm",
    "Al Qadsiah": "Al-Qadisiyah FC",
    "Al Fayha": "Al-Fayha",
    "Al Gharafa": "Al-Gharafa",
    "Al Hilal": "Al-Hilal",
    "Al Fateh": "Al-Fateh",
    "Shabab Al-Ahli": "Shabab Al Ahli Dubai",
    "Al Taawoun": "Al Taawon",
}


# ---------------------------------------------------------
# Preserve display name, but use canonical opponent
# for historical/state lookups
# ---------------------------------------------------------

future_fixtures = future_fixtures.copy()

future_fixtures["opponent_display"] = (
    future_fixtures["opponent"]
)

future_fixtures["opponent"] = (
    future_fixtures["opponent"]
    .replace(
        future_opponent_alias_map
    )
)


# ---------------------------------------------------------
# Rebuild opponent coverage after canonicalization
# ---------------------------------------------------------

future_opponents = (
    future_fixtures["opponent"]
    .drop_duplicates()
    .tolist()
)

coverage_rows = []

for opponent in future_opponents:

    opponent_history = (
        df[
            df["opponent"] == opponent
        ]
        .sort_values("date")
    )

    meetings = len(
        opponent_history
    )

    if meetings > 0:

        latest = (
            opponent_history
            .iloc[-1]
        )

        coverage_rows.append(
            {
                "opponent":
                    opponent,

                "historical_meetings":
                    meetings,

                "historical_goals":
                    int(
                        opponent_history[
                            "target_goals"
                        ].sum()
                    ),

                "last_meeting":
                    latest["date"],

                "last_team":
                    latest["team"],

                "last_competition":
                    latest["competition"],

                "has_history":
                    True,
            }
        )

    else:

        coverage_rows.append(
            {
                "opponent":
                    opponent,

                "historical_meetings":
                    0,

                "historical_goals":
                    0,

                "last_meeting":
                    pd.NaT,

                "last_team":
                    None,

                "last_competition":
                    None,

                "has_history":
                    False,
            }
        )


opponent_coverage = pd.DataFrame(
    coverage_rows
)


fixture_coverage = (
    future_fixtures[
        [
            "fixture_id",
            "date",
            "team",
            "opponent_display",
            "opponent",
            "competition",
            "venue",
        ]
    ]
    .merge(
        opponent_coverage,
        on="opponent",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

unique_seen = int(
    opponent_coverage[
        "has_history"
    ].sum()
)

unique_unseen = int(
    (
        ~opponent_coverage[
            "has_history"
        ]
    ).sum()
)

fixtures_seen = int(
    fixture_coverage[
        "has_history"
    ].sum()
)

fixtures_unseen = int(
    (
        ~fixture_coverage[
            "has_history"
        ]
    ).sum()
)


print(
    "CANONICALIZED FUTURE-OPPONENT COVERAGE"
)

print("=" * 105)

print(
    "Unique future opponents:",
    len(opponent_coverage)
)

print(
    "Previously faced:",
    unique_seen
)

print(
    "Never faced:",
    unique_unseen
)

print(
    "\nFuture fixtures:",
    len(fixture_coverage)
)

print(
    "Fixtures with opponent history:",
    fixtures_seen
)

print(
    "Fixtures against unseen opponents:",
    fixtures_unseen
)


print(
    "\nALIASES APPLIED"
)

print("=" * 105)

alias_report = (
    future_fixtures[
        [
            "opponent_display",
            "opponent",
        ]
    ]
    .drop_duplicates()
)

alias_report = (
    alias_report[
        alias_report[
            "opponent_display"
        ]
        !=
        alias_report[
            "opponent"
        ]
    ]
)

print(
    alias_report
    .to_string(
        index=False
    )
)


print(
    "\nUPDATED OPPONENT COVERAGE"
)

print("=" * 105)

print(
    opponent_coverage
    .sort_values(
        [
            "has_history",
            "historical_meetings",
            "opponent",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .to_string(
        index=False
    )
)


print(
    "\nREMAINING GENUINELY UNSEEN OPPONENTS"
)

print("=" * 105)

remaining_unseen = (
    opponent_coverage[
        ~opponent_coverage[
            "has_history"
        ]
    ]
)

if len(
    remaining_unseen
) == 0:

    print(
        "None"
    )

else:

    print(
        remaining_unseen[
            [
                "opponent",
                "historical_meetings",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ---------------------------------------------------------
# Integrity
# ---------------------------------------------------------

assert (
    len(
        fixture_coverage
    )
    ==
    45
)

assert (
    fixture_coverage[
        "has_history"
    ]
    .notna()
    .all()
)

assert (
    fixtures_seen
    +
    fixtures_unseen
    ==
    45
)


print(
    "\nCanonical opponent coverage audit: PASSED"
)

CANONICALIZED FUTURE-OPPONENT COVERAGE
Unique future opponents: 28
Previously faced: 22
Never faced: 6

Future fixtures: 45
Fixtures with opponent history: 37
Fixtures against unseen opponents: 8

ALIASES APPLIED
opponent_display             opponent
      Al Ittihad        Al-Ittihad FC
      Al Khaleej    Al Khaleej Saihat
         Al Ahli       Al-Ahli Jeddah
         Al Wasl           Al-Wasl FC
        Al Hazem              Al-Hazm
      Al Qadsiah      Al-Qadisiyah FC
        Al Fayha             Al-Fayha
      Al Gharafa           Al-Gharafa
        Al Hilal             Al-Hilal
        Al Fateh             Al-Fateh
  Shabab Al-Ahli Shabab Al Ahli Dubai
      Al Taawoun            Al Taawon

UPDATED OPPONENT COVERAGE
            opponent  historical_meetings  historical_goals last_meeting last_team           last_competition  has_history
            Al-Hilal                   11                 6   2026-05-12  Al-Nassr                 Pro League         True
           Al Shabab

In [17]:
# Learn the historical first-opponent fallback convention
# from the engineered training data itself.

# ---------------------------------------------------------
# Opponent-specific external features
# ---------------------------------------------------------

opponent_external_features = [
    "team_h2h_wins_pre",
    "team_h2h_draws_pre",
    "team_h2h_losses_pre",

    "opponent_h2h_points_per_game_pre",
    "opponent_h2h_unbeaten_rate_pre",

    "opponent_goals_for_per_h2h_scored_match_pre",
    "opponent_goals_against_per_h2h_scored_match_pre",
    "opponent_clean_sheet_rate_h2h_scored_match_pre",
    "opponent_score_data_h2h_matches_pre",

    "no_h2h_score_history_pre",

    "team_matchup_elo_pre",
    "opponent_matchup_elo_pre",
    "matchup_elo_diff_pre",
    "opponent_matchup_elo_games_pre",

    "opponent_elo_unseen_pre",
    "opponent_elo_experience_log_pre",
]


# ---------------------------------------------------------
# Genuine first meetings in the historical data
# ---------------------------------------------------------

first_meetings = (
    df[
        df["opponent_prior_meetings_pre"] == 0
    ]
    .copy()
)

print(
    "HISTORICAL FIRST-MEETING AUDIT"
)

print("=" * 100)

print(
    "First-meeting rows:",
    len(first_meetings)
)

print(
    "Unique opponents represented:",
    first_meetings["opponent"].nunique()
)


# ---------------------------------------------------------
# Inspect H2H defaults
# ---------------------------------------------------------

print(
    "\nH2H FIRST-MEETING DEFAULTS"
)

print("=" * 100)

h2h_features = [
    "team_h2h_wins_pre",
    "team_h2h_draws_pre",
    "team_h2h_losses_pre",
    "opponent_h2h_points_per_game_pre",
    "opponent_h2h_unbeaten_rate_pre",
    "opponent_goals_for_per_h2h_scored_match_pre",
    "opponent_goals_against_per_h2h_scored_match_pre",
    "opponent_clean_sheet_rate_h2h_scored_match_pre",
    "opponent_score_data_h2h_matches_pre",
    "no_h2h_score_history_pre",
]

for feature in h2h_features:

    values = (
        first_meetings[feature]
        .value_counts(
            dropna=False
        )
        .sort_index()
    )

    print(
        f"\n{feature}"
    )

    print(
        values.to_string()
    )


# ---------------------------------------------------------
# Inspect Elo behavior on first meetings
# ---------------------------------------------------------

print(
    "\nFIRST-MEETING ELO SUMMARY"
)

print("=" * 100)

elo_features = [
    "team_matchup_elo_pre",
    "opponent_matchup_elo_pre",
    "matchup_elo_diff_pre",
    "opponent_matchup_elo_games_pre",
    "opponent_elo_unseen_pre",
    "opponent_elo_experience_log_pre",
]

print(
    first_meetings[
        elo_features
    ]
    .describe()
    .T[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
    .round(4)
    .to_string()
)


# ---------------------------------------------------------
# How often does a first opponent truly have no Elo history?
# ---------------------------------------------------------

elo_unseen_counts = (
    first_meetings[
        "opponent_elo_unseen_pre"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nFIRST-MEETING OPPONENT ELO STATUS"
)

print("=" * 100)

print(
    elo_unseen_counts.to_string()
)


# ---------------------------------------------------------
# Show exceptions where first Ronaldo meeting did NOT
# mean default opponent Elo.
# ---------------------------------------------------------

elo_exceptions = (
    first_meetings[
        first_meetings[
            "opponent_matchup_elo_games_pre"
        ] > 0
    ]
    [
        [
            "date",
            "team",
            "opponent",
            "competition",
            "team_matchup_elo_pre",
            "opponent_matchup_elo_pre",
            "opponent_matchup_elo_games_pre",
        ]
    ]
)

print(
    "\nFIRST-MEETING ELO EXCEPTIONS"
)

print("=" * 100)

if len(
    elo_exceptions
) == 0:

    print(
        "None"
    )

else:

    print(
        elo_exceptions
        .to_string(
            index=False
        )
    )


# ---------------------------------------------------------
# Team-specific latest historical state at cutoff
# ---------------------------------------------------------

print(
    "\nLATEST HISTORICAL TEAM STATES AT FORECAST CUTOFF"
)

print("=" * 100)

for team_name in [
    "Al-Nassr",
    "Portugal",
]:

    team_history = (
        df[
            df["team"] == team_name
        ]
        .sort_values("date")
    )

    latest = (
        team_history.iloc[-1]
    )

    print(
        f"\n{team_name}"
    )

    print(
        "Last historical appearance:",
        latest["date"].date(),
        "vs",
        latest["opponent"],
    )

    print(
        "Pre-match team Elo:",
        round(
            float(
                latest[
                    "team_matchup_elo_pre"
                ]
            ),
            4,
        ),
    )

    print(
        "Result:",
        latest["result"],
    )


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert len(
    first_meetings
) == 283

assert (
    first_meetings[
        "opponent_prior_meetings_pre"
    ]
    .eq(0)
    .all()
)

assert (
    first_meetings[
        "opponent_score_data_h2h_matches_pre"
    ]
    .eq(0)
    .all()
)

assert (
    first_meetings[
        "no_h2h_score_history_pre"
    ]
    .eq(1)
    .all()
)


print(
    "\nFirst-meeting fallback audit: PASSED"
)

HISTORICAL FIRST-MEETING AUDIT
First-meeting rows: 283
Unique opponents represented: 283

H2H FIRST-MEETING DEFAULTS

team_h2h_wins_pre
team_h2h_wins_pre
0    283

team_h2h_draws_pre
team_h2h_draws_pre
0    283

team_h2h_losses_pre
team_h2h_losses_pre
0    283

opponent_h2h_points_per_game_pre
opponent_h2h_points_per_game_pre
0.0    283

opponent_h2h_unbeaten_rate_pre
opponent_h2h_unbeaten_rate_pre
0.0    283

opponent_goals_for_per_h2h_scored_match_pre
opponent_goals_for_per_h2h_scored_match_pre
0.0    283

opponent_goals_against_per_h2h_scored_match_pre
opponent_goals_against_per_h2h_scored_match_pre
0.0    283

opponent_clean_sheet_rate_h2h_scored_match_pre
opponent_clean_sheet_rate_h2h_scored_match_pre
0.0    283

opponent_score_data_h2h_matches_pre
opponent_score_data_h2h_matches_pre
0    283

no_h2h_score_history_pre
no_h2h_score_history_pre
1    283

FIRST-MEETING ELO SUMMARY
                                 count       mean      std      min       25%      50%       75%        

## Reconstruct the true external state at the forecast cutoff

In [19]:
# Reconstruct external-state values at the 2026-08-28 cutoff

from collections import defaultdict, deque


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def points_from_result(result):
    if result == "W":
        return 3
    if result == "D":
        return 1
    return 0


def elo_expected(rating_a, rating_b):
    return 1.0 / (
        1.0
        + 10.0 ** (
            (rating_b - rating_a) / 400.0
        )
    )


def elo_score(result):
    if result == "W":
        return 1.0
    if result == "D":
        return 0.5
    return 0.0


# ---------------------------------------------------------
# Stateful containers
# ---------------------------------------------------------

team_recent = defaultdict(
    lambda: {
        "points": deque(maxlen=10),
        "results": deque(maxlen=5),
        "goals_for": deque(maxlen=5),
        "goals_against": deque(maxlen=5),
    }
)

h2h_state = defaultdict(
    lambda: {
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "goals_for": [],
        "goals_against": [],
    }
)

# Elo is a shared matchup network.
elo_rating = defaultdict(
    lambda: 1500.0
)

elo_games = defaultdict(int)


# ---------------------------------------------------------
# Replay all 1,333 historical appearances
# ---------------------------------------------------------

for _, row in (
    df.sort_values("date")
    .iterrows()
):

    team = row["team"]
    opponent = row["opponent"]

    result = row["result"]

    team_goals = int(
        row["team_goals"]
    )

    opponent_goals = int(
        row["opponent_goals"]
    )


    # -----------------------------
    # Team recent form
    # -----------------------------

    team_recent[
        team
    ]["points"].append(
        points_from_result(
            result
        )
    )

    team_recent[
        team
    ]["results"].append(
        result
    )

    team_recent[
        team
    ]["goals_for"].append(
        team_goals
    )

    team_recent[
        team
    ]["goals_against"].append(
        opponent_goals
    )


    # -----------------------------
    # Team-opponent H2H
    # -----------------------------

    key = (
        team,
        opponent,
    )

    if result == "W":
        h2h_state[key][
            "wins"
        ] += 1

    elif result == "D":
        h2h_state[key][
            "draws"
        ] += 1

    else:
        h2h_state[key][
            "losses"
        ] += 1


    h2h_state[key][
        "goals_for"
    ].append(
        team_goals
    )

    h2h_state[key][
        "goals_against"
    ].append(
        opponent_goals
    )


    # -----------------------------
    # Matchup Elo update
    # -----------------------------

    team_rating = float(
        elo_rating[team]
    )

    opponent_rating = float(
        elo_rating[opponent]
    )

    expected = elo_expected(
        team_rating,
        opponent_rating,
    )

    actual = elo_score(
        result
    )

    change = (
        20.0
        * (
            actual
            - expected
        )
    )

    elo_rating[team] = (
        team_rating
        + change
    )

    elo_rating[opponent] = (
        opponent_rating
        - change
    )

    elo_games[team] += 1
    elo_games[opponent] += 1


# ---------------------------------------------------------
# Build cutoff team-state table
# ---------------------------------------------------------

team_state_rows = []

for team in [
    "Al-Nassr",
    "Portugal",
]:

    state = team_recent[
        team
    ]

    points = list(
        state["points"]
    )

    results = list(
        state["results"]
    )

    goals_for = list(
        state["goals_for"]
    )

    goals_against = list(
        state["goals_against"]
    )


    last_5_points = (
        points[-5:]
    )

    last_10_points = (
        points[-10:]
    )


    team_state_rows.append(
        {
            "team":
                team,

            "team_points_last_5_ronaldo_apps_pre":
                float(
                    sum(
                        last_5_points
                    )
                ),

            "team_points_last_10_ronaldo_apps_pre":
                float(
                    sum(
                        last_10_points
                    )
                ),

            "team_win_rate_last_5_ronaldo_apps_pre":
                float(
                    sum(
                        r == "W"
                        for r in results
                    )
                    / len(results)
                ),

            "team_unbeaten_rate_last_5_ronaldo_apps_pre":
                float(
                    sum(
                        r in {"W", "D"}
                        for r in results
                    )
                    / len(results)
                ),

            "team_goals_for_last_5_scored_matches_pre":
                float(
                    sum(
                        goals_for
                    )
                ),

            "team_goals_against_last_5_scored_matches_pre":
                float(
                    sum(
                        goals_against
                    )
                ),

            "team_score_coverage_last_5_pre":
                float(
                    len(
                        goals_for
                    )
                ),

            "no_team_score_history_pre":
                int(
                    len(
                        goals_for
                    )
                    == 0
                ),

            "team_matchup_elo_pre":
                float(
                    elo_rating[
                        team
                    ]
                ),

            "team_matchup_elo_games":
                int(
                    elo_games[
                        team
                    ]
                ),
        }
    )


cutoff_team_state = pd.DataFrame(
    team_state_rows
)


# ---------------------------------------------------------
# Build cutoff H2H/Elo state for all future opponents
# ---------------------------------------------------------

opponent_state_rows = []

for _, fixture in (
    future_fixtures[
        [
            "team",
            "opponent",
        ]
    ]
    .drop_duplicates()
    .iterrows()
):

    team = fixture["team"]
    opponent = fixture["opponent"]

    key = (
        team,
        opponent,
    )

    state = h2h_state[key]

    wins = int(
        state["wins"]
    )

    draws = int(
        state["draws"]
    )

    losses = int(
        state["losses"]
    )

    total_matches = (
        wins
        + draws
        + losses
    )

    goals_for = (
        state["goals_for"]
    )

    goals_against = (
        state["goals_against"]
    )

    scored_matches = len(
        goals_for
    )


    if total_matches > 0:

        opponent_points = (
            losses * 3
            + draws
        )

        opponent_ppg = (
            opponent_points
            / total_matches
        )

        opponent_unbeaten = (
            losses
            + draws
        ) / total_matches

    else:

        opponent_ppg = 0.0
        opponent_unbeaten = 0.0


    if scored_matches > 0:

        # From opponent perspective
        opponent_gf = (
            sum(
                goals_against
            )
            / scored_matches
        )

        opponent_ga = (
            sum(
                goals_for
            )
            / scored_matches
        )

        opponent_clean_sheet = (
            sum(
                gf == 0
                for gf in goals_for
            )
            / scored_matches
        )

    else:

        opponent_gf = 0.0
        opponent_ga = 0.0
        opponent_clean_sheet = 0.0


    opponent_rating = float(
        elo_rating[
            opponent
        ]
    )

    opponent_games = int(
        elo_games[
            opponent
        ]
    )

    team_rating = float(
        elo_rating[
            team
        ]
    )


    opponent_state_rows.append(
        {
            "team":
                team,

            "opponent":
                opponent,

            "team_h2h_wins_pre":
                wins,

            "team_h2h_draws_pre":
                draws,

            "team_h2h_losses_pre":
                losses,

            "opponent_h2h_points_per_game_pre":
                float(
                    opponent_ppg
                ),

            "opponent_h2h_unbeaten_rate_pre":
                float(
                    opponent_unbeaten
                ),

            "opponent_goals_for_per_h2h_scored_match_pre":
                float(
                    opponent_gf
                ),

            "opponent_goals_against_per_h2h_scored_match_pre":
                float(
                    opponent_ga
                ),

            "opponent_clean_sheet_rate_h2h_scored_match_pre":
                float(
                    opponent_clean_sheet
                ),

            "opponent_score_data_h2h_matches_pre":
                scored_matches,

            "no_h2h_score_history_pre":
                int(
                    scored_matches
                    == 0
                ),

            "opponent_matchup_elo_pre":
                opponent_rating,

            "opponent_matchup_elo_games_pre":
                opponent_games,

            "opponent_elo_unseen_pre":
                int(
                    opponent_games
                    == 0
                ),

            "opponent_elo_experience_log_pre":
                float(
                    np.log1p(
                        opponent_games
                    )
                ),

            "team_matchup_elo_pre":
                team_rating,

            "matchup_elo_diff_pre":
                float(
                    team_rating
                    - opponent_rating
                ),
        }
    )


cutoff_opponent_state = pd.DataFrame(
    opponent_state_rows
)


# ---------------------------------------------------------
# Audit
# ---------------------------------------------------------

print(
    "CUTOFF TEAM EXTERNAL STATE"
)

print("=" * 110)

print(
    cutoff_team_state
    .round(4)
    .to_string(
        index=False
    )
)


print(
    "\nFUTURE-OPPONENT CUTOFF STATE"
)

print("=" * 110)

print(
    cutoff_opponent_state[
        [
            "team",
            "opponent",
            "team_h2h_wins_pre",
            "team_h2h_draws_pre",
            "team_h2h_losses_pre",
            "opponent_score_data_h2h_matches_pre",
            "opponent_matchup_elo_pre",
            "opponent_matchup_elo_games_pre",
            "opponent_elo_unseen_pre",
            "matchup_elo_diff_pre",
        ]
    ]
    .sort_values(
        [
            "team",
            "opponent",
        ]
    )
    .round(4)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Key integrity checks
# ---------------------------------------------------------

al_nassr_elo = float(
    cutoff_team_state.loc[
        cutoff_team_state[
            "team"
        ]
        == "Al-Nassr",
        "team_matchup_elo_pre",
    ].iloc[0]
)

portugal_elo = float(
    cutoff_team_state.loc[
        cutoff_team_state[
            "team"
        ]
        == "Portugal",
        "team_matchup_elo_pre",
    ].iloc[0]
)


# All genuinely unseen opponents should use
# the historical network default.
unseen_names = set(
    opponent_coverage.loc[
        ~opponent_coverage[
            "has_history"
        ],
        "opponent",
    ]
)

unseen_state = (
    cutoff_opponent_state[
        cutoff_opponent_state[
            "opponent"
        ].isin(
            unseen_names
        )
    ]
)

assert (
    unseen_state[
        "opponent_matchup_elo_pre"
    ]
    .eq(1500.0)
    .all()
)

assert (
    unseen_state[
        "opponent_matchup_elo_games_pre"
    ]
    .eq(0)
    .all()
)

assert (
    unseen_state[
        "no_h2h_score_history_pre"
    ]
    .eq(1)
    .all()
)

assert (
    unseen_state[
        [
            "team_h2h_wins_pre",
            "team_h2h_draws_pre",
            "team_h2h_losses_pre",
        ]
    ]
    .eq(0)
    .all()
    .all()
)


print(
    "\nPOST-MATCH CUTOFF ELO"
)

print("=" * 60)

print(
    "Al-Nassr:",
    round(
        al_nassr_elo,
        4,
    )
)

print(
    "Portugal:",
    round(
        portugal_elo,
        4,
    )
)

print(
    "\nExternal-state reconstruction: PASSED"
)

CUTOFF TEAM EXTERNAL STATE
    team  team_points_last_5_ronaldo_apps_pre  team_points_last_10_ronaldo_apps_pre  team_win_rate_last_5_ronaldo_apps_pre  team_unbeaten_rate_last_5_ronaldo_apps_pre  team_goals_for_last_5_scored_matches_pre  team_goals_against_last_5_scored_matches_pre  team_score_coverage_last_5_pre  no_team_score_history_pre  team_matchup_elo_pre  team_matchup_elo_games
Al-Nassr                                 12.0                                  22.0                                    0.8                                         0.8                                      13.0                                           5.0                             5.0                          0             1707.3866                     151
Portugal                                  8.0                                  18.0                                    0.4                                         0.8                                       8.0                                           3.

In [21]:
# Reconstruct Ronaldo assist state after the 2026-08-28 match

history = (
    df.sort_values("date")
    .reset_index(drop=True)
    .copy()
)

assist_history = (
    history["ronaldo_assists"]
    .astype(float)
    .tolist()
)


# ---------------------------------------------------------
# Career assist state
# ---------------------------------------------------------

career_assists_cutoff = float(
    sum(assist_history)
)


# ---------------------------------------------------------
# Rolling assist state
# ---------------------------------------------------------

assists_last_5_cutoff = float(
    sum(
        assist_history[-5:]
    )
)

assists_last_10_cutoff = float(
    sum(
        assist_history[-10:]
    )
)


# ---------------------------------------------------------
# Determine the EWMA definition used in the dataset
#
# Rather than assuming the alpha, recover it by testing
# common alpha values against the historical *_pre feature.
# ---------------------------------------------------------

candidate_alphas = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.50,
]

ewma_results = []

for alpha in candidate_alphas:

    state = 0.0
    predicted_pre = []

    for assists in assist_history:

        predicted_pre.append(
            state
        )

        state = (
            alpha * assists
            + (1 - alpha) * state
        )

    actual_pre = (
        history[
            "assists_ewma_pre"
        ]
        .astype(float)
        .to_numpy()
    )

    predicted_pre = np.array(
        predicted_pre,
        dtype=float,
    )

    max_error = float(
        np.max(
            np.abs(
                actual_pre
                - predicted_pre
            )
        )
    )

    mean_error = float(
        np.mean(
            np.abs(
                actual_pre
                - predicted_pre
            )
        )
    )

    ewma_results.append(
        {
            "alpha": alpha,
            "max_abs_error": max_error,
            "mean_abs_error": mean_error,
            "post_cutoff_state": state,
        }
    )


ewma_audit = pd.DataFrame(
    ewma_results
).sort_values(
    [
        "max_abs_error",
        "mean_abs_error",
    ]
)

best_ewma = (
    ewma_audit
    .iloc[0]
)

best_alpha = float(
    best_ewma["alpha"]
)

assists_ewma_cutoff = float(
    best_ewma[
        "post_cutoff_state"
    ]
)


# ---------------------------------------------------------
# Season assist state
# ---------------------------------------------------------

cutoff_season = (
    history.iloc[-1][
        "season"
    ]
)

season_rows = (
    history[
        history["season"]
        == cutoff_season
    ]
)

season_assists_cutoff = float(
    season_rows[
        "ronaldo_assists"
    ].sum()
)


# ---------------------------------------------------------
# Ronaldo assists vs each future opponent
# ---------------------------------------------------------

opponent_assist_rows = []

for opponent in (
    future_fixtures[
        "opponent"
    ]
    .drop_duplicates()
):

    opponent_history = (
        history[
            history["opponent"]
            == opponent
        ]
    )

    opponent_assist_rows.append(
        {
            "opponent":
                opponent,

            "ronaldo_assists_vs_opponent_pre":
                float(
                    opponent_history[
                        "ronaldo_assists"
                    ].sum()
                ),

            "historical_meetings":
                int(
                    len(
                        opponent_history
                    )
                ),
        }
    )


cutoff_opponent_assists = pd.DataFrame(
    opponent_assist_rows
)


# ---------------------------------------------------------
# Cutoff snapshot
# ---------------------------------------------------------

cutoff_assist_state = {
    "career_assists_pre":
        career_assists_cutoff,

    "assists_last_5":
        assists_last_5_cutoff,

    "assists_last_10":
        assists_last_10_cutoff,

    "assists_ewma_pre":
        assists_ewma_cutoff,

    "season_assists_pre":
        season_assists_cutoff,
}


print(
    "ASSIST EWMA RECONSTRUCTION AUDIT"
)

print("=" * 90)

print(
    ewma_audit[
        [
            "alpha",
            "max_abs_error",
            "mean_abs_error",
        ]
    ]
    .round(8)
    .to_string(
        index=False
    )
)


print(
    "\nSELECTED EWMA"
)

print("=" * 90)

print(
    "Alpha:",
    best_alpha
)

print(
    "Maximum historical error:",
    round(
        float(
            best_ewma[
                "max_abs_error"
            ]
        ),
        10,
    )
)


print(
    "\nRONALDO ASSIST STATE AT 2026-08-28 CUTOFF"
)

print("=" * 90)

for feature, value in (
    cutoff_assist_state.items()
):

    print(
        f"{feature}: {value:.6f}"
    )


print(
    "\nLATEST MATCH"
)

print("=" * 90)

print(
    "Date:",
    history.iloc[-1]["date"]
)

print(
    "Opponent:",
    history.iloc[-1]["opponent"]
)

print(
    "Goals:",
    int(
        history.iloc[-1][
            "ronaldo_goals"
        ]
    )
)

print(
    "Assists:",
    float(
        history.iloc[-1][
            "ronaldo_assists"
        ]
    )
)


print(
    "\nASSISTS VS FUTURE OPPONENTS"
)

print("=" * 90)

print(
    cutoff_opponent_assists
    .sort_values(
        [
            "historical_meetings",
            "opponent",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Integrity
# ---------------------------------------------------------

assert (
    career_assists_cutoff
    ==
    float(
        history[
            "ronaldo_assists"
        ].sum()
    )
)

assert (
    assists_last_5_cutoff
    ==
    float(
        history[
            "ronaldo_assists"
        ]
        .tail(5)
        .sum()
    )
)

assert (
    assists_last_10_cutoff
    ==
    float(
        history[
            "ronaldo_assists"
        ]
        .tail(10)
        .sum()
    )
)

assert (
    cutoff_opponent_assists[
        "ronaldo_assists_vs_opponent_pre"
    ]
    .ge(0)
    .all()
)


print(
    "\nAssist-state reconstruction audit: PASSED"
)

ASSIST EWMA RECONSTRUCTION AUDIT
 alpha  max_abs_error  mean_abs_error
  0.30       0.000000        0.000000
  0.25       0.143235        0.019050
  0.35       0.146921        0.017962
  0.20       0.279595        0.039479
  0.40       0.295631        0.035091
  0.15       0.428857        0.061699
  0.50       0.594961        0.067247
  0.10       0.603564        0.086167
  0.05       0.782467        0.113721

SELECTED EWMA
Alpha: 0.3
Maximum historical error: 0.0

RONALDO ASSIST STATE AT 2026-08-28 CUTOFF
career_assists_pre: 261.000000
assists_last_5: 0.000000
assists_last_10: 0.000000
assists_ewma_pre: 0.000119
season_assists_pre: 0.000000

LATEST MATCH
Date: 2026-08-28 00:00:00
Opponent: Al Taawon
Goals: 1
Assists: 0.0

ASSISTS VS FUTURE OPPONENTS
            opponent  ronaldo_assists_vs_opponent_pre  historical_meetings
            Al-Hilal                              1.0                   11
           Al Shabab                              1.0                    9
           Al 

## Lock the forecast-state policy

In [23]:
# Lock and save the Phase 9 forecast-state policy


forecast_state_policy = {
    "forecast_cutoff": "2026-08-28",

    "historical_starting_state": {
        "career_appearances": 1333,
        "career_goals": 978,
        "goals_remaining_to_1000": 22,
    },

    "model": {
        "primary": "ensemble",
        "xgboost_weight": 0.60,
        "catboost_weight": 0.40,
        "simulation_distribution": "poisson",
    },

    "feature_dependency_counts": {
        "xgboost": {
            "total": 80,
            "fixture_known": 15,
            "ronaldo_recursive": 35,
            "external_state": 30,
        },

        "catboost": {
            "total": 84,
            "fixture_known": 19,
            "ronaldo_recursive": 35,
            "external_state": 30,
        },
    },

    "state_update_policy": {
        "ronaldo_recursive": (
            "Update after each simulated Ronaldo appearance "
            "using simulated Ronaldo goals and appearance history."
        ),

        "fixture_known": (
            "Calculate from the future fixture calendar using only "
            "information available independently of future match results."
        ),

        "external_state": (
            "Initialize from the reconstructed historical state at the "
            "2026-08-28 cutoff and carry forward without updates from "
            "unknown future team results, scores, assists, or Elo changes."
        ),
    },

    "external_state_groups": {
        "assist_state": [
            "career_assists_pre",
            "assists_last_5",
            "assists_last_10",
            "assists_ewma_pre",
            "season_assists_pre",
            "ronaldo_assists_vs_opponent_pre",
        ],

        "team_form_state": [
            "team_points_last_5_ronaldo_apps_pre",
            "team_points_last_10_ronaldo_apps_pre",
            "team_win_rate_last_5_ronaldo_apps_pre",
            "team_unbeaten_rate_last_5_ronaldo_apps_pre",
            "team_goals_for_last_5_scored_matches_pre",
            "team_goals_against_last_5_scored_matches_pre",
            "team_score_coverage_last_5_pre",
            "no_team_score_history_pre",
        ],

        "h2h_result_state": [
            "team_h2h_wins_pre",
            "team_h2h_draws_pre",
            "team_h2h_losses_pre",
            "opponent_h2h_points_per_game_pre",
            "opponent_h2h_unbeaten_rate_pre",
            "opponent_goals_for_per_h2h_scored_match_pre",
            "opponent_goals_against_per_h2h_scored_match_pre",
            "opponent_clean_sheet_rate_h2h_scored_match_pre",
            "opponent_score_data_h2h_matches_pre",
            "no_h2h_score_history_pre",
        ],

        "elo_state": [
            "team_matchup_elo_pre",
            "opponent_matchup_elo_pre",
            "matchup_elo_diff_pre",
            "opponent_matchup_elo_games_pre",
            "opponent_elo_unseen_pre",
            "opponent_elo_experience_log_pre",
        ],
    },

    "assist_ewma": {
        "alpha": 0.30,
        "historical_reconstruction_max_error": 0.0,
    },

    "unseen_opponent_policy": {
        "h2h_wins": 0,
        "h2h_draws": 0,
        "h2h_losses": 0,
        "h2h_score_matches": 0,
        "h2h_rates": 0.0,
        "opponent_elo": 1500.0,
        "opponent_elo_games": 0,
        "opponent_elo_unseen": 1,
        "opponent_elo_experience_log": 0.0,
        "ronaldo_assists_vs_opponent": 0.0,
        "basis": (
            "Matches the historical first-meeting convention. "
            "281 of 283 historical first meetings had unseen-opponent "
            "Elo state; the two exceptions were opponents already present "
            "elsewhere in the historical Elo network."
        ),
    },

    "future_fixture_policy": {
        "confirmed_fixture_rows": 45,
        "fixture_horizon_end": "2027-05-28",
        "conditional_future_rounds_included": False,
        "post_cutoff_results_allowed": False,
    },

    "known_limitations": [
        (
            "Future assists are not simulated, so assist-dependent "
            "features are held at their cutoff state."
        ),
        (
            "Future team scores and results are not simulated, so "
            "team-form and score-based H2H features are held at "
            "their cutoff state."
        ),
        (
            "Future Elo changes are not simulated; cutoff Elo ratings "
            "are carried forward."
        ),
        (
            "Opponent-specific historical state is preserved rather "
            "than replacing seen opponents with generic defaults."
        ),
        (
            "Appearance uncertainty is not yet resolved and will be "
            "handled separately from goal-generation uncertainty."
        ),
    ],
}


# ---------------------------------------------------------
# Save artifact
# ---------------------------------------------------------

artifact_path = Path(
    "../artifacts/forecast_state_policy.json"
)

artifact_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    artifact_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        forecast_state_policy,
        f,
        indent=2,
    )


# ---------------------------------------------------------
# Verify saved artifact
# ---------------------------------------------------------

with open(
    artifact_path,
    "r",
    encoding="utf-8",
) as f:

    loaded_policy = json.load(
        f
    )


assert (
    loaded_policy[
        "forecast_cutoff"
    ]
    == "2026-08-28"
)

assert (
    loaded_policy[
        "historical_starting_state"
    ][
        "career_goals"
    ]
    == 978
)

assert (
    loaded_policy[
        "historical_starting_state"
    ][
        "goals_remaining_to_1000"
    ]
    == 22
)

assert (
    loaded_policy[
        "model"
    ][
        "simulation_distribution"
    ]
    == "poisson"
)

assert (
    loaded_policy[
        "state_update_policy"
    ][
        "external_state"
    ]
)

assert (
    loaded_policy[
        "known_limitations"
    ]
)


print(
    "FORECAST STATE POLICY"
)

print("=" * 90)

print(
    "Cutoff:",
    loaded_policy[
        "forecast_cutoff"
    ]
)

print(
    "Starting career goals:",
    loaded_policy[
        "historical_starting_state"
    ][
        "career_goals"
    ]
)

print(
    "Goals remaining:",
    loaded_policy[
        "historical_starting_state"
    ][
        "goals_remaining_to_1000"
    ]
)

print(
    "XGB dependencies:",
    loaded_policy[
        "feature_dependency_counts"
    ][
        "xgboost"
    ]
)

print(
    "CatBoost dependencies:",
    loaded_policy[
        "feature_dependency_counts"
    ][
        "catboost"
    ]
)

print(
    "Assist EWMA alpha:",
    loaded_policy[
        "assist_ewma"
    ][
        "alpha"
    ]
)

print(
    "Fixture horizon:",
    loaded_policy[
        "future_fixture_policy"
    ][
        "fixture_horizon_end"
    ]
)

print(
    "\nSaved:",
    artifact_path
)

print(
    "\nForecast-state policy: LOCKED"
)

FORECAST STATE POLICY
Cutoff: 2026-08-28
Starting career goals: 978
Goals remaining: 22
XGB dependencies: {'total': 80, 'fixture_known': 15, 'ronaldo_recursive': 35, 'external_state': 30}
CatBoost dependencies: {'total': 84, 'fixture_known': 19, 'ronaldo_recursive': 35, 'external_state': 30}
Assist EWMA alpha: 0.3
Fixture horizon: 2027-05-28

Saved: ..\artifacts\forecast_state_policy.json

Forecast-state policy: LOCKED


## Identify and audit all 35 Ronaldo-recursive features

In [25]:
# Audit the complete Ronaldo-recursive feature contract

ronaldo_recursive_features.discard(
    "scoring_rate_last_20"
)


# ---------------------------------------------------------
# Correct recursive feature groups
# ---------------------------------------------------------

recursive_groups = {

    "career": [
        "career_appearances_pre",
        "career_goals_pre",
        "career_goals_per_game_pre",
        "no_prior_career_goal_pre",
        "first_career_appearance",
    ],

    "recent_scoring": [
        "goals_last_3",
        "goals_per_game_last_5",
        "goals_per_game_last_10",
        "goals_per_game_last_20",
        "goals_ewma_pre",
        "scoring_rate_last_5",
        "scoring_rate_last_10",
        "multi_goal_rate_last_10",
        "consecutive_scoring_games_pre",
        "goalless_streak_games_pre",
        "days_since_last_goal",
    ],

    "season": [
        "season_appearances_pre",
        "season_goals_pre",
        "season_goals_per_game_pre",
    ],

    "team": [
        "team_appearances_with_ronaldo_pre",
        "team_goals_by_ronaldo_pre",
        "team_ronaldo_goals_per_game_pre",
    ],

    "opponent": [
        "opponent_prior_meetings_pre",
        "ronaldo_goals_vs_opponent_pre",
        "ronaldo_goals_per_game_vs_opponent_pre",
        "ronaldo_scoring_rate_vs_opponent_pre",
        "ronaldo_goals_last_3_vs_opponent_pre",
        "first_time_opponent_pre",
        "opponent_meetings_log_pre",
    ],

    "competition": [
        "competition_appearances_pre",
        "competition_goals_pre",
        "competition_goals_per_game_pre",
    ],

    "venue": [
        "venue_appearances_pre",
        "venue_goals_pre",
        "venue_goals_per_game_pre",
    ],
}


# ---------------------------------------------------------
# Flatten contract
# ---------------------------------------------------------

recursive_contract = [
    feature
    for features in recursive_groups.values()
    for feature in features
]

expected_recursive = set(
    ronaldo_recursive_features
)

contract_set = set(
    recursive_contract
)


# ---------------------------------------------------------
# Check against ACTUAL locked model features
# ---------------------------------------------------------

actual_recursive_model_features = (
    (
        set(XGB_FEATURES)
        |
        set(CATBOOST_FEATURES)
    )
    &
    expected_recursive
)


missing_from_contract = sorted(
    actual_recursive_model_features
    - contract_set
)

unexpected_in_contract = sorted(
    contract_set
    - actual_recursive_model_features
)

missing_from_df = sorted(
    contract_set
    - set(df.columns)
)

duplicates = sorted(
    {
        feature
        for feature in recursive_contract
        if recursive_contract.count(feature) > 1
    }
)


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print(
    "RONALDO RECURSIVE FEATURE CONTRACT"
)

print("=" * 95)

for group_name, features in recursive_groups.items():

    print(
        f"\n{group_name.upper()} ({len(features)})"
    )

    print("-" * 95)

    for feature in features:
        print(
            " -",
            feature,
        )


print(
    "\nCONTRACT AUDIT"
)

print("=" * 95)

print(
    "Recursive classification size:",
    len(expected_recursive)
)

print(
    "Actual recursive model features:",
    len(actual_recursive_model_features)
)

print(
    "Features in contract:",
    len(recursive_contract)
)

print(
    "Unique contract features:",
    len(contract_set)
)

print(
    "Missing from contract:",
    len(missing_from_contract)
)

print(
    "Unexpected in contract:",
    len(unexpected_in_contract)
)

print(
    "Missing from dataframe:",
    len(missing_from_df)
)

print(
    "Duplicates:",
    len(duplicates)
)


if missing_from_contract:
    print(
        "\nMISSING FROM CONTRACT:"
    )
    for feature in missing_from_contract:
        print(" -", feature)


if unexpected_in_contract:
    print(
        "\nUNEXPECTED IN CONTRACT:"
    )
    for feature in unexpected_in_contract:
        print(" -", feature)


if missing_from_df:
    print(
        "\nMISSING FROM DATAFRAME:"
    )
    for feature in missing_from_df:
        print(" -", feature)


# ---------------------------------------------------------
# Integrity
# ---------------------------------------------------------

assert (
    "scoring_rate_last_20"
    not in ronaldo_recursive_features
)

assert (
    len(actual_recursive_model_features)
    == 35
)

assert (
    len(recursive_contract)
    == 35
)

assert (
    len(contract_set)
    == 35
)

assert not missing_from_contract
assert not unexpected_in_contract
assert not missing_from_df
assert not duplicates


print(
    "\nRecursive feature contract: PASSED"
)

RONALDO RECURSIVE FEATURE CONTRACT

CAREER (5)
-----------------------------------------------------------------------------------------------
 - career_appearances_pre
 - career_goals_pre
 - career_goals_per_game_pre
 - no_prior_career_goal_pre
 - first_career_appearance

RECENT_SCORING (11)
-----------------------------------------------------------------------------------------------
 - goals_last_3
 - goals_per_game_last_5
 - goals_per_game_last_10
 - goals_per_game_last_20
 - goals_ewma_pre
 - scoring_rate_last_5
 - scoring_rate_last_10
 - multi_goal_rate_last_10
 - consecutive_scoring_games_pre
 - goalless_streak_games_pre
 - days_since_last_goal

SEASON (3)
-----------------------------------------------------------------------------------------------
 - season_appearances_pre
 - season_goals_pre
 - season_goals_per_game_pre

TEAM (3)
-----------------------------------------------------------------------------------------------
 - team_appearances_with_ronaldo_pre
 - team_goals

In [27]:
# Reconstruct all Ronaldo-recursive features historically
# and compare them against the engineered dataset.



# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def safe_rate(numerator, denominator):
    return (
        float(numerator) / float(denominator)
        if denominator > 0
        else 0.0
    )


def rolling_sum(values, window):
    vals = list(values)[-window:]
    return float(sum(vals))


def rolling_mean(values, window):
    vals = list(values)[-window:]

    return (
        float(sum(vals)) / len(vals)
        if len(vals) > 0
        else 0.0
    )


def rolling_scoring_rate(values, window):
    vals = list(values)[-window:]

    return (
        float(
            sum(
                g > 0
                for g in vals
            )
        )
        / len(vals)
        if len(vals) > 0
        else 0.0
    )


def rolling_multi_goal_rate(values, window):
    vals = list(values)[-window:]

    return (
        float(
            sum(
                g >= 2
                for g in vals
            )
        )
        / len(vals)
        if len(vals) > 0
        else 0.0
    )


# ---------------------------------------------------------
# Historical data
# ---------------------------------------------------------

hist = (
    df.sort_values("date")
    .reset_index(drop=True)
    .copy()
)


# ---------------------------------------------------------
# Stateful containers
# ---------------------------------------------------------

career_goals = []
career_appearances = 0
career_goal_total = 0

# Goal EWMA — historical engineering used alpha=0.30
goal_ewma_state = 0.0
GOAL_EWMA_ALPHA = 0.30

last_goal_date = None

consecutive_scoring = 0
goalless_streak = 0


season_state = defaultdict(
    lambda: {
        "apps": 0,
        "goals": 0,
    }
)

team_state = defaultdict(
    lambda: {
        "apps": 0,
        "goals": 0,
    }
)

opponent_state = defaultdict(
    lambda: {
        "apps": 0,
        "goals": 0,
        "goal_history": [],
    }
)

competition_state = defaultdict(
    lambda: {
        "apps": 0,
        "goals": 0,
    }
)

venue_state = defaultdict(
    lambda: {
        "apps": 0,
        "goals": 0,
    }
)


# ---------------------------------------------------------
# Replay historical appearances
# ---------------------------------------------------------

reconstructed_rows = []


for _, row in hist.iterrows():

    date = pd.Timestamp(
        row["date"]
    )

    season = row["season"]
    team = row["team"]
    opponent = row["opponent"]
    competition = row["competition"]
    venue = row["venue"]


    # =====================================================
    # PRE-MATCH STATE
    # =====================================================

    season_s = season_state[
        season
    ]

    team_s = team_state[
        team
    ]

    opponent_s = opponent_state[
        opponent
    ]

    competition_s = competition_state[
        competition
    ]

    venue_s = venue_state[
        venue
    ]


    # -----------------------------------------------------
    # Career
    # -----------------------------------------------------

    career_appearances_pre = (
        career_appearances
    )

    career_goals_pre = (
        career_goal_total
    )

    career_goals_per_game_pre = safe_rate(
        career_goal_total,
        career_appearances,
    )


    # -----------------------------------------------------
    # Recent scoring
    # -----------------------------------------------------

    goals_last_3 = rolling_sum(
        career_goals,
        3,
    )

    goals_per_game_last_5 = rolling_mean(
        career_goals,
        5,
    )

    goals_per_game_last_10 = rolling_mean(
        career_goals,
        10,
    )

    goals_per_game_last_20 = rolling_mean(
        career_goals,
        20,
    )

    scoring_rate_last_5 = rolling_scoring_rate(
        career_goals,
        5,
    )

    scoring_rate_last_10 = rolling_scoring_rate(
        career_goals,
        10,
    )

    multi_goal_rate_last_10 = rolling_multi_goal_rate(
        career_goals,
        10,
    )


    if last_goal_date is None:

        days_since_last_goal = np.nan

    else:

        days_since_last_goal = float(
            (
                date
                - last_goal_date
            ).days
        )


    # -----------------------------------------------------
    # Season
    # -----------------------------------------------------

    season_appearances_pre = (
        season_s["apps"]
    )

    season_goals_pre = (
        season_s["goals"]
    )

    season_goals_per_game_pre = safe_rate(
        season_s["goals"],
        season_s["apps"],
    )


    # -----------------------------------------------------
    # Team
    # -----------------------------------------------------

    team_appearances_with_ronaldo_pre = (
        team_s["apps"]
    )

    team_goals_by_ronaldo_pre = (
        team_s["goals"]
    )

    team_ronaldo_goals_per_game_pre = safe_rate(
        team_s["goals"],
        team_s["apps"],
    )


    # -----------------------------------------------------
    # Opponent
    # -----------------------------------------------------

    opponent_prior_meetings_pre = (
        opponent_s["apps"]
    )

    ronaldo_goals_vs_opponent_pre = (
        opponent_s["goals"]
    )

    ronaldo_goals_per_game_vs_opponent_pre = safe_rate(
        opponent_s["goals"],
        opponent_s["apps"],
    )

    ronaldo_scoring_rate_vs_opponent_pre = (
        safe_rate(
            sum(
                g > 0
                for g in opponent_s[
                    "goal_history"
                ]
            ),
            opponent_s["apps"],
        )
    )

    ronaldo_goals_last_3_vs_opponent_pre = (
        rolling_sum(
            opponent_s[
                "goal_history"
            ],
            3,
        )
    )


    # -----------------------------------------------------
    # Competition
    # -----------------------------------------------------

    competition_appearances_pre = (
        competition_s["apps"]
    )

    competition_goals_pre = (
        competition_s["goals"]
    )

    competition_goals_per_game_pre = safe_rate(
        competition_s["goals"],
        competition_s["apps"],
    )


    # -----------------------------------------------------
    # Venue
    # -----------------------------------------------------

    venue_appearances_pre = (
        venue_s["apps"]
    )

    venue_goals_pre = (
        venue_s["goals"]
    )

    venue_goals_per_game_pre = safe_rate(
        venue_s["goals"],
        venue_s["apps"],
    )


    reconstructed_rows.append(
        {
            "career_appearances_pre":
                career_appearances_pre,

            "career_goals_pre":
                career_goals_pre,

            "career_goals_per_game_pre":
                career_goals_per_game_pre,

            "no_prior_career_goal_pre":
                int(
                    career_goal_total == 0
                ),

            "first_career_appearance":
                int(
                    career_appearances == 0
                ),

            "goals_last_3":
                goals_last_3,

            "goals_per_game_last_5":
                goals_per_game_last_5,

            "goals_per_game_last_10":
                goals_per_game_last_10,

            "goals_per_game_last_20":
                goals_per_game_last_20,

            "goals_ewma_pre":
                goal_ewma_state,

            "scoring_rate_last_5":
                scoring_rate_last_5,

            "scoring_rate_last_10":
                scoring_rate_last_10,

            "multi_goal_rate_last_10":
                multi_goal_rate_last_10,

            "consecutive_scoring_games_pre":
                consecutive_scoring,

            "goalless_streak_games_pre":
                goalless_streak,

            "days_since_last_goal":
                days_since_last_goal,

            "season_appearances_pre":
                season_appearances_pre,

            "season_goals_pre":
                season_goals_pre,

            "season_goals_per_game_pre":
                season_goals_per_game_pre,

            "team_appearances_with_ronaldo_pre":
                team_appearances_with_ronaldo_pre,

            "team_goals_by_ronaldo_pre":
                team_goals_by_ronaldo_pre,

            "team_ronaldo_goals_per_game_pre":
                team_ronaldo_goals_per_game_pre,

            "opponent_prior_meetings_pre":
                opponent_prior_meetings_pre,

            "ronaldo_goals_vs_opponent_pre":
                ronaldo_goals_vs_opponent_pre,

            "ronaldo_goals_per_game_vs_opponent_pre":
                ronaldo_goals_per_game_vs_opponent_pre,

            "ronaldo_scoring_rate_vs_opponent_pre":
                ronaldo_scoring_rate_vs_opponent_pre,

            "ronaldo_goals_last_3_vs_opponent_pre":
                ronaldo_goals_last_3_vs_opponent_pre,

            "first_time_opponent_pre":
                int(
                    opponent_s["apps"] == 0
                ),

            "opponent_meetings_log_pre":
                float(
                    np.log1p(
                        opponent_s["apps"]
                    )
                ),

            "competition_appearances_pre":
                competition_appearances_pre,

            "competition_goals_pre":
                competition_goals_pre,

            "competition_goals_per_game_pre":
                competition_goals_per_game_pre,

            "venue_appearances_pre":
                venue_appearances_pre,

            "venue_goals_pre":
                venue_goals_pre,

            "venue_goals_per_game_pre":
                venue_goals_per_game_pre,
        }
    )


    # =====================================================
    # POST-MATCH UPDATE
    # =====================================================

    goals = int(
        row["target_goals"]
    )


    career_goals.append(
        goals
    )

    career_appearances += 1
    career_goal_total += goals


    # Goal EWMA
    goal_ewma_state = (
        GOAL_EWMA_ALPHA * goals
        + (
            1.0
            - GOAL_EWMA_ALPHA
        )
        * goal_ewma_state
    )


    # Scoring / goalless streak
    if goals > 0:

        consecutive_scoring += 1
        goalless_streak = 0
        last_goal_date = date

    else:

        consecutive_scoring = 0
        goalless_streak += 1


    # Season
    season_s["apps"] += 1
    season_s["goals"] += goals


    # Team
    team_s["apps"] += 1
    team_s["goals"] += goals


    # Opponent
    opponent_s["apps"] += 1
    opponent_s["goals"] += goals

    opponent_s[
        "goal_history"
    ].append(
        goals
    )


    # Competition
    competition_s["apps"] += 1
    competition_s["goals"] += goals


    # Venue
    venue_s["apps"] += 1
    venue_s["goals"] += goals


# ---------------------------------------------------------
# Reconstructed feature dataframe
# ---------------------------------------------------------

recursive_replay = pd.DataFrame(
    reconstructed_rows
)


assert (
    len(recursive_replay)
    ==
    len(hist)
)


# ---------------------------------------------------------
# Compare every feature with historical engineered values
# ---------------------------------------------------------

audit_rows = []


for feature in recursive_contract:

    actual = pd.to_numeric(
        hist[feature],
        errors="coerce",
    )

    rebuilt = pd.to_numeric(
        recursive_replay[feature],
        errors="coerce",
    )


    both_missing = (
        actual.isna()
        &
        rebuilt.isna()
    )

    valid = ~both_missing


    # NaN mismatch means one side is missing and the other isn't.
    nan_mismatch = (
        actual.isna()
        != rebuilt.isna()
    )


    abs_error = (
        actual
        - rebuilt
    ).abs()


    comparable_error = (
        abs_error[
            actual.notna()
            &
            rebuilt.notna()
        ]
    )


    max_error = (
        float(
            comparable_error.max()
        )
        if len(
            comparable_error
        ) > 0
        else 0.0
    )

    mean_error = (
        float(
            comparable_error.mean()
        )
        if len(
            comparable_error
        ) > 0
        else 0.0
    )


    mismatch_count = int(
        (
            (
                abs_error.fillna(0)
                > 1e-8
            )
            |
            nan_mismatch
        ).sum()
    )


    audit_rows.append(
        {
            "feature":
                feature,

            "mismatches":
                mismatch_count,

            "max_abs_error":
                max_error,

            "mean_abs_error":
                mean_error,

            "nan_mismatches":
                int(
                    nan_mismatch.sum()
                ),
        }
    )


recursive_replay_audit = (
    pd.DataFrame(
        audit_rows
    )
    .sort_values(
        [
            "mismatches",
            "max_abs_error",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print(
    "HISTORICAL RECURSIVE REPLAY AUDIT"
)

print("=" * 110)

print(
    recursive_replay_audit
    .to_string(
        index=False,
        formatters={
            "max_abs_error":
                lambda x: f"{x:.10f}",

            "mean_abs_error":
                lambda x: f"{x:.10f}",
        },
    )
)


perfect_features = int(
    (
        recursive_replay_audit[
            "mismatches"
        ]
        == 0
    ).sum()
)


problem_features = (
    recursive_replay_audit[
        recursive_replay_audit[
            "mismatches"
        ]
        > 0
    ]
)


print(
    "\nSUMMARY"
)

print("=" * 110)

print(
    "Rows replayed:",
    len(
        recursive_replay
    )
)

print(
    "Recursive features:",
    len(
        recursive_contract
    )
)

print(
    "Perfectly reconstructed:",
    perfect_features
)

print(
    "Features requiring investigation:",
    len(
        problem_features
    )
)


if len(
    problem_features
) > 0:

    print(
        "\nFEATURES REQUIRING INVESTIGATION"
    )

    print("-" * 110)

    print(
        problem_features
        .to_string(
            index=False,
            formatters={
                "max_abs_error":
                    lambda x: f"{x:.10f}",

                "mean_abs_error":
                    lambda x: f"{x:.10f}",
            },
        )
    )


print(
    "\nHistorical recursive replay audit: COMPLETE"
)

HISTORICAL RECURSIVE REPLAY AUDIT
                               feature  mismatches max_abs_error mean_abs_error  nan_mismatches
                  days_since_last_goal           5  0.0000000000   0.0000000000               5
                        goals_ewma_pre           0  0.0000000000   0.0000000000               0
ronaldo_goals_per_game_vs_opponent_pre           0  0.0000000000   0.0000000000               0
             opponent_meetings_log_pre           0  0.0000000000   0.0000000000               0
             season_goals_per_game_pre           0  0.0000000000   0.0000000000               0
       team_ronaldo_goals_per_game_pre           0  0.0000000000   0.0000000000               0
        competition_goals_per_game_pre           0  0.0000000000   0.0000000000               0
             career_goals_per_game_pre           0  0.0000000000   0.0000000000               0
  ronaldo_scoring_rate_vs_opponent_pre           0  0.0000000000   0.0000000000               0
      

In [29]:
# Inspect the only remaining recursive-feature mismatch

actual_days = pd.to_numeric(
    hist["days_since_last_goal"],
    errors="coerce",
)

rebuilt_days = pd.to_numeric(
    recursive_replay["days_since_last_goal"],
    errors="coerce",
)


nan_mismatch_mask = (
    actual_days.isna()
    != rebuilt_days.isna()
)


days_mismatch = (
    hist.loc[
        nan_mismatch_mask,
        [
            "match_id",
            "date",
            "team",
            "opponent",
            "competition",
            "target_goals",
            "career_appearances_pre",
            "career_goals_pre",
        ],
    ]
    .copy()
)


days_mismatch[
    "dataset_days_since_last_goal"
] = (
    actual_days[
        nan_mismatch_mask
    ].values
)

days_mismatch[
    "rebuilt_days_since_last_goal"
] = (
    rebuilt_days[
        nan_mismatch_mask
    ].values
)


print(
    "DAYS-SINCE-LAST-GOAL MISMATCHES"
)

print("=" * 115)

print(
    days_mismatch
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Inspect Ronaldo's opening career sequence
# ---------------------------------------------------------

print(
    "\nOPENING CAREER SEQUENCE"
)

print("=" * 115)

opening_rows = (
    hist[
        [
            "match_id",
            "date",
            "team",
            "opponent",
            "competition",
            "target_goals",
            "career_appearances_pre",
            "career_goals_pre",
            "days_since_last_goal",
        ]
    ]
    .head(15)
    .copy()
)

opening_rows[
    "rebuilt_days_since_last_goal"
] = (
    recursive_replay[
        "days_since_last_goal"
    ]
    .head(15)
    .values
)

print(
    opening_rows
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Locate first career goal
# ---------------------------------------------------------

first_goal_idx = (
    hist.index[
        hist["target_goals"] > 0
    ][0]
)

first_goal = (
    hist.loc[
        first_goal_idx,
        [
            "match_id",
            "date",
            "team",
            "opponent",
            "competition",
            "target_goals",
        ],
    ]
)


print(
    "\nFIRST CAREER GOAL IN DATASET"
)

print("=" * 115)

print(
    first_goal.to_string()
)


# ---------------------------------------------------------
# Confirm all numerical values match wherever both exist
# ---------------------------------------------------------

both_present = (
    actual_days.notna()
    &
    rebuilt_days.notna()
)

max_numeric_error = float(
    (
        actual_days[
            both_present
        ]
        -
        rebuilt_days[
            both_present
        ]
    )
    .abs()
    .max()
)


print(
    "\nNUMERICAL CHECK"
)

print("=" * 115)

print(
    "NaN mismatches:",
    int(
        nan_mismatch_mask.sum()
    )
)

print(
    "Rows where both values exist:",
    int(
        both_present.sum()
    )
)

print(
    "Maximum numerical error:",
    max_numeric_error
)


assert (
    nan_mismatch_mask.sum()
    == 5
)

assert (
    max_numeric_error
    == 0.0
)


print(
    "\nDays-since-last-goal investigation: COMPLETE"
)

DAYS-SINCE-LAST-GOAL MISMATCHES
                                  match_id       date        team          opponent                 competition  target_goals  career_appearances_pre  career_goals_pre  dataset_days_since_last_goal  rebuilt_days_since_last_goal
      2002-08-14_Sporting_CP_Inter_Milan_1 2002-08-14 Sporting CP       Inter Milan Champions League Qualifying             0                       0                 0                           0.0                           NaN
2002-09-19_Sporting_CP_Partizan_Belgrade_2 2002-09-19 Sporting CP Partizan Belgrade          UEFA Europa League             0                       1                 0                           0.0                           NaN
   2002-09-30_Sporting_CP_Sporting_Braga_3 2002-09-30 Sporting CP    Sporting Braga               Primeira Liga             0                       2                 0                           0.0                           NaN
2002-10-03_Sporting_CP_Partizan_Belgrade_4 2002-10-03 Sp

In [31]:
# Apply the original pre-first-goal convention
# and perform the final recursive-engine validation.

# ---------------------------------------------------------
# Fix the five historical pre-first-goal rows
# ---------------------------------------------------------

recursive_replay_final = (
    recursive_replay.copy()
)

recursive_replay_final[
    "days_since_last_goal"
] = (
    recursive_replay_final[
        "days_since_last_goal"
    ]
    .fillna(0.0)
)


# ---------------------------------------------------------
# Final comparison across all 35 recursive features
# ---------------------------------------------------------

final_audit_rows = []


for feature in recursive_contract:

    actual = pd.to_numeric(
        hist[feature],
        errors="coerce",
    )

    rebuilt = pd.to_numeric(
        recursive_replay_final[
            feature
        ],
        errors="coerce",
    )


    nan_mismatch = (
        actual.isna()
        != rebuilt.isna()
    )


    both_present = (
        actual.notna()
        &
        rebuilt.notna()
    )


    abs_error = pd.Series(
        0.0,
        index=hist.index,
    )

    abs_error.loc[
        both_present
    ] = (
        actual.loc[
            both_present
        ]
        -
        rebuilt.loc[
            both_present
        ]
    ).abs()


    mismatch_mask = (
        nan_mismatch
        |
        (
            abs_error
            > 1e-8
        )
    )


    final_audit_rows.append(
        {
            "feature":
                feature,

            "mismatches":
                int(
                    mismatch_mask.sum()
                ),

            "max_abs_error":
                float(
                    abs_error.max()
                ),

            "mean_abs_error":
                float(
                    abs_error.mean()
                ),

            "nan_mismatches":
                int(
                    nan_mismatch.sum()
                ),
        }
    )


recursive_engine_audit = (
    pd.DataFrame(
        final_audit_rows
    )
    .sort_values(
        [
            "mismatches",
            "max_abs_error",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

perfect_features = int(
    (
        recursive_engine_audit[
            "mismatches"
        ]
        == 0
    ).sum()
)

total_mismatches = int(
    recursive_engine_audit[
        "mismatches"
    ].sum()
)

max_error = float(
    recursive_engine_audit[
        "max_abs_error"
    ].max()
)


print(
    "FINAL HISTORICAL RECURSIVE ENGINE AUDIT"
)

print("=" * 105)

print(
    recursive_engine_audit
    .to_string(
        index=False,
        formatters={
            "max_abs_error":
                lambda x: f"{x:.10f}",

            "mean_abs_error":
                lambda x: f"{x:.10f}",
        },
    )
)


print(
    "\nSUMMARY"
)

print("=" * 105)

print(
    "Historical appearances replayed:",
    len(
        hist
    )
)

print(
    "Recursive features validated:",
    len(
        recursive_contract
    )
)

print(
    "Perfectly reconstructed features:",
    perfect_features
)

print(
    "Total feature-row mismatches:",
    total_mismatches
)

print(
    "Maximum absolute error:",
    max_error
)


# ---------------------------------------------------------
# Validate the final post-match state
# ---------------------------------------------------------

final_hist = (
    hist.iloc[-1]
)

print(
    "\nFINAL HISTORICAL STATE"
)

print("=" * 105)

print(
    "Cutoff:",
    final_hist["date"].date()
)

print(
    "Historical appearances:",
    len(hist)
)

print(
    "Historical goals:",
    int(
        hist[
            "target_goals"
        ].sum()
    )
)

print(
    "Last 3 goals:",
    career_goals[-3:]
)

print(
    "Last 5 goals:",
    career_goals[-5:]
)

print(
    "Last 10 goals:",
    career_goals[-10:]
)

print(
    "Last 20 goals:",
    career_goals[-20:]
)

print(
    "Goal EWMA state:",
    round(
        goal_ewma_state,
        10,
    )
)

print(
    "Consecutive scoring games:",
    consecutive_scoring
)

print(
    "Goalless streak:",
    goalless_streak
)

print(
    "Last goal date:",
    last_goal_date.date()
)


# ---------------------------------------------------------
# Hard integrity checks
# ---------------------------------------------------------

assert (
    len(hist)
    == 1333
)

assert (
    int(
        hist[
            "target_goals"
        ].sum()
    )
    == 978
)

assert (
    len(
        recursive_contract
    )
    == 35
)

assert (
    perfect_features
    == 35
)

assert (
    total_mismatches
    == 0
)

assert (
    max_error
    <= 1e-8
)

assert (
    career_appearances
    == 1333
)

assert (
    career_goal_total
    == 978
)


print(
    "\nHistorical recursive state engine: VALIDATED"
)

FINAL HISTORICAL RECURSIVE ENGINE AUDIT
                               feature  mismatches max_abs_error mean_abs_error  nan_mismatches
                        goals_ewma_pre           0  0.0000000000   0.0000000000               0
ronaldo_goals_per_game_vs_opponent_pre           0  0.0000000000   0.0000000000               0
             opponent_meetings_log_pre           0  0.0000000000   0.0000000000               0
             season_goals_per_game_pre           0  0.0000000000   0.0000000000               0
       team_ronaldo_goals_per_game_pre           0  0.0000000000   0.0000000000               0
        competition_goals_per_game_pre           0  0.0000000000   0.0000000000               0
             career_goals_per_game_pre           0  0.0000000000   0.0000000000               0
  ronaldo_scoring_rate_vs_opponent_pre           0  0.0000000000   0.0000000000               0
              venue_goals_per_game_pre           0  0.0000000000   0.0000000000               0


In [33]:
# Reverse-engineer the fixture-known feature definitions



hist = (
    df.sort_values("date")
    .reset_index(drop=True)
    .copy()
)

hist["date"] = pd.to_datetime(
    hist["date"]
)


# =========================================================
# 1. FEATURE CONTRACT
# =========================================================

fixture_known_features = [
    "season",
    "team",
    "opponent",
    "competition",
    "competition_type",
    "venue",
    "is_national_team",
    "competition_importance",
    "age_years",
    "days_rest",
    "matches_last_7d",
    "matches_last_14d",
    "matches_last_30d",
    "month",
    "day_of_week",
    "season_progress_pct",
    "is_season_opening_phase",
    "is_season_run_in",
    "is_dec_jan_congestion",
]


print(
    "FIXTURE-KNOWN FEATURE CONTRACT"
)

print("=" * 105)

print(
    "Total:",
    len(fixture_known_features)
)

for feature in fixture_known_features:
    print(" -", feature)


# =========================================================
# 2. SIMPLE DATE-DERIVED FEATURES
# =========================================================

print(
    "\nSIMPLE DATE-DERIVED AUDIT"
)

print("=" * 105)


# Month
month_rebuilt = hist["date"].dt.month

month_mismatches = int(
    (
        pd.to_numeric(
            hist["month"],
            errors="coerce",
        )
        != month_rebuilt
    ).sum()
)

print(
    "month mismatches:",
    month_mismatches
)


# Day of week
day_name_rebuilt = (
    hist["date"]
    .dt.day_name()
)

dow_equal_full = (
    hist["day_of_week"]
    .astype(str)
    == day_name_rebuilt
).sum()

dow_equal_short = (
    hist["day_of_week"]
    .astype(str)
    == day_name_rebuilt.str[:3]
).sum()

print(
    "day_of_week full-name matches:",
    int(dow_equal_full),
    "/",
    len(hist),
)

print(
    "day_of_week 3-letter matches:",
    int(dow_equal_short),
    "/",
    len(hist),
)

print(
    "Stored day_of_week values:",
    sorted(
        hist["day_of_week"]
        .astype(str)
        .unique()
    )
)


# National-team flag
national_rebuilt = (
    hist["team"]
    .eq("Portugal")
    .astype(int)
)

national_mismatches = int(
    (
        pd.to_numeric(
            hist["is_national_team"],
            errors="coerce",
        )
        != national_rebuilt
    ).sum()
)

print(
    "is_national_team mismatches:",
    national_mismatches
)


# =========================================================
# 3. AGE FORMULA AUDIT
# =========================================================

print(
    "\nAGE FORMULA AUDIT"
)

print("=" * 105)


birth_date = pd.Timestamp(
    "1985-02-05"
)

age_days_exact = (
    hist["date"]
    - birth_date
).dt.days.astype(float)


age_candidates = {
    "days / 365.0":
        age_days_exact / 365.0,

    "days / 365.25":
        age_days_exact / 365.25,

    "days / 365.2425":
        age_days_exact / 365.2425,
}


age_actual = pd.to_numeric(
    hist["age_years"],
    errors="coerce",
)


age_audit = []

for name, values in age_candidates.items():

    err = (
        age_actual
        - values
    ).abs()

    age_audit.append(
        {
            "formula": name,
            "max_abs_error": float(
                err.max()
            ),
            "mean_abs_error": float(
                err.mean()
            ),
        }
    )


age_audit_df = (
    pd.DataFrame(age_audit)
    .sort_values("max_abs_error")
)

print(
    age_audit_df.to_string(
        index=False,
        formatters={
            "max_abs_error":
                lambda x: f"{x:.12f}",

            "mean_abs_error":
                lambda x: f"{x:.12f}",
        },
    )
)


# =========================================================
# 4. REST-DAY FORMULA AUDIT
# =========================================================

print(
    "\nREST-DAY AUDIT"
)

print("=" * 105)


previous_date = (
    hist["date"]
    .shift(1)
)

rest_rebuilt = (
    hist["date"]
    - previous_date
).dt.days.astype(float)


rest_actual = pd.to_numeric(
    hist["days_rest"],
    errors="coerce",
)


rest_nan_mismatch = (
    rest_actual.isna()
    != rest_rebuilt.isna()
)

rest_both = (
    rest_actual.notna()
    &
    rest_rebuilt.notna()
)

rest_error = (
    rest_actual[rest_both]
    - rest_rebuilt[rest_both]
).abs()


print(
    "NaN mismatches:",
    int(
        rest_nan_mismatch.sum()
    )
)

print(
    "Maximum numeric error:",
    float(
        rest_error.max()
        if len(rest_error)
        else 0.0
    )
)

print(
    "First 8 stored/rebuilt values:"
)

print(
    pd.DataFrame(
        {
            "date":
                hist["date"].head(8),

            "stored_days_rest":
                rest_actual.head(8),

            "rebuilt_date_difference":
                rest_rebuilt.head(8),
        }
    ).to_string(
        index=False
    )
)


# =========================================================
# 5. CONGESTION WINDOW AUDIT
# =========================================================

print(
    "\nMATCH-CONGESTION WINDOW AUDIT"
)

print("=" * 105)


def rebuild_prior_matches_within_days(
    dates,
    window_days,
    inclusive=True,
):
    rebuilt = []

    for i, current_date in enumerate(dates):

        previous = dates.iloc[:i]

        if inclusive:
            count = (
                (
                    current_date
                    - previous
                ).dt.days
                <= window_days
            ).sum()

        else:
            count = (
                (
                    current_date
                    - previous
                ).dt.days
                < window_days
            ).sum()

        rebuilt.append(
            int(count)
        )

    return pd.Series(
        rebuilt,
        index=dates.index,
    )


window_results = []


for days, column in [
    (7, "matches_last_7d"),
    (14, "matches_last_14d"),
    (30, "matches_last_30d"),
]:

    actual = pd.to_numeric(
        hist[column],
        errors="coerce",
    )

    inclusive = (
        rebuild_prior_matches_within_days(
            hist["date"],
            days,
            inclusive=True,
        )
    )

    exclusive = (
        rebuild_prior_matches_within_days(
            hist["date"],
            days,
            inclusive=False,
        )
    )

    inclusive_mismatches = int(
        (
            actual
            != inclusive
        ).sum()
    )

    exclusive_mismatches = int(
        (
            actual
            != exclusive
        ).sum()
    )

    window_results.append(
        {
            "feature":
                column,

            "<= window mismatches":
                inclusive_mismatches,

            "< window mismatches":
                exclusive_mismatches,
        }
    )


print(
    pd.DataFrame(
        window_results
    ).to_string(
        index=False
    )
)


# =========================================================
# 6. COMPETITION IMPORTANCE MAPPING
# =========================================================

print(
    "\nCOMPETITION IMPORTANCE AUDIT"
)

print("=" * 105)


competition_importance_counts = (
    hist.groupby(
        "competition"
    )[
        "competition_importance"
    ]
    .nunique()
    .sort_values(
        ascending=False
    )
)

non_constant_competitions = (
    competition_importance_counts[
        competition_importance_counts
        > 1
    ]
)

print(
    "Competitions:",
    hist["competition"].nunique()
)

print(
    "Competitions with >1 importance value:",
    len(
        non_constant_competitions
    )
)


competition_importance_map = (
    hist.groupby(
        [
            "competition",
            "competition_type",
        ],
        as_index=False,
    )[
        "competition_importance"
    ]
    .agg(
        [
            "min",
            "max",
            "median",
            "count",
        ]
    )
    .reset_index()
    .sort_values(
        [
            "median",
            "competition",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

print(
    "\nHistorical competition importance map:"
)

print(
    competition_importance_map
    .to_string(
        index=False
    )
)


# =========================================================
# 7. SEASON-PROGRESS / PHASE FLAGS
# =========================================================

print(
    "\nSEASON-PROGRESS AUDIT"
)

print("=" * 105)


season_summary = (
    hist.groupby(
        "season",
        as_index=False,
    )
    .agg(
        first_date=(
            "date",
            "min",
        ),
        last_date=(
            "date",
            "max",
        ),
        appearances=(
            "date",
            "size",
        ),
        progress_min=(
            "season_progress_pct",
            "min",
        ),
        progress_max=(
            "season_progress_pct",
            "max",
        ),
        opening_rows=(
            "is_season_opening_phase",
            "sum",
        ),
        run_in_rows=(
            "is_season_run_in",
            "sum",
        ),
    )
)


print(
    season_summary
    .tail(10)
    .to_string(
        index=False
    )
)


# Check whether phase flags are simple thresholds
progress = pd.to_numeric(
    hist["season_progress_pct"],
    errors="coerce",
)

opening = pd.to_numeric(
    hist["is_season_opening_phase"],
    errors="coerce",
)

run_in = pd.to_numeric(
    hist["is_season_run_in"],
    errors="coerce",
)


print(
    "\nOpening-phase progress range:"
)

print(
    "flag=1 min/max:",
    float(
        progress[
            opening == 1
        ].min()
    ),
    float(
        progress[
            opening == 1
        ].max()
    ),
)

print(
    "flag=0 min:",
    float(
        progress[
            opening == 0
        ].min()
    ),
)


print(
    "\nRun-in progress range:"
)

print(
    "flag=1 min/max:",
    float(
        progress[
            run_in == 1
        ].min()
    ),
    float(
        progress[
            run_in == 1
        ].max()
    ),
)

print(
    "flag=0 max:",
    float(
        progress[
            run_in == 0
        ].max()
    ),
)


# =========================================================
# 8. DEC/JAN CONGESTION FLAG
# =========================================================

print(
    "\nDEC/JAN CONGESTION FLAG AUDIT"
)

print("=" * 105)


dec_jan_simple = (
    hist["date"]
    .dt.month
    .isin(
        [12, 1]
    )
    .astype(int)
)

dec_jan_actual = pd.to_numeric(
    hist["is_dec_jan_congestion"],
    errors="coerce",
)

print(
    "Simple Dec/Jan month-rule mismatches:",
    int(
        (
            dec_jan_actual
            != dec_jan_simple
        ).sum()
    )
)


print(
    "\nFlag counts by month:"
)

print(
    pd.crosstab(
        hist["date"].dt.month,
        dec_jan_actual,
    ).to_string()
)


# =========================================================
# 9. FINAL STATUS
# =========================================================

print(
    "\nFIXTURE-KNOWN AUDIT COMPLETE"
)

print("=" * 105)

print(
    "No future rows have been generated yet."
)

print(
    "This step only recovers the historical feature-engineering conventions."
)

FIXTURE-KNOWN FEATURE CONTRACT
Total: 19
 - season
 - team
 - opponent
 - competition
 - competition_type
 - venue
 - is_national_team
 - competition_importance
 - age_years
 - days_rest
 - matches_last_7d
 - matches_last_14d
 - matches_last_30d
 - month
 - day_of_week
 - season_progress_pct
 - is_season_opening_phase
 - is_season_run_in
 - is_dec_jan_congestion

SIMPLE DATE-DERIVED AUDIT
month mismatches: 0
day_of_week full-name matches: 1333 / 1333
day_of_week 3-letter matches: 0 / 1333
Stored day_of_week values: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
is_national_team mismatches: 0

AGE FORMULA AUDIT
        formula  max_abs_error mean_abs_error
days / 365.2425 0.000049948322 0.000024585971
  days / 365.25 0.000891444216 0.000603653942
   days / 365.0 0.027623287671 0.019515125014

REST-DAY AUDIT
NaN mismatches: 1
Maximum numeric error: 0.0
First 8 stored/rebuilt values:
      date  stored_days_rest  rebuilt_date_difference
2002-08-14          

In [34]:
# Recover the exact historical season_progress_pct formula



hist = (
    df.sort_values("date")
    .reset_index(drop=True)
    .copy()
)

hist["date"] = pd.to_datetime(
    hist["date"]
)


def season_start_from_label(season_label):
    start_year = int(
        str(season_label).split("/")[0]
    )

    return pd.Timestamp(
        year=start_year,
        month=8,
        day=1,
    )


def season_end_from_label(season_label):
    start_year = int(
        str(season_label).split("/")[0]
    )

    return pd.Timestamp(
        year=start_year + 1,
        month=7,
        day=31,
    )


def next_season_start_from_label(season_label):
    start_year = int(
        str(season_label).split("/")[0]
    )

    return pd.Timestamp(
        year=start_year + 1,
        month=8,
        day=1,
    )


actual = pd.to_numeric(
    hist["season_progress_pct"],
    errors="coerce",
)


# ---------------------------------------------------------
# Candidate formulas
# ---------------------------------------------------------

candidate_results = []


# Candidate A:
# elapsed days / days from Aug 1 to Jul 31
candidate_a = []

# Candidate B:
# elapsed days / days from Aug 1 to next Aug 1
candidate_b = []

# Candidate C:
# inclusive day position / total inclusive season days
candidate_c = []

# Candidate D:
# elapsed days / 365 or 366 depending on season
candidate_d = []


for _, row in hist.iterrows():

    start = season_start_from_label(
        row["season"]
    )

    end = season_end_from_label(
        row["season"]
    )

    next_start = next_season_start_from_label(
        row["season"]
    )

    elapsed = (
        row["date"]
        - start
    ).days

    days_to_july31 = (
        end
        - start
    ).days

    days_to_next_aug1 = (
        next_start
        - start
    ).days

    inclusive_total_days = (
        days_to_july31
        + 1
    )

    # A
    value_a = (
        elapsed
        / days_to_july31
    )

    # B
    value_b = (
        elapsed
        / days_to_next_aug1
    )

    # C
    value_c = (
        (
            elapsed + 1
        )
        / inclusive_total_days
    )

    # D
    season_days = (
        366
        if days_to_next_aug1 == 366
        else 365
    )

    value_d = (
        elapsed
        / season_days
    )

    candidate_a.append(
        round(
            min(max(value_a, 0.0), 1.0),
            4,
        )
    )

    candidate_b.append(
        round(
            min(max(value_b, 0.0), 1.0),
            4,
        )
    )

    candidate_c.append(
        round(
            min(max(value_c, 0.0), 1.0),
            4,
        )
    )

    candidate_d.append(
        round(
            min(max(value_d, 0.0), 1.0),
            4,
        )
    )


candidates = {
    "elapsed / (Jul31 - Aug1)":
        pd.Series(candidate_a),

    "elapsed / (next Aug1 - Aug1)":
        pd.Series(candidate_b),

    "(elapsed + 1) / inclusive season days":
        pd.Series(candidate_c),

    "elapsed / 365-or-366":
        pd.Series(candidate_d),
}


# ---------------------------------------------------------
# Audit candidates
# ---------------------------------------------------------

for name, rebuilt in candidates.items():

    error = (
        actual
        - rebuilt
    ).abs()

    mismatches = int(
        (
            error > 1e-8
        ).sum()
    )

    candidate_results.append(
        {
            "formula": name,
            "mismatches": mismatches,
            "max_abs_error": float(
                error.max()
            ),
            "mean_abs_error": float(
                error.mean()
            ),
        }
    )


audit = (
    pd.DataFrame(
        candidate_results
    )
    .sort_values(
        [
            "mismatches",
            "max_abs_error",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "SEASON PROGRESS FORMULA AUDIT"
)

print("=" * 110)

print(
    audit.to_string(
        index=False,
        formatters={
            "max_abs_error":
                lambda x: f"{x:.10f}",

            "mean_abs_error":
                lambda x: f"{x:.10f}",
        },
    )
)


# ---------------------------------------------------------
# Best candidate inspection
# ---------------------------------------------------------

best_name = audit.iloc[0]["formula"]

best = candidates[
    best_name
]


best_error = (
    actual
    - best
).abs()


problem_mask = (
    best_error > 1e-8
)


print(
    "\nBEST CANDIDATE"
)

print("=" * 110)

print(
    "Formula:",
    best_name
)

print(
    "Mismatches:",
    int(
        problem_mask.sum()
    )
)

print(
    "Max error:",
    float(
        best_error.max()
    )
)


if problem_mask.any():

    comparison = (
        hist.loc[
            problem_mask,
            [
                "date",
                "season",
                "season_progress_pct",
                "is_season_opening_phase",
                "is_season_run_in",
            ],
        ]
        .copy()
    )

    comparison[
        "rebuilt_progress"
    ] = best[
        problem_mask
    ].values

    comparison[
        "abs_error"
    ] = best_error[
        problem_mask
    ].values


    print(
        "\nFIRST 30 REMAINING MISMATCHES"
    )

    print("-" * 110)

    print(
        comparison
        .head(30)
        .to_string(
            index=False
        )
    )


# ---------------------------------------------------------
# Show season lengths for diagnosis
# ---------------------------------------------------------

season_lengths = []

for season in sorted(
    hist["season"].unique()
):

    start = season_start_from_label(
        season
    )

    next_start = next_season_start_from_label(
        season
    )

    season_lengths.append(
        {
            "season": season,
            "days_Aug1_to_next_Aug1":
                (
                    next_start
                    - start
                ).days,
        }
    )


print(
    "\nSEASON LENGTHS"
)

print("=" * 110)

print(
    pd.DataFrame(
        season_lengths
    ).to_string(
        index=False
    )
)

SEASON PROGRESS FORMULA AUDIT
                              formula  mismatches max_abs_error mean_abs_error
             elapsed / (Jul31 - Aug1)           0  0.0000000000   0.0000000000
(elapsed + 1) / inclusive season days        1327  0.0027000000   0.0014887472
         elapsed / (next Aug1 - Aug1)        1327  0.0028000000   0.0012480870
                 elapsed / 365-or-366        1327  0.0028000000   0.0012480870

BEST CANDIDATE
Formula: elapsed / (Jul31 - Aug1)
Mismatches: 0
Max error: 0.0

SEASON LENGTHS
 season  days_Aug1_to_next_Aug1
2002/03                     365
2003/04                     366
2004/05                     365
2005/06                     365
2006/07                     365
2007/08                     366
2008/09                     365
2009/10                     365
2010/11                     365
2011/12                     366
2012/13                     365
2013/14                     365
2014/15                     365
2015/16                     366


In [39]:
for competition in ["King Cup", "Nations League"]:
    print(f"\n{competition}")
    print(
        df.loc[
            df["competition"]
            .str.contains(competition, case=False, na=False),
            ["competition", "competition_type", "competition_importance"]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


King Cup
Empty DataFrame
Columns: [competition, competition_type, competition_importance]
Index: []

Nations League
        competition competition_type  competition_importance
UEFA Nations League    international                    0.82


In [41]:
print(
    df[
        ["competition", "competition_type", "competition_importance"]
    ]
    .drop_duplicates()
    .sort_values("competition")
    .to_string(index=False)
)

                     competition      competition_type  competition_importance
            AFC Champions League      continental_club                    0.84
      AFC Champions League Elite      continental_club                    0.84
        AFC Champions League Two      continental_club                    0.84
         Arab Club Champions Cup domestic_or_other_cup                    0.72
     Champions League Qualifying      continental_club                    0.72
              Confederations Cup         international                    0.72
                    Copa del Rey domestic_or_other_cup                    0.72
                    Coppa Italia domestic_or_other_cup                    0.72
               Euro Championship         international                    0.95
                 Euros Qualifier         international                    0.76
             FA Community Shield domestic_or_other_cup                    0.72
                          FA Cup domestic_or_other_c

In [43]:
# Standardize future competition names to match historical training data

competition_name_fixes = {
    "King Cup": "King's Cup",
    "Nations League": "UEFA Nations League",
}

future_fixtures["competition"] = (
    future_fixtures["competition"]
    .replace(competition_name_fixes)
)

print(
    future_fixtures[
        future_fixtures["competition"].isin(competition_name_fixes.values())
    ][["date", "competition"]].to_string(index=False)
)

      date         competition
2026-09-24 UEFA Nations League
2026-09-27 UEFA Nations League
2026-10-01 UEFA Nations League
2026-10-04 UEFA Nations League
2026-11-14 UEFA Nations League
2026-11-17 UEFA Nations League
2026-12-01          King's Cup


In [45]:
# Finalize the reusable fixture-known feature engine



BIRTH_DATE = pd.Timestamp("1985-02-05")


# ---------------------------------------------------------
# Competition metadata recovered from historical data
# ---------------------------------------------------------

competition_contract = (
    hist[
        [
            "competition",
            "competition_type",
            "competition_importance",
        ]
    ]
    .drop_duplicates()
    .set_index("competition")
)


competition_type_map = (
    competition_contract[
        "competition_type"
    ].to_dict()
)

competition_importance_map = (
    competition_contract[
        "competition_importance"
    ].to_dict()
)


# ---------------------------------------------------------
# Season helpers
# ---------------------------------------------------------

def get_season_bounds(season_label):

    start_year = int(
        str(season_label).split("/")[0]
    )

    season_start = pd.Timestamp(
        year=start_year,
        month=8,
        day=1,
    )

    season_end = pd.Timestamp(
        year=start_year + 1,
        month=7,
        day=31,
    )

    return season_start, season_end


def calculate_season_progress(
    date,
    season_label,
):

    date = pd.Timestamp(date)

    start, end = get_season_bounds(
        season_label
    )

    denominator = (
        end - start
    ).days

    elapsed = (
        date - start
    ).days

    progress = (
        elapsed / denominator
    )

    progress = np.clip(
        progress,
        0.0,
        1.0,
    )

    return round(
        float(progress),
        4,
    )


# ---------------------------------------------------------
# Appearance-schedule helpers
#
# IMPORTANT:
# These operate on Ronaldo APPEARANCES, not every team game.
# ---------------------------------------------------------

def calculate_days_rest(
    date,
    prior_appearance_dates,
):

    date = pd.Timestamp(date)

    if len(prior_appearance_dates) == 0:
        return 0.0

    previous_date = pd.Timestamp(
        prior_appearance_dates[-1]
    )

    return float(
        (
            date - previous_date
        ).days
    )


def count_recent_appearances(
    date,
    prior_appearance_dates,
    window_days,
):

    date = pd.Timestamp(date)

    return int(
        sum(
            0
            <
            (
                date
                - pd.Timestamp(previous_date)
            ).days
            <= window_days

            for previous_date
            in prior_appearance_dates
        )
    )


# ---------------------------------------------------------
# Fixture-known feature generator
# ---------------------------------------------------------

def build_fixture_known_features(
    date,
    season,
    team,
    opponent,
    competition,
    venue,
    prior_appearance_dates,
):

    date = pd.Timestamp(date)

    if competition not in competition_type_map:

        raise ValueError(
            f"Unknown competition: {competition}"
        )

    progress = (
        calculate_season_progress(
            date,
            season,
        )
    )

    features = {

        # Categorical fixture identity
        "season":
            season,

        "team":
            team,

        "opponent":
            opponent,

        "competition":
            competition,

        "competition_type":
            competition_type_map[
                competition
            ],

        "venue":
            venue,

        # Known numeric/context features
        "is_national_team":
            int(
                team == "Portugal"
            ),

        "competition_importance":
            float(
                competition_importance_map[
                    competition
                ]
            ),

        "age_years":
            round(
                (
                    (
                        date
                        - BIRTH_DATE
                    ).days
                    / 365.2425
                ),
                4,
            ),

        "days_rest":
            calculate_days_rest(
                date,
                prior_appearance_dates,
            ),

        "matches_last_7d":
            count_recent_appearances(
                date,
                prior_appearance_dates,
                7,
            ),

        "matches_last_14d":
            count_recent_appearances(
                date,
                prior_appearance_dates,
                14,
            ),

        "matches_last_30d":
            count_recent_appearances(
                date,
                prior_appearance_dates,
                30,
            ),

        "month":
            int(
                date.month
            ),

        "day_of_week":
            date.day_name(),

        "season_progress_pct":
            progress,

        "is_season_opening_phase":
            int(
                progress <= 0.20
            ),

        "is_season_run_in":
            int(
                progress >= 0.80
            ),

        "is_dec_jan_congestion":
            int(
                date.month
                in [12, 1]
            ),
    }

    return features


# =========================================================
# HISTORICAL REPLAY
# =========================================================

fixture_rebuilt_rows = []

prior_appearance_dates = []


for _, row in hist.iterrows():

    rebuilt = (
        build_fixture_known_features(
            date=row["date"],
            season=row["season"],
            team=row["team"],
            opponent=row["opponent"],
            competition=row["competition"],
            venue=row["venue"],
            prior_appearance_dates=prior_appearance_dates,
        )
    )

    fixture_rebuilt_rows.append(
        rebuilt
    )

    # Update only AFTER generating the pre-match row.
    prior_appearance_dates.append(
        pd.Timestamp(
            row["date"]
        )
    )


fixture_replay = pd.DataFrame(
    fixture_rebuilt_rows
)


# =========================================================
# COMPARE ALL 19 FEATURES
# =========================================================

fixture_audit_rows = []


for feature in fixture_known_features:

    actual = hist[
        feature
    ]

    rebuilt = fixture_replay[
        feature
    ]


    # Categorical/string features
    if (
        actual.dtype == "object"
        or rebuilt.dtype == "object"
    ):

        mismatch_count = int(
            (
                actual.astype(str)
                !=
                rebuilt.astype(str)
            ).sum()
        )

        max_error = np.nan


    # Numeric features
    else:

        actual_num = pd.to_numeric(
            actual,
            errors="coerce",
        )

        rebuilt_num = pd.to_numeric(
            rebuilt,
            errors="coerce",
        )

        nan_mismatch = (
            actual_num.isna()
            != rebuilt_num.isna()
        )

        both_present = (
            actual_num.notna()
            &
            rebuilt_num.notna()
        )

        error = pd.Series(
            0.0,
            index=hist.index,
        )

        error.loc[
            both_present
        ] = (
            actual_num.loc[
                both_present
            ]
            -
            rebuilt_num.loc[
                both_present
            ]
        ).abs()

        mismatch_count = int(
            (
                nan_mismatch
                |
                (
                    error > 1e-8
                )
            ).sum()
        )

        max_error = float(
            error.max()
        )


    fixture_audit_rows.append(
        {
            "feature":
                feature,

            "mismatches":
                mismatch_count,

            "max_abs_error":
                max_error,
        }
    )


fixture_engine_audit = (
    pd.DataFrame(
        fixture_audit_rows
    )
    .sort_values(
        [
            "mismatches",
            "feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# REPORT
# =========================================================

print(
    "FINAL FIXTURE-KNOWN ENGINE AUDIT"
)

print("=" * 100)

print(
    fixture_engine_audit
    .to_string(
        index=False,
        formatters={
            "max_abs_error":
                lambda x: (
                    ""
                    if pd.isna(x)
                    else f"{x:.10f}"
                )
        },
    )
)


perfect_features = int(
    (
        fixture_engine_audit[
            "mismatches"
        ]
        == 0
    ).sum()
)

total_mismatches = int(
    fixture_engine_audit[
        "mismatches"
    ].sum()
)


print(
    "\nSUMMARY"
)

print("=" * 100)

print(
    "Historical appearances replayed:",
    len(hist)
)

print(
    "Fixture-known features validated:",
    len(
        fixture_known_features
    )
)

print(
    "Perfectly reconstructed:",
    perfect_features
)

print(
    "Total feature-row mismatches:",
    total_mismatches
)


# =========================================================
# FUTURE CALENDAR CONTRACT
# =========================================================

future_unmapped = sorted(
    set(
        future_fixtures[
            "competition"
        ]
    )
    -
    set(
        competition_type_map
    )
)


print(
    "\nFUTURE CALENDAR CONTRACT"
)

print("=" * 100)

print(
    "Future fixtures:",
    len(
        future_fixtures
    )
)

print(
    "First fixture:",
    future_fixtures[
        "date"
    ].min().date()
)

print(
    "Last fixture:",
    future_fixtures[
        "date"
    ].max().date()
)

print(
    "Unmapped competitions:",
    future_unmapped
)


# =========================================================
# HARD CHECKS
# =========================================================

assert len(
    fixture_known_features
) == 19

assert len(
    fixture_replay
) == 1333

assert perfect_features == 19

assert total_mismatches == 0

assert len(
    future_unmapped
) == 0

assert len(
    future_fixtures
) == 45


print(
    "\nFixture-known feature engine: VALIDATED"
)

FINAL FIXTURE-KNOWN ENGINE AUDIT
                feature  mismatches max_abs_error
              age_years           0  0.0000000000
            competition           0           NaN
 competition_importance           0  0.0000000000
       competition_type           0           NaN
            day_of_week           0           NaN
              days_rest           0  0.0000000000
  is_dec_jan_congestion           0  0.0000000000
       is_national_team           0  0.0000000000
is_season_opening_phase           0  0.0000000000
       is_season_run_in           0  0.0000000000
       matches_last_14d           0  0.0000000000
       matches_last_30d           0  0.0000000000
        matches_last_7d           0  0.0000000000
                  month           0  0.0000000000
               opponent           0           NaN
                 season           0           NaN
    season_progress_pct           0  0.0000000000
                   team           0           NaN
                 

## Initialize the recursive simulator at the cutoff

In [47]:
# Freeze the validated post-cutoff Ronaldo state
# and generate the recursive pre-match features for the
# first future fixture.

from copy import deepcopy


# =========================================================
# 1. FREEZE THE POST-2026-08-28 STATE
# =========================================================

initial_recursive_state = {

    # Career
    "career_appearances":
        int(career_appearances),

    "career_goal_total":
        int(career_goal_total),

    "career_goals":
        list(career_goals),

    # Recent scoring state
    "goal_ewma":
        float(goal_ewma_state),

    "consecutive_scoring":
        int(consecutive_scoring),

    "goalless_streak":
        int(goalless_streak),

    "last_goal_date":
        pd.Timestamp(last_goal_date),

    # Grouped historical state
    "season_state":
        deepcopy(season_state),

    "team_state":
        deepcopy(team_state),

    "opponent_state":
        deepcopy(opponent_state),

    "competition_state":
        deepcopy(competition_state),

    "venue_state":
        deepcopy(venue_state),

    # Appearance dates are needed for rest/congestion features
    "appearance_dates":
        hist["date"]
        .tolist(),
}


# =========================================================
# 2. RECURSIVE FEATURE EXTRACTOR
# =========================================================

def build_recursive_features(
    state,
    date,
    season,
    team,
    opponent,
    competition,
    venue,
):
    """
    Build all 35 Ronaldo-recursive PRE-MATCH features
    from the current simulator state.

    This function does not mutate the state.
    """

    date = pd.Timestamp(date)

    career_history = state[
        "career_goals"
    ]

    career_apps = state[
        "career_appearances"
    ]

    career_goal_total = state[
        "career_goal_total"
    ]


    season_s = state[
        "season_state"
    ][season]

    team_s = state[
        "team_state"
    ][team]

    opponent_s = state[
        "opponent_state"
    ][opponent]

    competition_s = state[
        "competition_state"
    ][competition]

    venue_s = state[
        "venue_state"
    ][venue]


    # -----------------------------------------------------
    # Career
    # -----------------------------------------------------

    features = {

        "career_appearances_pre":
            career_apps,

        "career_goals_pre":
            career_goal_total,

        "career_goals_per_game_pre":
            safe_rate(
                career_goal_total,
                career_apps,
            ),

        "no_prior_career_goal_pre":
            int(
                career_goal_total == 0
            ),

        "first_career_appearance":
            int(
                career_apps == 0
            ),
    }


    # -----------------------------------------------------
    # Recent scoring
    # -----------------------------------------------------

    features.update(
        {
            "goals_last_3":
                rolling_sum(
                    career_history,
                    3,
                ),

            "goals_per_game_last_5":
                rolling_mean(
                    career_history,
                    5,
                ),

            "goals_per_game_last_10":
                rolling_mean(
                    career_history,
                    10,
                ),

            "goals_per_game_last_20":
                rolling_mean(
                    career_history,
                    20,
                ),

            "goals_ewma_pre":
                float(
                    state[
                        "goal_ewma"
                    ]
                ),

            "scoring_rate_last_5":
                rolling_scoring_rate(
                    career_history,
                    5,
                ),

            "scoring_rate_last_10":
                rolling_scoring_rate(
                    career_history,
                    10,
                ),

            "multi_goal_rate_last_10":
                rolling_multi_goal_rate(
                    career_history,
                    10,
                ),

            "consecutive_scoring_games_pre":
                int(
                    state[
                        "consecutive_scoring"
                    ]
                ),

            "goalless_streak_games_pre":
                int(
                    state[
                        "goalless_streak"
                    ]
                ),

            "days_since_last_goal":
                (
                    0.0
                    if state[
                        "last_goal_date"
                    ] is None
                    else float(
                        (
                            date
                            -
                            state[
                                "last_goal_date"
                            ]
                        ).days
                    )
                ),
        }
    )


    # -----------------------------------------------------
    # Season
    # -----------------------------------------------------

    features.update(
        {
            "season_appearances_pre":
                season_s["apps"],

            "season_goals_pre":
                season_s["goals"],

            "season_goals_per_game_pre":
                safe_rate(
                    season_s["goals"],
                    season_s["apps"],
                ),
        }
    )


    # -----------------------------------------------------
    # Team
    # -----------------------------------------------------

    features.update(
        {
            "team_appearances_with_ronaldo_pre":
                team_s["apps"],

            "team_goals_by_ronaldo_pre":
                team_s["goals"],

            "team_ronaldo_goals_per_game_pre":
                safe_rate(
                    team_s["goals"],
                    team_s["apps"],
                ),
        }
    )


    # -----------------------------------------------------
    # Opponent
    # -----------------------------------------------------

    features.update(
        {
            "opponent_prior_meetings_pre":
                opponent_s["apps"],

            "ronaldo_goals_vs_opponent_pre":
                opponent_s["goals"],

            "ronaldo_goals_per_game_vs_opponent_pre":
                safe_rate(
                    opponent_s["goals"],
                    opponent_s["apps"],
                ),

            "ronaldo_scoring_rate_vs_opponent_pre":
                safe_rate(
                    sum(
                        g > 0
                        for g in opponent_s[
                            "goal_history"
                        ]
                    ),
                    opponent_s["apps"],
                ),

            "ronaldo_goals_last_3_vs_opponent_pre":
                rolling_sum(
                    opponent_s[
                        "goal_history"
                    ],
                    3,
                ),

            "first_time_opponent_pre":
                int(
                    opponent_s["apps"] == 0
                ),

            "opponent_meetings_log_pre":
                float(
                    np.log1p(
                        opponent_s["apps"]
                    )
                ),
        }
    )


    # -----------------------------------------------------
    # Competition
    # -----------------------------------------------------

    features.update(
        {
            "competition_appearances_pre":
                competition_s["apps"],

            "competition_goals_pre":
                competition_s["goals"],

            "competition_goals_per_game_pre":
                safe_rate(
                    competition_s["goals"],
                    competition_s["apps"],
                ),
        }
    )


    # -----------------------------------------------------
    # Venue
    # -----------------------------------------------------

    features.update(
        {
            "venue_appearances_pre":
                venue_s["apps"],

            "venue_goals_pre":
                venue_s["goals"],

            "venue_goals_per_game_pre":
                safe_rate(
                    venue_s["goals"],
                    venue_s["apps"],
                ),
        }
    )


    return features


# =========================================================
# 3. HARD CHECK THE FROZEN CUTOFF STATE
# =========================================================

print(
    "FROZEN RECURSIVE STATE"
)

print("=" * 100)

print(
    "Career appearances:",
    initial_recursive_state[
        "career_appearances"
    ]
)

print(
    "Career goals:",
    initial_recursive_state[
        "career_goal_total"
    ]
)

print(
    "Goals remaining:",
    1000
    -
    initial_recursive_state[
        "career_goal_total"
    ]
)

print(
    "Goal EWMA:",
    round(
        initial_recursive_state[
            "goal_ewma"
        ],
        10,
    )
)

print(
    "Last goal date:",
    initial_recursive_state[
        "last_goal_date"
    ].date()
)


assert (
    initial_recursive_state[
        "career_appearances"
    ]
    == 1333
)

assert (
    initial_recursive_state[
        "career_goal_total"
    ]
    == 978
)

assert (
    initial_recursive_state[
        "last_goal_date"
    ]
    ==
    pd.Timestamp(
        "2026-08-28"
    )
)


# =========================================================
# 4. FIRST FUTURE FIXTURE
# =========================================================

first_future = (
    future_fixtures
    .sort_values("date")
    .iloc[0]
)


first_recursive_features = (
    build_recursive_features(
        state=initial_recursive_state,
        date=first_future["date"],
        season="2026/27",
        team=first_future["team"],
        opponent=first_future["opponent"],
        competition=first_future["competition"],
        venue=first_future["venue"],
    )
)


print(
    "\nFIRST FUTURE FIXTURE"
)

print("=" * 100)

print(
    "Date:",
    pd.Timestamp(
        first_future["date"]
    ).date()
)

print(
    "Team:",
    first_future["team"]
)

print(
    "Opponent:",
    first_future["opponent"]
)

print(
    "Competition:",
    first_future["competition"]
)

print(
    "Venue:",
    first_future["venue"]
)


# =========================================================
# 5. DISPLAY THE 35 RECURSIVE INPUTS
# =========================================================

first_recursive_df = pd.DataFrame(
    {
        "feature":
            recursive_contract,

        "value":
            [
                first_recursive_features[
                    feature
                ]
                for feature
                in recursive_contract
            ],
    }
)


print(
    "\nFIRST FUTURE RECURSIVE FEATURE STATE"
)

print("=" * 100)

print(
    first_recursive_df
    .to_string(
        index=False
    )
)


# =========================================================
# 6. CONTRACT CHECK
# =========================================================

assert (
    set(
        first_recursive_features
    )
    ==
    set(
        recursive_contract
    )
)

assert len(
    first_recursive_features
) == 35

assert (
    first_recursive_features[
        "career_appearances_pre"
    ]
    == 1333
)

assert (
    first_recursive_features[
        "career_goals_pre"
    ]
    == 978
)

assert (
    first_recursive_features[
        "days_since_last_goal"
    ]
    == 8.0
)


print(
    "\nInitial recursive simulator state: READY"
)

FROZEN RECURSIVE STATE
Career appearances: 1333
Career goals: 978
Goals remaining: 22
Goal EWMA: 0.6128439198
Last goal date: 2026-08-28

FIRST FUTURE FIXTURE
Date: 2026-09-05
Team: Al-Nassr
Opponent: Al-Ittihad FC
Competition: Pro League
Venue: A

FIRST FUTURE RECURSIVE FEATURE STATE
                               feature       value
                career_appearances_pre 1333.000000
                      career_goals_pre  978.000000
             career_goals_per_game_pre    0.733683
              no_prior_career_goal_pre    0.000000
               first_career_appearance    0.000000
                          goals_last_3    2.000000
                 goals_per_game_last_5    0.600000
                goals_per_game_last_10    0.500000
                goals_per_game_last_20    0.550000
                        goals_ewma_pre    0.612844
                   scoring_rate_last_5    0.600000
                  scoring_rate_last_10    0.400000
               multi_goal_rate_last_10    0.100000


In [49]:
# Reconstruct cutoff external state and assemble
# the complete model row for the first future fixture.



# =========================================================
# 1. REPLAY EXTERNAL STATE THROUGH THE CUTOFF
# =========================================================

history = (
    df.sort_values("date")
    .reset_index(drop=True)
    .copy()
)

history["date"] = pd.to_datetime(
    history["date"]
)


# ---------------------------------------------------------
# Team recent-form state
# ---------------------------------------------------------

team_recent_results = defaultdict(
    lambda: deque(maxlen=10)
)

team_recent_scores = defaultdict(
    lambda: deque(maxlen=5)
)


# ---------------------------------------------------------
# Team-v-opponent H2H result/score state
# ---------------------------------------------------------

h2h_external_state = defaultdict(
    lambda: {
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "scored_matches": 0,
        "opponent_goals_for": 0.0,
        "opponent_goals_against": 0.0,
        "opponent_clean_sheets": 0,
    }
)


# ---------------------------------------------------------
# Shared Elo network
# ---------------------------------------------------------

elo_rating = defaultdict(
    lambda: 1500.0
)

elo_games = defaultdict(
    int
)

ELO_K = 20.0


def elo_expected(
    rating_a,
    rating_b,
):
    return (
        1.0
        /
        (
            1.0
            +
            10.0 ** (
                (
                    rating_b
                    - rating_a
                )
                / 400.0
            )
        )
    )


# ---------------------------------------------------------
# Assist state
# ---------------------------------------------------------

assist_history = []

assist_ewma_state = 0.0
ASSIST_EWMA_ALPHA = 0.30

season_assist_state = defaultdict(
    float
)

opponent_assist_state = defaultdict(
    float
)


# =========================================================
# 2. HISTORICAL REPLAY
# =========================================================

for _, row in history.iterrows():

    team = row["team"]
    opponent = row["opponent"]

    team_goals = float(
        row["team_goals"]
    )

    opponent_goals = float(
        row["opponent_goals"]
    )

    assists = float(
        row["ronaldo_assists"]
    )

    result = row["result"]


    # -----------------------------------------------------
    # Team recent form
    # -----------------------------------------------------

    if result == "W":
        points = 3

    elif result == "D":
        points = 1

    else:
        points = 0


    team_recent_results[
        team
    ].append(
        {
            "result": result,
            "points": points,
        }
    )


    team_recent_scores[
        team
    ].append(
        {
            "gf": team_goals,
            "ga": opponent_goals,
        }
    )


    # -----------------------------------------------------
    # Team-v-opponent H2H
    # -----------------------------------------------------

    pair = (
        team,
        opponent,
    )

    h2h = h2h_external_state[
        pair
    ]


    if result == "W":
        h2h["wins"] += 1

    elif result == "D":
        h2h["draws"] += 1

    else:
        h2h["losses"] += 1


    h2h["scored_matches"] += 1

    # These are from the OPPONENT perspective.
    h2h[
        "opponent_goals_for"
    ] += opponent_goals

    h2h[
        "opponent_goals_against"
    ] += team_goals

    if team_goals == 0:
        h2h[
            "opponent_clean_sheets"
        ] += 1


    # -----------------------------------------------------
    # Elo
    # -----------------------------------------------------

    team_rating_pre = (
        elo_rating[
            team
        ]
    )

    opponent_rating_pre = (
        elo_rating[
            opponent
        ]
    )

    expected_team = (
        elo_expected(
            team_rating_pre,
            opponent_rating_pre,
        )
    )


    if result == "W":
        actual_team = 1.0

    elif result == "D":
        actual_team = 0.5

    else:
        actual_team = 0.0


    change = (
        ELO_K
        *
        (
            actual_team
            - expected_team
        )
    )


    elo_rating[
        team
    ] = (
        team_rating_pre
        + change
    )

    elo_rating[
        opponent
    ] = (
        opponent_rating_pre
        - change
    )


    elo_games[
        team
    ] += 1

    elo_games[
        opponent
    ] += 1


    # -----------------------------------------------------
    # Assists
    # -----------------------------------------------------

    assist_history.append(
        assists
    )

    assist_ewma_state = (
        ASSIST_EWMA_ALPHA
        * assists
        +
        (
            1.0
            - ASSIST_EWMA_ALPHA
        )
        * assist_ewma_state
    )


    season_assist_state[
        row["season"]
    ] += assists


    opponent_assist_state[
        opponent
    ] += assists


# =========================================================
# 3. EXTERNAL FEATURE BUILDER
# =========================================================

def build_external_cutoff_features(
    team,
    opponent,
    season,
):
    """
    Build the 30 external-state features from the frozen
    2026-08-28 historical state.

    These values are carried forward during simulation.
    """

    # -----------------------------------------------------
    # Assists
    # -----------------------------------------------------

    career_assists = float(
        sum(
            assist_history
        )
    )

    assists_last_5 = float(
        sum(
            assist_history[-5:]
        )
    )

    assists_last_10 = float(
        sum(
            assist_history[-10:]
        )
    )


    # -----------------------------------------------------
    # Team recent form
    # -----------------------------------------------------

    recent_results = list(
        team_recent_results[
            team
        ]
    )

    last_5_results = (
        recent_results[-5:]
    )

    last_10_results = (
        recent_results[-10:]
    )


    points_5 = float(
        sum(
            item["points"]
            for item
            in last_5_results
        )
    )

    points_10 = float(
        sum(
            item["points"]
            for item
            in last_10_results
        )
    )


    win_rate_5 = (
        float(
            sum(
                item["result"] == "W"
                for item
                in last_5_results
            )
        )
        / len(last_5_results)
        if last_5_results
        else 0.0
    )


    unbeaten_rate_5 = (
        float(
            sum(
                item["result"]
                in ["W", "D"]
                for item
                in last_5_results
            )
        )
        / len(last_5_results)
        if last_5_results
        else 0.0
    )


    score_history = list(
        team_recent_scores[
            team
        ]
    )

    team_gf_5 = float(
        sum(
            item["gf"]
            for item
            in score_history[-5:]
        )
    )

    team_ga_5 = float(
        sum(
            item["ga"]
            for item
            in score_history[-5:]
        )
    )

    score_coverage_5 = int(
        len(
            score_history[-5:]
        )
    )


    # -----------------------------------------------------
    # H2H result/score state
    # -----------------------------------------------------

    pair = (
        team,
        opponent,
    )

    h2h = h2h_external_state[
        pair
    ]

    h2h_matches = (
        h2h["wins"]
        + h2h["draws"]
        + h2h["losses"]
    )

    scored_matches = (
        h2h[
            "scored_matches"
        ]
    )


    opponent_points = (
        3 * h2h["losses"]
        + 1 * h2h["draws"]
    )


    opponent_h2h_ppg = (
        opponent_points
        / h2h_matches
        if h2h_matches > 0
        else 0.0
    )


    opponent_unbeaten = (
        (
            h2h["losses"]
            + h2h["draws"]
        )
        / h2h_matches
        if h2h_matches > 0
        else 0.0
    )


    opponent_gf_rate = (
        h2h[
            "opponent_goals_for"
        ]
        / scored_matches
        if scored_matches > 0
        else 0.0
    )


    opponent_ga_rate = (
        h2h[
            "opponent_goals_against"
        ]
        / scored_matches
        if scored_matches > 0
        else 0.0
    )


    opponent_cs_rate = (
        h2h[
            "opponent_clean_sheets"
        ]
        / scored_matches
        if scored_matches > 0
        else 0.0
    )


    # -----------------------------------------------------
    # Elo
    # -----------------------------------------------------

    team_elo = float(
        elo_rating[
            team
        ]
    )

    opponent_games = int(
        elo_games[
            opponent
        ]
    )


    # Genuinely unseen opponent convention.
    if opponent_games == 0:

        opponent_elo = 1500.0
        opponent_unseen = 1

    else:

        opponent_elo = float(
            elo_rating[
                opponent
            ]
        )

        opponent_unseen = 0


    features = {

        # Assist state
        "career_assists_pre":
            career_assists,

        "assists_last_5":
            assists_last_5,

        "assists_last_10":
            assists_last_10,

        "assists_ewma_pre":
            float(
                assist_ewma_state
            ),

        "season_assists_pre":
            float(
                season_assist_state[
                    season
                ]
            ),

        "ronaldo_assists_vs_opponent_pre":
            float(
                opponent_assist_state[
                    opponent
                ]
            ),


        # Team form
        "team_points_last_5_ronaldo_apps_pre":
            points_5,

        "team_points_last_10_ronaldo_apps_pre":
            points_10,

        "team_win_rate_last_5_ronaldo_apps_pre":
            win_rate_5,

        "team_unbeaten_rate_last_5_ronaldo_apps_pre":
            unbeaten_rate_5,

        "team_goals_for_last_5_scored_matches_pre":
            team_gf_5,

        "team_goals_against_last_5_scored_matches_pre":
            team_ga_5,

        "team_score_coverage_last_5_pre":
            score_coverage_5,

        "no_team_score_history_pre":
            int(
                score_coverage_5 == 0
            ),


        # H2H result/score state
        "team_h2h_wins_pre":
            int(
                h2h["wins"]
            ),

        "team_h2h_draws_pre":
            int(
                h2h["draws"]
            ),

        "team_h2h_losses_pre":
            int(
                h2h["losses"]
            ),

        "opponent_h2h_points_per_game_pre":
            float(
                opponent_h2h_ppg
            ),

        "opponent_h2h_unbeaten_rate_pre":
            float(
                opponent_unbeaten
            ),

        "opponent_goals_for_per_h2h_scored_match_pre":
            float(
                opponent_gf_rate
            ),

        "opponent_goals_against_per_h2h_scored_match_pre":
            float(
                opponent_ga_rate
            ),

        "opponent_clean_sheet_rate_h2h_scored_match_pre":
            float(
                opponent_cs_rate
            ),

        "opponent_score_data_h2h_matches_pre":
            int(
                scored_matches
            ),

        "no_h2h_score_history_pre":
            int(
                scored_matches == 0
            ),


        # Elo
        "team_matchup_elo_pre":
            team_elo,

        "opponent_matchup_elo_pre":
            opponent_elo,

        "matchup_elo_diff_pre":
            float(
                team_elo
                - opponent_elo
            ),

        "opponent_matchup_elo_games_pre":
            opponent_games,

        "opponent_elo_unseen_pre":
            opponent_unseen,

        "opponent_elo_experience_log_pre":
            float(
                np.log1p(
                    opponent_games
                )
            ),
    }

    return features


# =========================================================
# 4. BUILD ALL THREE FEATURE GROUPS FOR SEP 5
# =========================================================

first_future = (
    future_fixtures
    .sort_values("date")
    .iloc[0]
)


first_fixture_features = (
    build_fixture_known_features(
        date=first_future["date"],
        season="2026/27",
        team=first_future["team"],
        opponent=first_future["opponent"],
        competition=first_future["competition"],
        venue=first_future["venue"],
        prior_appearance_dates=(
            initial_recursive_state[
                "appearance_dates"
            ]
        ),
    )
)


first_recursive_features = (
    build_recursive_features(
        state=initial_recursive_state,
        date=first_future["date"],
        season="2026/27",
        team=first_future["team"],
        opponent=first_future["opponent"],
        competition=first_future["competition"],
        venue=first_future["venue"],
    )
)


first_external_features = (
    build_external_cutoff_features(
        team=first_future["team"],
        opponent=first_future["opponent"],
        season="2026/27",
    )
)


# =========================================================
# 5. COMBINE INTO FULL CATBOOST ROW
# =========================================================

first_full_feature_row = {
    **first_fixture_features,
    **first_recursive_features,
    **first_external_features,
}


first_full_df = pd.DataFrame(
    [
        first_full_feature_row
    ]
)


# =========================================================
# 6. CONTRACT AUDIT
# =========================================================

full_keys = set(
    first_full_feature_row
)

catboost_expected = set(
    CATBOOST_FEATURES
)

xgb_expected = set(
    XGB_FEATURES
)


missing_catboost = sorted(
    catboost_expected
    - full_keys
)

extra_vs_catboost = sorted(
    full_keys
    - catboost_expected
)

missing_xgb = sorted(
    xgb_expected
    - full_keys
)


catboost_row = (
    first_full_df[
        CATBOOST_FEATURES
    ]
    .copy()
)


xgb_row = (
    first_full_df[
        XGB_FEATURES
    ]
    .copy()
)


print(
    "FIRST COMPLETE FUTURE FEATURE ROW"
)

print("=" * 105)

print(
    "Fixture:",
    pd.Timestamp(
        first_future["date"]
    ).date(),
    first_future["team"],
    "vs",
    first_future["opponent"],
)

print(
    "Combined features:",
    len(
        first_full_feature_row
    )
)

print(
    "CatBoost expected:",
    len(
        CATBOOST_FEATURES
    )
)

print(
    "XGBoost expected:",
    len(
        XGB_FEATURES
    )
)


print(
    "\nCONTRACT AUDIT"
)

print("=" * 105)

print(
    "Missing CatBoost features:",
    missing_catboost
)

print(
    "Extra vs CatBoost:",
    extra_vs_catboost
)

print(
    "Missing XGBoost features:",
    missing_xgb
)

print(
    "CatBoost missing values:",
    int(
        catboost_row
        .isna()
        .sum()
        .sum()
    )
)

print(
    "XGBoost missing values:",
    int(
        xgb_row
        .isna()
        .sum()
        .sum()
    )
)


# =========================================================
# 7. KEY SEP-5 STATE CHECK
# =========================================================

key_features = [
    "career_appearances_pre",
    "career_goals_pre",
    "goals_ewma_pre",
    "days_since_last_goal",
    "days_rest",
    "matches_last_7d",
    "matches_last_14d",
    "matches_last_30d",
    "team_points_last_5_ronaldo_apps_pre",
    "team_points_last_10_ronaldo_apps_pre",
    "team_goals_for_last_5_scored_matches_pre",
    "team_goals_against_last_5_scored_matches_pre",
    "team_matchup_elo_pre",
    "opponent_matchup_elo_pre",
    "matchup_elo_diff_pre",
    "team_h2h_wins_pre",
    "team_h2h_draws_pre",
    "team_h2h_losses_pre",
    "ronaldo_assists_vs_opponent_pre",
]


print(
    "\nKEY FIRST-FIXTURE FEATURES"
)

print("=" * 105)

print(
    first_full_df[
        key_features
    ]
    .T
    .rename(
        columns={
            0: "value"
        }
    )
    .to_string()
)


# =========================================================
# 8. HARD CHECKS
# =========================================================

assert len(
    first_recursive_features
) == 35

assert len(
    first_fixture_features
) == 19

assert len(
    first_external_features
) == 30

assert len(
    first_full_feature_row
) == 84

assert not missing_catboost

assert not extra_vs_catboost

assert not missing_xgb

assert (
    catboost_row
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    xgb_row
    .isna()
    .sum()
    .sum()
    == 0
)


print(
    "\nFirst complete future model row: VALIDATED"
)

FIRST COMPLETE FUTURE FEATURE ROW
Fixture: 2026-09-05 Al-Nassr vs Al-Ittihad FC
Combined features: 84
CatBoost expected: 84
XGBoost expected: 80

CONTRACT AUDIT
Missing CatBoost features: []
Extra vs CatBoost: []
Missing XGBoost features: []
CatBoost missing values: 0
XGBoost missing values: 0

KEY FIRST-FIXTURE FEATURES
                                                    value
career_appearances_pre                        1333.000000
career_goals_pre                               978.000000
goals_ewma_pre                                   0.612844
days_since_last_goal                             8.000000
days_rest                                        8.000000
matches_last_7d                                  0.000000
matches_last_14d                                 2.000000
matches_last_30d                                 3.000000
team_points_last_5_ronaldo_apps_pre             12.000000
team_points_last_10_ronaldo_apps_pre            22.000000
team_goals_for_last_5_scored_matches_pr

In [51]:
# Reload the locked Phase 7 model specifications
# before production refitting on all 1,333 cutoff appearances.


ARTIFACT_DIR = Path("../artifacts")


# =========================================================
# 1. LOAD LOCKED SPECS
# =========================================================

with open(
    ARTIFACT_DIR / "main_model_spec.json",
    "r",
    encoding="utf-8",
) as f:
    main_model_spec = json.load(f)


with open(
    ARTIFACT_DIR / "preprocessing_spec.json",
    "r",
    encoding="utf-8",
) as f:
    preprocessing_spec = json.load(f)


with open(
    ARTIFACT_DIR / "feature_spec.json",
    "r",
    encoding="utf-8",
) as f:
    feature_spec = json.load(f)


# =========================================================
# 2. PRINT THE SAVED STRUCTURE
# =========================================================

print(
    "LOCKED MODEL SPECIFICATION"
)

print("=" * 100)

print(
    json.dumps(
        main_model_spec,
        indent=2,
    )
)


print(
    "\nLOCKED PREPROCESSING SPECIFICATION"
)

print("=" * 100)

print(
    json.dumps(
        preprocessing_spec,
        indent=2,
    )
)


# =========================================================
# 3. FEATURE CONTRACT SUMMARY
# =========================================================

print(
    "\nFEATURE SPEC SUMMARY"
)

print("=" * 100)

print(
    "Top-level keys:",
    list(
        feature_spec.keys()
    )
)

print(
    "Full features:",
    len(
        feature_spec.get(
            "full_features",
            []
        )
    )
)

print(
    "Numeric features:",
    len(
        feature_spec.get(
            "numeric_features",
            []
        )
    )
)

print(
    "Categorical features:",
    len(
        feature_spec.get(
            "categorical_features",
            []
        )
    )
)


# =========================================================
# 4. CURRENT IN-MEMORY CONTRACT
# =========================================================

print(
    "\nCURRENT FORECAST FEATURE CONTRACT"
)

print("=" * 100)

print(
    "CATBOOST_FEATURES:",
    len(
        CATBOOST_FEATURES
    )
)

print(
    "XGB_FEATURES:",
    len(
        XGB_FEATURES
    )
)


print(
    "\nCatBoost first 10:"
)

print(
    CATBOOST_FEATURES[:10]
)


print(
    "\nXGBoost first 10:"
)

print(
    XGB_FEATURES[:10]
)


# =========================================================
# 5. BASIC LOCKED-MODEL CHECKS FROM PHASE 7
# =========================================================

print(
    "\nEXPECTED LOCKED PHASE-7 DECISIONS"
)

print("=" * 100)

print(
    "Primary forecast model:",
    "60% XGBoost + 40% CatBoost"
)

print(
    "CatBoost expected feature set:",
    "full"
)

print(
    "CatBoost expected loss:",
    "Poisson"
)

print(
    "CatBoost expected iterations:",
    100
)

print(
    "CatBoost expected depth:",
    2
)

print(
    "CatBoost expected learning rate:",
    0.03
)

print(
    "CatBoost expected l2_leaf_reg:",
    20
)

print(
    "\nXGBoost expected feature set:",
    "context"
)

print(
    "XGBoost expected objective:",
    "count:poisson"
)

print(
    "XGBoost expected n_estimators:",
    15
)

print(
    "XGBoost expected max_depth:",
    2
)

print(
    "XGBoost expected learning rate:",
    0.03
)

print(
    "XGBoost expected min_child_weight:",
    10
)

print(
    "XGBoost expected subsample:",
    0.9
)

print(
    "XGBoost expected colsample_bytree:",
    0.8
)

print(
    "XGBoost expected reg_alpha:",
    0
)

print(
    "XGBoost expected reg_lambda:",
    20
)


print(
    "\nNo model has been refit in this step."
)

print(
    "No future prediction has been made yet."
)

LOCKED MODEL SPECIFICATION
{
  "phase": "Phase 7 - Main Models",
  "selection_policy": {
    "primary_metric": "macro_poisson_deviance",
    "secondary_checks": [
      "pooled_poisson_deviance",
      "mae",
      "rmse",
      "mean_bias",
      "fold_stability"
    ],
    "final_test_used_for_selection": false
  },
  "selected_ensemble": {
    "xgboost_weight": 0.6,
    "catboost_weight": 0.4,
    "macro_poisson_deviance": 1.106496,
    "pooled_poisson_deviance": 1.096307,
    "pooled_mae": 0.701696,
    "pooled_bias": 0.007642
  },
  "xgboost": {
    "feature_set": "context",
    "excluded_identity_categoricals": [
      "season",
      "team",
      "opponent",
      "competition"
    ],
    "params": {
      "objective": "count:poisson",
      "n_estimators": 15,
      "max_depth": 2,
      "learning_rate": 0.03,
      "min_child_weight": 10.0,
      "subsample": 0.9,
      "colsample_bytree": 0.8,
      "reg_alpha": 0.0,
      "reg_lambda": 20.0,
      "random_state": 42,
      

In [53]:
# Production refit on all 1,333 cutoff appearances
# and first prospective prediction for 2026-09-05.


from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


# =========================================================
# 1. LOCKED FEATURE SETS FROM SAVED SPEC
# =========================================================

FULL_FEATURES = (
    preprocessing_spec[
        "feature_sets"
    ]["full"]
)

CONTEXT_FEATURES = (
    preprocessing_spec[
        "feature_sets"
    ]["context"]
)

STABLE_CATEGORICALS = (
    preprocessing_spec[
        "stable_categoricals"
    ]
)

ALL_CATEGORICALS = (
    feature_spec[
        "categorical_features"
    ]
)


print(
    "PRODUCTION REFIT CONTRACT"
)

print("=" * 100)

print(
    "Historical rows:",
    len(df)
)

print(
    "Full features:",
    len(FULL_FEATURES)
)

print(
    "Context features:",
    len(CONTEXT_FEATURES)
)

print(
    "Stable categoricals:",
    STABLE_CATEGORICALS
)

print(
    "All categoricals:",
    ALL_CATEGORICALS
)


# =========================================================
# 2. TRAINING DATA
# =========================================================

production_df = (
    df.sort_values("date")
    .reset_index(drop=True)
    .copy()
)


X_full = (
    production_df[
        FULL_FEATURES
    ]
    .copy()
)

X_context = (
    production_df[
        CONTEXT_FEATURES
    ]
    .copy()
)

y = (
    production_df[
        "target_goals"
    ]
    .astype(float)
    .copy()
)


# CatBoost categorical columns must be strings.
for col in ALL_CATEGORICALS:

    X_full[col] = (
        X_full[col]
        .astype(str)
    )


# =========================================================
# 3. XGBOOST PREPROCESSING
#
# Phase 5 policy:
# tree numeric -> unchanged
# tree categoricals -> model-specific encoding
#
# For XGBoost context model, stable categoricals are OHE.
# =========================================================

xgb_numeric_features = [
    col
    for col
    in CONTEXT_FEATURES
    if col
    not in STABLE_CATEGORICALS
]


xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            STABLE_CATEGORICALS,
        ),
        (
            "numeric",
            "passthrough",
            xgb_numeric_features,
        ),
    ],
    remainder="drop",
)


X_context_transformed = (
    xgb_preprocessor.fit_transform(
        X_context
    )
)


print(
    "\nXGBOOST DESIGN MATRIX"
)

print("=" * 100)

print(
    "Raw context shape:",
    X_context.shape
)

print(
    "Transformed shape:",
    X_context_transformed.shape
)


# =========================================================
# 4. REFIT LOCKED XGBOOST
# =========================================================

xgb_params = (
    main_model_spec[
        "xgboost"
    ]["params"]
    .copy()
)


production_xgb = XGBRegressor(
    **xgb_params
)


production_xgb.fit(
    X_context_transformed,
    y,
)


print(
    "\nLocked XGBoost refit: COMPLETE"
)


# =========================================================
# 5. REFIT LOCKED CATBOOST
# =========================================================

cat_params = (
    main_model_spec[
        "catboost"
    ]["params"]
    .copy()
)


cat_feature_indices = [
    X_full.columns.get_loc(
        col
    )
    for col
    in ALL_CATEGORICALS
]


production_catboost = CatBoostRegressor(
    **cat_params
)


production_catboost.fit(
    X_full,
    y,
    cat_features=cat_feature_indices,
)


print(
    "Locked CatBoost refit: COMPLETE"
)


# =========================================================
# 6. PREPARE SEP-5 FUTURE ROW
# =========================================================

future_xgb_raw = (
    first_full_df[
        CONTEXT_FEATURES
    ]
    .copy()
)


future_xgb_transformed = (
    xgb_preprocessor.transform(
        future_xgb_raw
    )
)


future_cat = (
    first_full_df[
        FULL_FEATURES
    ]
    .copy()
)


for col in ALL_CATEGORICALS:

    future_cat[col] = (
        future_cat[col]
        .astype(str)
    )


# =========================================================
# 7. FIRST PROSPECTIVE PREDICTION
# =========================================================

lambda_xgb = float(
    production_xgb.predict(
        future_xgb_transformed
    )[0]
)


lambda_cat = float(
    production_catboost.predict(
        future_cat
    )[0]
)


XGB_WEIGHT = float(
    main_model_spec[
        "selected_ensemble"
    ]["xgboost_weight"]
)

CAT_WEIGHT = float(
    main_model_spec[
        "selected_ensemble"
    ]["catboost_weight"]
)


lambda_ensemble = (
    XGB_WEIGHT
    * lambda_xgb
    +
    CAT_WEIGHT
    * lambda_cat
)


# =========================================================
# 8. POISSON PROBABILITIES
# =========================================================

p_0 = np.exp(
    -lambda_ensemble
)

p_1 = (
    lambda_ensemble
    * p_0
)

p_2 = (
    (
        lambda_ensemble ** 2
    )
    / 2.0
    * p_0
)

p_3 = (
    (
        lambda_ensemble ** 3
    )
    / 6.0
    * p_0
)

p_scored = (
    1.0
    - p_0
)

p_multi = (
    1.0
    - p_0
    - p_1
)

p_4_plus = max(
    0.0,
    1.0
    - p_0
    - p_1
    - p_2
    - p_3
)


# =========================================================
# 9. REPORT
# =========================================================

print(
    "\nFIRST PROSPECTIVE FORECAST"
)

print("=" * 100)

print(
    "Forecast cutoff:",
    "2026-08-28"
)

print(
    "Fixture:",
    "2026-09-05",
    first_future["team"],
    "vs",
    first_future["opponent"],
)

print(
    "Starting career goals:",
    978
)


print(
    "\nMODEL EXPECTED GOALS (LAMBDA)"
)

print("-" * 100)

print(
    f"XGBoost λ:  {lambda_xgb:.6f}"
)

print(
    f"CatBoost λ: {lambda_cat:.6f}"
)

print(
    f"Ensemble λ: {lambda_ensemble:.6f}"
)


print(
    "\nPOISSON GOAL PROBABILITIES"
)

print("-" * 100)

print(
    f"P(0 goals):  {p_0:.4%}"
)

print(
    f"P(1 goal):   {p_1:.4%}"
)

print(
    f"P(2 goals):  {p_2:.4%}"
)

print(
    f"P(3 goals):  {p_3:.4%}"
)

print(
    f"P(4+ goals): {p_4_plus:.4%}"
)

print(
    f"P(scores):   {p_scored:.4%}"
)

print(
    f"P(2+ goals): {p_multi:.4%}"
)


# =========================================================
# 10. PRODUCTION CONTRACT CHECKS
# =========================================================

assert len(
    production_df
) == 1333

assert int(
    y.sum()
) == 978

assert len(
    FULL_FEATURES
) == 84

assert len(
    CONTEXT_FEATURES
) == 80

assert abs(
    XGB_WEIGHT
    - 0.60
) < 1e-12

assert abs(
    CAT_WEIGHT
    - 0.40
) < 1e-12

assert np.isfinite(
    lambda_xgb
)

assert np.isfinite(
    lambda_cat
)

assert np.isfinite(
    lambda_ensemble
)

assert (
    lambda_xgb
    > 0
)

assert (
    lambda_cat
    > 0
)

assert (
    lambda_ensemble
    > 0
)

assert abs(
    (
        p_0
        + p_1
        + p_2
        + p_3
        + p_4_plus
    )
    - 1.0
) < 1e-10


print(
    "\nProduction models and first prospective forecast: VALIDATED"
)

PRODUCTION REFIT CONTRACT
Historical rows: 1333
Full features: 84
Context features: 80
Stable categoricals: ['competition_type', 'venue', 'day_of_week']
All categoricals: ['competition_type', 'venue', 'day_of_week', 'season', 'team', 'opponent', 'competition']

XGBOOST DESIGN MATRIX
Raw context shape: (1333, 80)
Transformed shape: (1333, 91)

Locked XGBoost refit: COMPLETE
Locked CatBoost refit: COMPLETE

FIRST PROSPECTIVE FORECAST
Forecast cutoff: 2026-08-28
Fixture: 2026-09-05 Al-Nassr vs Al-Ittihad FC
Starting career goals: 978

MODEL EXPECTED GOALS (LAMBDA)
----------------------------------------------------------------------------------------------------
XGBoost λ:  0.727964
CatBoost λ: 0.812593
Ensemble λ: 0.761815

POISSON GOAL PROBABILITIES
----------------------------------------------------------------------------------------------------
P(0 goals):  46.6818%
P(1 goal):   35.5629%
P(2 goals):  13.5462%
P(3 goals):  3.4399%
P(4+ goals): 0.7692%
P(scores):   53.3182%
P(2+ goal

In [57]:
# Build and validate the recursive state-update engine.
#
# Test scenario only:
# Assume Ronaldo scores exactly 1 goal on 2026-09-05.
#
# This is NOT the real result and is NOT being added to history.
# It is only a deterministic unit test of simulator mechanics.



# =========================================================
# 1. SAFE STATE HELPERS
# =========================================================

def ensure_group_state(
    mapping,
    key,
):
    """
    Ensure a grouped recursive state exists.

    Expected structure:
        {
            "apps": int,
            "goals": float,
            "goal_history": list   # opponent state only
        }
    """

    if key not in mapping:

        mapping[key] = {
            "apps": 0,
            "goals": 0.0,
        }

    return mapping[key]


def ensure_opponent_state(
    mapping,
    opponent,
):

    if opponent not in mapping:

        mapping[opponent] = {
            "apps": 0,
            "goals": 0.0,
            "goal_history": [],
        }

    # Protect against an older state structure
    # that may not yet contain goal_history.
    if (
        "goal_history"
        not in mapping[opponent]
    ):

        mapping[opponent][
            "goal_history"
        ] = []

    return mapping[opponent]


# =========================================================
# 2. RECURSIVE STATE UPDATE
# =========================================================

def update_recursive_state(
    state,
    date,
    season,
    team,
    opponent,
    competition,
    venue,
    simulated_goals,
):
    """
    Advance Ronaldo's recursive state AFTER one simulated
    appearance.

    Only Ronaldo-goal / Ronaldo-appearance dependent state
    is changed.

    External match-result, score, assist and Elo state is
    deliberately NOT changed.
    """

    date = pd.Timestamp(
        date
    )

    goals = int(
        simulated_goals
    )

    if goals < 0:
        raise ValueError(
            "simulated_goals cannot be negative."
        )


    # -----------------------------------------------------
    # Career
    # -----------------------------------------------------

    state[
        "career_appearances"
    ] += 1

    state[
        "career_goal_total"
    ] += goals

    state[
        "career_goals"
    ].append(
        goals
    )


    # -----------------------------------------------------
    # Goal EWMA
    #
    # Historical reconstruction proved alpha = 0.30.
    # -----------------------------------------------------

    state[
        "goal_ewma"
    ] = (
        0.30
        * goals
        +
        0.70
        * state[
            "goal_ewma"
        ]
    )


    # -----------------------------------------------------
    # Scoring / goalless streak
    # -----------------------------------------------------

    if goals > 0:

        state[
            "consecutive_scoring"
        ] += 1

        state[
            "goalless_streak"
        ] = 0

        state[
            "last_goal_date"
        ] = date

    else:

        state[
            "consecutive_scoring"
        ] = 0

        state[
            "goalless_streak"
        ] += 1


    # -----------------------------------------------------
    # Season
    # -----------------------------------------------------

    season_s = ensure_group_state(
        state[
            "season_state"
        ],
        season,
    )

    season_s[
        "apps"
    ] += 1

    season_s[
        "goals"
    ] += goals


    # -----------------------------------------------------
    # Team
    # -----------------------------------------------------

    team_s = ensure_group_state(
        state[
            "team_state"
        ],
        team,
    )

    team_s[
        "apps"
    ] += 1

    team_s[
        "goals"
    ] += goals


    # -----------------------------------------------------
    # Opponent
    # -----------------------------------------------------

    opponent_s = ensure_opponent_state(
        state[
            "opponent_state"
        ],
        opponent,
    )

    opponent_s[
        "apps"
    ] += 1

    opponent_s[
        "goals"
    ] += goals

    opponent_s[
        "goal_history"
    ].append(
        goals
    )


    # -----------------------------------------------------
    # Competition
    # -----------------------------------------------------

    competition_s = ensure_group_state(
        state[
            "competition_state"
        ],
        competition,
    )

    competition_s[
        "apps"
    ] += 1

    competition_s[
        "goals"
    ] += goals


    # -----------------------------------------------------
    # Venue
    # -----------------------------------------------------

    venue_s = ensure_group_state(
        state[
            "venue_state"
        ],
        venue,
    )

    venue_s[
        "apps"
    ] += 1

    venue_s[
        "goals"
    ] += goals


    # -----------------------------------------------------
    # Appearance schedule
    # -----------------------------------------------------

    state[
        "appearance_dates"
    ].append(
        date
    )


    return state


# =========================================================
# 3. CREATE AN ISOLATED TEST PATH
# =========================================================

test_state = deepcopy(
    initial_recursive_state
)


first_fixture = (
    future_fixtures
    .sort_values("date")
    .iloc[0]
)

second_fixture = (
    future_fixtures
    .sort_values("date")
    .iloc[1]
)


print(
    "RECURSIVE UPDATE UNIT TEST"
)

print("=" * 105)

print(
    "Initial:",
    test_state[
        "career_appearances"
    ],
    "appearances /",
    test_state[
        "career_goal_total"
    ],
    "goals"
)

print(
    "Test fixture:",
    pd.Timestamp(
        first_fixture["date"]
    ).date(),
    first_fixture["team"],
    "vs",
    first_fixture["opponent"],
)

print(
    "TEST simulated goals:",
    1
)


# =========================================================
# 4. APPLY TEST UPDATE
# =========================================================

update_recursive_state(
    state=test_state,
    date=first_fixture["date"],
    season="2026/27",
    team=first_fixture["team"],
    opponent=first_fixture["opponent"],
    competition=first_fixture["competition"],
    venue=first_fixture["venue"],
    simulated_goals=1,
)


print(
    "\nPOST-TEST STATE"
)

print("=" * 105)

print(
    "Career appearances:",
    test_state[
        "career_appearances"
    ]
)

print(
    "Career goals:",
    test_state[
        "career_goal_total"
    ]
)

print(
    "Goal EWMA:",
    round(
        test_state[
            "goal_ewma"
        ],
        10,
    )
)

print(
    "Consecutive scoring:",
    test_state[
        "consecutive_scoring"
    ]
)

print(
    "Goalless streak:",
    test_state[
        "goalless_streak"
    ]
)

print(
    "Last goal date:",
    test_state[
        "last_goal_date"
    ].date()
)


# =========================================================
# 5. BUILD SEP-9 PRE-MATCH RECURSIVE FEATURES
# =========================================================

second_recursive = (
    build_recursive_features(
        state=test_state,
        date=second_fixture["date"],
        season="2026/27",
        team=second_fixture["team"],
        opponent=second_fixture["opponent"],
        competition=second_fixture["competition"],
        venue=second_fixture["venue"],
    )
)


second_fixture_known = (
    build_fixture_known_features(
        date=second_fixture["date"],
        season="2026/27",
        team=second_fixture["team"],
        opponent=second_fixture["opponent"],
        competition=second_fixture["competition"],
        venue=second_fixture["venue"],
        prior_appearance_dates=(
            test_state[
                "appearance_dates"
            ]
        ),
    )
)


# =========================================================
# 6. EXPECTED STATE TRANSITIONS
# =========================================================

expected_ewma = (
    0.30 * 1
    +
    0.70 * 0.6128439198
)


checks = {

    "career appearances":
        (
            second_recursive[
                "career_appearances_pre"
            ]
            == 1334
        ),

    "career goals":
        (
            second_recursive[
                "career_goals_pre"
            ]
            == 979
        ),

    "last 3 goals":
        (
            second_recursive[
                "goals_last_3"
            ]
            == 2
        ),

    "last 5 GPG":
        np.isclose(
            second_recursive[
                "goals_per_game_last_5"
            ],
            0.6,
        ),

    "last 10 GPG":
        np.isclose(
            second_recursive[
                "goals_per_game_last_10"
            ],
            0.6,
        ),

    "EWMA":
        np.isclose(
            second_recursive[
                "goals_ewma_pre"
            ],
            expected_ewma,
        ),

    "scoring streak":
        (
            second_recursive[
                "consecutive_scoring_games_pre"
            ]
            == 2
        ),

    "goalless streak":
        (
            second_recursive[
                "goalless_streak_games_pre"
            ]
            == 0
        ),

    "days since goal":
        (
            second_recursive[
                "days_since_last_goal"
            ]
            == 4.0
        ),

    "season appearances":
        (
            second_recursive[
                "season_appearances_pre"
            ]
            == 4
        ),

    "season goals":
        (
            second_recursive[
                "season_goals_pre"
            ]
            == 3
        ),

    "Al-Nassr appearances":
        (
            second_recursive[
                "team_appearances_with_ronaldo_pre"
            ]
            == 152
        ),

    "Al-Nassr Ronaldo goals":
        (
            second_recursive[
                "team_goals_by_ronaldo_pre"
            ]
            == 132
        ),

    "days rest":
        (
            second_fixture_known[
                "days_rest"
            ]
            == 4.0
        ),

    "matches last 7d":
        (
            second_fixture_known[
                "matches_last_7d"
            ]
            == 1
        ),

    "matches last 14d":
        (
            second_fixture_known[
                "matches_last_14d"
            ]
            == 2
        ),
}


# =========================================================
# 7. REPORT
# =========================================================

check_df = pd.DataFrame(
    {
        "check":
            list(
                checks.keys()
            ),

        "passed":
            list(
                checks.values()
            ),
    }
)


print(
    "\nSEP-9 PRE-MATCH TRANSITION CHECK"
)

print("=" * 105)

print(
    check_df.to_string(
        index=False
    )
)


print(
    "\nKEY SEP-9 RECURSIVE VALUES"
)

print("=" * 105)

key_next_features = [
    "career_appearances_pre",
    "career_goals_pre",
    "goals_last_3",
    "goals_per_game_last_5",
    "goals_per_game_last_10",
    "goals_ewma_pre",
    "consecutive_scoring_games_pre",
    "goalless_streak_games_pre",
    "days_since_last_goal",
    "season_appearances_pre",
    "season_goals_pre",
    "team_appearances_with_ronaldo_pre",
    "team_goals_by_ronaldo_pre",
]


print(
    pd.DataFrame(
        {
            feature:
                [
                    second_recursive[
                        feature
                    ]
                ]

            for feature
            in key_next_features
        }
    )
    .T
    .rename(
        columns={
            0: "value"
        }
    )
    .to_string()
)


# =========================================================
# 8. PROTECT ORIGINAL CUTOFF STATE
# =========================================================

assert (
    initial_recursive_state[
        "career_appearances"
    ]
    == 1333
)

assert (
    initial_recursive_state[
        "career_goal_total"
    ]
    == 978
)


# =========================================================
# 9. HARD UNIT-TEST CHECK
# =========================================================

failed_checks = [
    name
    for name, passed
    in checks.items()
    if not passed
]


assert not failed_checks, (
    "Failed recursive transition checks: "
    + str(
        failed_checks
    )
)


print(
    "\nOriginal cutoff state unchanged: YES"
)

print(
    "Recursive state-update engine: VALIDATED"
)

RECURSIVE UPDATE UNIT TEST
Initial: 1333 appearances / 978 goals
Test fixture: 2026-09-05 Al-Nassr vs Al-Ittihad FC
TEST simulated goals: 1

POST-TEST STATE
Career appearances: 1334
Career goals: 979
Goal EWMA: 0.7289907439
Consecutive scoring: 2
Goalless streak: 0
Last goal date: 2026-09-05

SEP-9 PRE-MATCH TRANSITION CHECK
                 check  passed
    career appearances    True
          career goals    True
          last 3 goals    True
            last 5 GPG    True
           last 10 GPG    True
                  EWMA    True
        scoring streak    True
       goalless streak    True
       days since goal    True
    season appearances    True
          season goals    True
  Al-Nassr appearances    True
Al-Nassr Ronaldo goals    True
             days rest    True
       matches last 7d    True
      matches last 14d    True

KEY SEP-9 RECURSIVE VALUES
                                         value
career_appearances_pre             1334.000000
career_goals_pre        

## Forecast engine and compact starting state
Retain only the reusable goal predictor from the earlier probability-tree experiment; do not run the old appearance-assumed Monte Carlo.

In [62]:
def predict_ensemble_lambda(
    state,
    fixture,
):
    fixture_date = pd.Timestamp(
        fixture["date"]
    )

    season = "2026/27"


    recursive_features = (
        build_recursive_features(
            state=state,
            date=fixture_date,
            season=season,
            team=fixture["team"],
            opponent=fixture["opponent"],
            competition=fixture["competition"],
            venue=fixture["venue"],
        )
    )


    fixture_features = (
        build_fixture_known_features(
            date=fixture_date,
            season=season,
            team=fixture["team"],
            opponent=fixture["opponent"],
            competition=fixture["competition"],
            venue=fixture["venue"],
            prior_appearance_dates=(
                state[
                    "appearance_dates"
                ]
            ),
        )
    )


    external_features = (
        build_external_cutoff_features(
            team=fixture["team"],
            opponent=fixture["opponent"],
            season=season,
        )
    )


    full_feature_row = {
        **fixture_features,
        **recursive_features,
        **external_features,
    }


    row_df = pd.DataFrame(
        [full_feature_row]
    )


    # XGBoost
    xgb_raw = (
        row_df[
            CONTEXT_FEATURES
        ]
        .copy()
    )

    xgb_transformed = (
        xgb_preprocessor.transform(
            xgb_raw
        )
    )

    lambda_xgb = float(
        production_xgb.predict(
            xgb_transformed
        )[0]
    )


    # CatBoost
    cat_row = (
        row_df[
            FULL_FEATURES
        ]
        .copy()
    )

    for col in ALL_CATEGORICALS:
        cat_row[col] = (
            cat_row[col]
            .astype(str)
        )


    lambda_cat = float(
        production_catboost.predict(
            cat_row
        )[0]
    )


    lambda_ensemble = (
        0.60 * lambda_xgb
        +
        0.40 * lambda_cat
    )


    return (
        lambda_ensemble,
        lambda_xgb,
        lambda_cat,
    )

# Candidate fixtures; appearance is simulated separately.
simulation_fixtures = (future_fixtures.sort_values(["date", "fixture_id"]).reset_index(drop=True).copy())
simulation_fixtures["date"] = pd.to_datetime(simulation_fixtures["date"])
assert len(simulation_fixtures) == 45 and simulation_fixtures["fixture_id"].is_unique
print("Goal predictor and 45-fixture calendar ready.")

Goal predictor and 45-fixture calendar ready.


In [64]:
# Build a compact future-only recursive state.
#
# Goal:
# Remove historical information that can no longer affect
# any of the 35 recursive future features.
#
# We validate that the compact state produces EXACTLY the
# same recursive + schedule-derived inputs for the first
# future fixture as the full 1,333-match state.


# =========================================================
# 1. FUTURE ENTITIES THAT MUST REMAIN IN STATE
# =========================================================

future_sorted = (
    future_fixtures
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)


future_teams = set(
    future_sorted[
        "team"
    ].unique()
)

future_opponents = set(
    future_sorted[
        "opponent"
    ].unique()
)

future_competitions = set(
    future_sorted[
        "competition"
    ].unique()
)

future_venues = set(
    future_sorted[
        "venue"
    ].unique()
)

future_seasons = {
    "2026/27"
}


print(
    "COMPACT-STATE ENTITY COVERAGE"
)

print("=" * 100)

print(
    "Future teams:",
    len(future_teams),
    sorted(future_teams),
)

print(
    "Future opponents:",
    len(future_opponents),
)

print(
    "Future competitions:",
    len(future_competitions),
    sorted(future_competitions),
)

print(
    "Future venues:",
    len(future_venues),
    sorted(future_venues),
)


# =========================================================
# 2. HELPER — COPY ONLY REQUIRED GROUP STATES
# =========================================================

def compact_group_state(
    original_mapping,
    required_keys,
    opponent_mode=False,
):
    """
    Retain only future-relevant keys.

    Missing entities receive the same zero-history convention
    used by the validated recursive engine.
    """

    compact = defaultdict(
        lambda: (
            {
                "apps": 0,
                "goals": 0.0,
                "goal_history": [],
            }
            if opponent_mode
            else
            {
                "apps": 0,
                "goals": 0.0,
            }
        )
    )


    for key in required_keys:

        if key in original_mapping:

            compact[key] = deepcopy(
                original_mapping[
                    key
                ]
            )

        else:

            if opponent_mode:

                compact[key] = {
                    "apps": 0,
                    "goals": 0.0,
                    "goal_history": [],
                }

            else:

                compact[key] = {
                    "apps": 0,
                    "goals": 0.0,
                }


    return compact


# =========================================================
# 3. CUT DOWN CAREER GOAL HISTORY
#
# Only the last 20 are needed because:
# - last 3
# - last 5
# - last 10
# - last 20
# - scoring-rate windows
# - multi-goal rate last 10
#
# all depend only on these recent outcomes.
# =========================================================

recent_goal_history = (
    initial_recursive_state[
        "career_goals"
    ][-20:]
)


# =========================================================
# 4. CUT DOWN APPEARANCE-DATE HISTORY
#
# Future schedule features require:
# - previous appearance for days_rest
# - appearances within 7/14/30 days
#
# Therefore anything >30 days before the cutoff can never
# affect a future fixture.
# =========================================================

cutoff_date = pd.Timestamp(
    "2026-08-28"
)

appearance_history_start = (
    cutoff_date
    - pd.Timedelta(
        days=30
    )
)


recent_appearance_dates = [

    pd.Timestamp(date)

    for date in initial_recursive_state[
        "appearance_dates"
    ]

    if pd.Timestamp(date)
    >= appearance_history_start
]


# =========================================================
# 5. CONSTRUCT COMPACT STATE
# =========================================================

compact_initial_state = {

    # Career cumulative state
    "career_appearances":
        int(
            initial_recursive_state[
                "career_appearances"
            ]
        ),

    "career_goal_total":
        int(
            initial_recursive_state[
                "career_goal_total"
            ]
        ),

    # Only last 20 outcomes needed going forward
    "career_goals":
        list(
            recent_goal_history
        ),

    # Stateful scoring indicators
    "goal_ewma":
        float(
            initial_recursive_state[
                "goal_ewma"
            ]
        ),

    "consecutive_scoring":
        int(
            initial_recursive_state[
                "consecutive_scoring"
            ]
        ),

    "goalless_streak":
        int(
            initial_recursive_state[
                "goalless_streak"
            ]
        ),

    "last_goal_date":
        pd.Timestamp(
            initial_recursive_state[
                "last_goal_date"
            ]
        ),

    # Only current/future-relevant grouped states
    "season_state":
        compact_group_state(
            initial_recursive_state[
                "season_state"
            ],
            future_seasons,
        ),

    "team_state":
        compact_group_state(
            initial_recursive_state[
                "team_state"
            ],
            future_teams,
        ),

    "opponent_state":
        compact_group_state(
            initial_recursive_state[
                "opponent_state"
            ],
            future_opponents,
            opponent_mode=True,
        ),

    "competition_state":
        compact_group_state(
            initial_recursive_state[
                "competition_state"
            ],
            future_competitions,
        ),

    "venue_state":
        compact_group_state(
            initial_recursive_state[
                "venue_state"
            ],
            future_venues,
        ),

    # Only recent dates needed for rest/congestion features
    "appearance_dates":
        list(
            recent_appearance_dates
        ),
}


# =========================================================
# 6. SIZE COMPARISON
# =========================================================

print(
    "\nFULL VS COMPACT STATE"
)

print("=" * 100)

print(
    "Career goal history:",
    len(
        initial_recursive_state[
            "career_goals"
        ]
    ),
    "->",
    len(
        compact_initial_state[
            "career_goals"
        ]
    ),
)

print(
    "Appearance dates:",
    len(
        initial_recursive_state[
            "appearance_dates"
        ]
    ),
    "->",
    len(
        compact_initial_state[
            "appearance_dates"
        ]
    ),
)

print(
    "Season states:",
    len(
        initial_recursive_state[
            "season_state"
        ]
    ),
    "->",
    len(
        compact_initial_state[
            "season_state"
        ]
    ),
)

print(
    "Team states:",
    len(
        initial_recursive_state[
            "team_state"
        ]
    ),
    "->",
    len(
        compact_initial_state[
            "team_state"
        ]
    ),
)

print(
    "Opponent states:",
    len(
        initial_recursive_state[
            "opponent_state"
        ]
    ),
    "->",
    len(
        compact_initial_state[
            "opponent_state"
        ]
    ),
)

print(
    "Competition states:",
    len(
        initial_recursive_state[
            "competition_state"
        ]
    ),
    "->",
    len(
        compact_initial_state[
            "competition_state"
        ]
    ),
)

print(
    "Venue states:",
    len(
        initial_recursive_state[
            "venue_state"
        ]
    ),
    "->",
    len(
        compact_initial_state[
            "venue_state"
        ]
    ),
)


# =========================================================
# 7. FIRST-FIXTURE RECURSIVE FEATURE EQUIVALENCE
# =========================================================

first_fixture = (
    future_sorted.iloc[0]
)

first_date = pd.Timestamp(
    first_fixture[
        "date"
    ]
)


full_recursive = (
    build_recursive_features(
        state=initial_recursive_state,
        date=first_date,
        season="2026/27",
        team=first_fixture["team"],
        opponent=first_fixture["opponent"],
        competition=first_fixture["competition"],
        venue=first_fixture["venue"],
    )
)


compact_recursive = (
    build_recursive_features(
        state=compact_initial_state,
        date=first_date,
        season="2026/27",
        team=first_fixture["team"],
        opponent=first_fixture["opponent"],
        competition=first_fixture["competition"],
        venue=first_fixture["venue"],
    )
)


recursive_comparison = []

for feature in recursive_contract:

    full_value = (
        full_recursive[
            feature
        ]
    )

    compact_value = (
        compact_recursive[
            feature
        ]
    )


    if (
        isinstance(
            full_value,
            (int, float, np.integer, np.floating),
        )
        and
        isinstance(
            compact_value,
            (int, float, np.integer, np.floating),
        )
    ):

        equal = np.isclose(
            full_value,
            compact_value,
            equal_nan=True,
        )

    else:

        equal = (
            full_value
            == compact_value
        )


    recursive_comparison.append(
        {
            "feature":
                feature,

            "full":
                full_value,

            "compact":
                compact_value,

            "match":
                bool(
                    equal
                ),
        }
    )


recursive_comparison_df = (
    pd.DataFrame(
        recursive_comparison
    )
)


# =========================================================
# 8. FIXTURE-KNOWN EQUIVALENCE
# =========================================================

full_fixture_known = (
    build_fixture_known_features(
        date=first_date,
        season="2026/27",
        team=first_fixture["team"],
        opponent=first_fixture["opponent"],
        competition=first_fixture["competition"],
        venue=first_fixture["venue"],
        prior_appearance_dates=(
            initial_recursive_state[
                "appearance_dates"
            ]
        ),
    )
)


compact_fixture_known = (
    build_fixture_known_features(
        date=first_date,
        season="2026/27",
        team=first_fixture["team"],
        opponent=first_fixture["opponent"],
        competition=first_fixture["competition"],
        venue=first_fixture["venue"],
        prior_appearance_dates=(
            compact_initial_state[
                "appearance_dates"
            ]
        ),
    )
)


fixture_comparison = []

for feature in fixture_known_features:

    full_value = (
        full_fixture_known[
            feature
        ]
    )

    compact_value = (
        compact_fixture_known[
            feature
        ]
    )


    if (
        isinstance(
            full_value,
            (int, float, np.integer, np.floating),
        )
        and
        isinstance(
            compact_value,
            (int, float, np.integer, np.floating),
        )
    ):

        equal = np.isclose(
            full_value,
            compact_value,
            equal_nan=True,
        )

    else:

        equal = (
            full_value
            == compact_value
        )


    fixture_comparison.append(
        {
            "feature":
                feature,

            "full":
                full_value,

            "compact":
                compact_value,

            "match":
                bool(
                    equal
                ),
        }
    )


fixture_comparison_df = (
    pd.DataFrame(
        fixture_comparison
    )
)


# =========================================================
# 9. REPORT DIFFERENCES
# =========================================================

recursive_failures = (
    recursive_comparison_df[
        ~recursive_comparison_df[
            "match"
        ]
    ]
)

fixture_failures = (
    fixture_comparison_df[
        ~fixture_comparison_df[
            "match"
        ]
    ]
)


print(
    "\nCOMPACT-STATE EQUIVALENCE AUDIT"
)

print("=" * 100)

print(
    "Recursive features checked:",
    len(
        recursive_comparison_df
    )
)

print(
    "Recursive mismatches:",
    len(
        recursive_failures
    )
)

print(
    "Fixture-known features checked:",
    len(
        fixture_comparison_df
    )
)

print(
    "Fixture-known mismatches:",
    len(
        fixture_failures
    )
)


if len(
    recursive_failures
) > 0:

    print(
        "\nRECURSIVE FAILURES"
    )

    print(
        recursive_failures.to_string(
            index=False
        )
    )


if len(
    fixture_failures
) > 0:

    print(
        "\nFIXTURE FAILURES"
    )

    print(
        fixture_failures.to_string(
            index=False
        )
    )


# =========================================================
# 10. HARD VALIDATION
# =========================================================

assert len(
    compact_initial_state[
        "career_goals"
    ]
) == 20


assert len(
    recursive_failures
) == 0


assert len(
    fixture_failures
) == 0


assert (
    compact_initial_state[
        "career_appearances"
    ]
    == 1333
)

assert (
    compact_initial_state[
        "career_goal_total"
    ]
    == 978
)


print(
    "\nCompact recursive state reproduces full cutoff state: YES"
)

print(
    "Compact Monte Carlo state: VALIDATED"
)

COMPACT-STATE ENTITY COVERAGE
Future teams: 2 ['Al-Nassr', 'Portugal']
Future opponents: 28
Future competitions: 4 ['AFC Champions League Elite', "King's Cup", 'Pro League', 'UEFA Nations League']
Future venues: 2 ['A', 'H']

FULL VS COMPACT STATE
Career goal history: 1333 -> 20
Appearance dates: 1333 -> 3
Season states: 25 -> 1
Team states: 6 -> 2
Opponent states: 283 -> 28
Competition states: 31 -> 4
Venue states: 3 -> 2

COMPACT-STATE EQUIVALENCE AUDIT
Recursive features checked: 35
Recursive mismatches: 0
Fixture-known features checked: 19
Fixture-known mismatches: 0

Compact recursive state reproduces full cutoff state: YES
Compact Monte Carlo state: VALIDATED


## Load the saved production appearance model
The model was already fitted in the appearance phase. Reconstruct historical predictors as necessary, but do not refit or overwrite model artifacts.

In [66]:
# Refit the locked appearance model on all historical opportunities




DATA_PATH = Path(
    "../data/appearance/ronaldo_appearance_opportunities.csv"
)

SPEC_PATH = Path(
    "../artifacts/appearance_model_spec.json"
)


appearance_df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"],
).sort_values(
    ["date", "team", "opponent"]
).reset_index(drop=True)


with open(SPEC_PATH, "r", encoding="utf-8") as f:
    appearance_spec = json.load(f)


features = appearance_spec["features"]
categorical = appearance_spec["categorical_features"]
target = appearance_spec["target"]

print("APPEARANCE PRODUCTION ARTIFACT — INPUT AUDIT")
print("=" * 65)

print("Historical rows:", len(appearance_df))
print("Appearances:", int(appearance_df[target].sum()))
print("Nonappearances:", int((1 - appearance_df[target]).sum()))
print("Date cutoff:", appearance_df["date"].max().date())

missing_features = [
    feature
    for feature in features
    if feature not in appearance_df.columns
]

print("Missing model features:", missing_features)

assert len(appearance_df) == 1584
assert appearance_df["date"].max() == pd.Timestamp("2026-08-28")
assert not missing_features, (
    "The master opportunity CSV does not contain the engineered features. "
    "We must reconstruct them before fitting."
)



APPEARANCE PRODUCTION ARTIFACT — INPUT AUDIT
Historical rows: 1584
Appearances: 1333
Nonappearances: 251
Date cutoff: 2026-08-28
Missing model features: ['appeared_previous_pre', 'appearance_rate_last_5_pre', 'appearance_rate_last_10_pre', 'consecutive_appearances_pre', 'days_since_last_fixture_pre', 'fixtures_last_7_days_pre', 'fixtures_last_14_days_pre', 'age_years_pre', 'is_national_team', 'fixture_month', 'team_appeared_previous_pre', 'team_appearance_rate_last_5_pre']


AssertionError: The master opportunity CSV does not contain the engineered features. We must reconstruct them before fitting.

In [68]:
# Rebuild the 12 appearance predictors

import numpy as np

appearance_df = appearance_df.sort_values(
    ["date", "team", "opponent"]
).reset_index(drop=True)

dob = pd.Timestamp("1985-02-05")

appearance_df["age_years_pre"] = (
    (appearance_df["date"] - dob).dt.days / 365.2425
)

appearance_df["is_national_team"] = (
    appearance_df["team"].eq("Portugal").astype(int)
)

appearance_df["fixture_month"] = appearance_df["date"].dt.month

# Global history: only fixtures on strictly earlier dates are visible.
previous_outcomes = []
previous_dates = []

for date, indices in appearance_df.groupby(
    "date", sort=True
).groups.items():

    history = np.asarray(previous_outcomes, dtype=int)

    appearance_df.loc[indices, "appeared_previous_pre"] = (
        history[-1] if len(history) else 0
    )

    for window in (5, 10):
        rate = (
            history[-window:].mean()
            if len(history)
            else 0.0
        )

        appearance_df.loc[
            indices, f"appearance_rate_last_{window}_pre"
        ] = rate

    streak = 0

    for outcome in history[::-1]:
        if outcome == 0:
            break
        streak += 1

    appearance_df.loc[
        indices, "consecutive_appearances_pre"
    ] = streak

    appearance_df.loc[
        indices, "days_since_last_fixture_pre"
    ] = (
        (date - previous_dates[-1]).days
        if previous_dates
        else 0
    )

    previous_outcomes.extend(
        appearance_df.loc[indices, "appeared"].tolist()
    )

    previous_dates.extend([date] * len(indices))


# Workload: count fixtures before the current date, never on it.
dates = appearance_df["date"].to_numpy()

for window in (7, 14):
    appearance_df[f"fixtures_last_{window}_days_pre"] = [
        int(
            (
                (dates >= current_date - np.timedelta64(window, "D"))
                & (dates < current_date)
            ).sum()
        )
        for current_date in dates
    ]


# Team-specific history is independent for each club and Portugal.
team_history = appearance_df.groupby("team")["appeared"]

appearance_df["team_appeared_previous_pre"] = (
    team_history.shift(1).fillna(0)
)

appearance_df["team_appearance_rate_last_5_pre"] = (
    appearance_df.groupby("team")["appeared"]
    .transform(
        lambda series:
            series.shift(1).rolling(5, min_periods=1).mean()
    )
    .fillna(0)
)


missing_features = [
    feature
    for feature in features
    if feature not in appearance_df.columns
]

assert not missing_features, missing_features
assert appearance_df[features].notna().all().all()

assert len(appearance_df) == 1584
assert int(appearance_df["appeared"].sum()) == 1333

same_date = appearance_df.loc[
    appearance_df["date"].eq("2017-06-03"),
    [
        "team",
        "appeared",
        "appeared_previous_pre",
        "appearance_rate_last_5_pre",
        "consecutive_appearances_pre",
        "days_since_last_fixture_pre",
    ],
]

assert len(same_date) == 2
assert same_date["appeared_previous_pre"].nunique() == 1
assert same_date["appearance_rate_last_5_pre"].nunique() == 1

print("APPEARANCE FEATURE RECONSTRUCTION")
print("=" * 65)
print("Rows:", len(appearance_df))
print("Model features:", len(features))
print("Missing features:", missing_features)
print("Missing predictor values:", int(appearance_df[features].isna().sum().sum()))

print("\nSame-date leakage check:")
print(same_date.to_string(index=False))

print("\n✓ All 15 model predictors are available.")
print("✓ Same-date fixtures share strictly prior-date history.")
print("✓ Ready for the locked production refit.")

APPEARANCE FEATURE RECONSTRUCTION
Rows: 1584
Model features: 15
Missing features: []
Missing predictor values: 0

Same-date leakage check:
       team  appeared  appeared_previous_pre  appearance_rate_last_5_pre  consecutive_appearances_pre  days_since_last_fixture_pre
   Portugal         0                    1.0                         0.8                          4.0                         13.0
Real Madrid         1                    1.0                         0.8                          4.0                         13.0

✓ All 15 model predictors are available.
✓ Same-date fixtures share strictly prior-date history.
✓ Ready for the locked production refit.


In [70]:
from catboost import CatBoostClassifier
ARTIFACT_DIR = Path("../artifacts")
model_path = ARTIFACT_DIR / "appearance_catboost_production.cbm"
state_path = ARTIFACT_DIR / "appearance_production_state.json"
assert model_path.is_file() and state_path.is_file(), "Saved appearance artifacts are required."
with open(state_path, "r", encoding="utf-8") as f:
    production_state = json.load(f)
appearance_model = CatBoostClassifier()
appearance_model.load_model(str(model_path))
assert appearance_model.is_fitted()
assert production_state["training_rows"] == len(appearance_df) == 1584
assert production_state["appearances"] == int(appearance_df["appeared"].sum()) == 1333
assert production_state["features"] == features
assert production_state["categorical_features"] == categorical
assert production_state["cutoff"] == "2026-08-28"
print("Saved appearance model loaded; no refitting or artifact overwrites.")

Saved appearance model loaded; no refitting or artifact overwrites.


## Inspect the future fixture calendar

In [72]:
# Validate the existing future fixture calendar

calendar = simulation_fixtures.copy()

calendar["date"] = pd.to_datetime(calendar["date"])

calendar = calendar.sort_values(
    ["date", "fixture_id"]
).reset_index(drop=True)

print("SIMULATION CALENDAR AUDIT")
print("=" * 70)

print("Total fixtures:", len(calendar))
print("First fixture:", calendar["date"].min().date())
print("Last fixture:", calendar["date"].max().date())

print("\nFixtures by team:")
print(calendar["team"].value_counts().to_string())

print("\nForecast cutoff values:")
print(calendar["forecast_cutoff"].value_counts(dropna=False).to_string())

print("\nFixture status:")
print(calendar["fixture_status"].value_counts(dropna=False).to_string())

print("\nFirst five fixtures:")
print(
    calendar[
        [
            "fixture_id",
            "date",
            "team",
            "opponent",
            "competition",
            "venue",
        ]
    ].head(5).to_string(index=False)
)

assert len(calendar) == 45
assert calendar["fixture_id"].is_unique
assert calendar["date"].notna().all()
assert calendar["date"].min() > pd.Timestamp("2026-08-28")

print("\n✓ Calendar contains 45 unique future fixtures.")
print("✓ Every fixture occurs after the historical cutoff.")

SIMULATION CALENDAR AUDIT
Total fixtures: 45
First fixture: 2026-09-05
Last fixture: 2027-05-28

Fixtures by team:
team
Al-Nassr    39
Portugal     6

Forecast cutoff values:
forecast_cutoff
2026-08-28    45

Fixture status:
fixture_status
confirmed    45

First five fixtures:
fixture_id       date     team          opponent                competition venue
future_001 2026-09-05 Al-Nassr     Al-Ittihad FC                 Pro League     A
future_002 2026-09-09 Al-Nassr              Abha                 Pro League     H
future_003 2026-09-12 Al-Nassr Al Khaleej Saihat                 Pro League     A
future_004 2026-09-15 Al-Nassr            Al Ain AFC Champions League Elite     A
future_005 2026-09-24 Portugal             Wales        UEFA Nations League     H

✓ Calendar contains 45 unique future fixtures.
✓ Every fixture occurs after the historical cutoff.


In [74]:
# Construct the first future appearance feature row

first_fixture = calendar.iloc[0]
fixture_date = first_fixture["date"]
fixture_team = first_fixture["team"]

history = appearance_df.loc[
    appearance_df["date"] < fixture_date
].sort_values(
    ["date", "team", "opponent"]
)

team_history = history.loc[
    history["team"] == fixture_team
]

outcomes = history["appeared"].to_numpy()
team_outcomes = team_history["appeared"].to_numpy()

streak = 0

for outcome in outcomes[::-1]:
    if outcome == 0:
        break
    streak += 1

appearance_row = {
    "appeared_previous_pre": int(outcomes[-1]),
    "appearance_rate_last_5_pre": outcomes[-5:].mean(),
    "appearance_rate_last_10_pre": outcomes[-10:].mean(),
    "consecutive_appearances_pre": streak,
    "days_since_last_fixture_pre": (
        fixture_date - history["date"].max()
    ).days,
    "fixtures_last_7_days_pre": int(
        (
            (history["date"] >= fixture_date - pd.Timedelta(days=7))
            & (history["date"] < fixture_date)
        ).sum()
    ),
    "fixtures_last_14_days_pre": int(
        (
            (history["date"] >= fixture_date - pd.Timedelta(days=14))
            & (history["date"] < fixture_date)
        ).sum()
    ),
    "age_years_pre": (
        fixture_date - pd.Timestamp("1985-02-05")
    ).days / 365.2425,
    "is_national_team": int(fixture_team == "Portugal"),
    "fixture_month": fixture_date.month,
    "team_appeared_previous_pre": int(team_outcomes[-1]),
    "team_appearance_rate_last_5_pre": team_outcomes[-5:].mean(),
    "team": fixture_team,
    "competition": first_fixture["competition"],
    "venue": first_fixture["venue"],
}

first_appearance_X = pd.DataFrame(
    [appearance_row],
    columns=features,
)

for column in categorical:
    first_appearance_X[column] = (
        first_appearance_X[column].astype(str)
    )

assert first_appearance_X.notna().all().all()
assert first_appearance_X.columns.tolist() == features

print("FIRST FUTURE APPEARANCE FEATURE ROW")
print("=" * 65)

print("Fixture:", first_fixture["fixture_id"])
print("Date:", fixture_date.date())
print("Team:", fixture_team)
print("Opponent:", first_fixture["opponent"])

print("\nPredictors:")
print(first_appearance_X.T.to_string(header=False))

print("\n✓ All 15 appearance predictors constructed.")
print("✓ Only pre-fixture historical outcomes used.")

FIRST FUTURE APPEARANCE FEATURE ROW
Fixture: future_001
Date: 2026-09-05
Team: Al-Nassr
Opponent: Al-Ittihad FC

Predictors:
appeared_previous_pre                     1
appearance_rate_last_5_pre              0.6
appearance_rate_last_10_pre             0.8
consecutive_appearances_pre               3
days_since_last_fixture_pre               8
fixtures_last_7_days_pre                  0
fixtures_last_14_days_pre                 2
age_years_pre                     41.580594
is_national_team                          0
fixture_month                             9
team_appeared_previous_pre                1
team_appearance_rate_last_5_pre         0.6
team                               Al-Nassr
competition                      Pro League
venue                                     A

✓ All 15 appearance predictors constructed.
✓ Only pre-fixture historical outcomes used.


In [76]:
# Predict appearance probability for the first fixture


model_path = Path(
    "../artifacts/appearance_catboost_production.cbm"
)

appearance_model = CatBoostClassifier()
appearance_model.load_model(str(model_path))

catboost_probability = float(
    appearance_model.predict_proba(first_appearance_X)[0, 1]
)

historical_probability = float(
    production_state["historical_appearance_rate"]
)

weights = production_state["blend"]

appearance_probability = (
    weights["catboost_weight"] * catboost_probability
    + weights["historical_rate_weight"] * historical_probability
)

assert np.isfinite(appearance_probability)
assert 0 <= appearance_probability <= 1
assert np.isclose(
    appearance_probability,
    0.5 * catboost_probability + 0.5 * historical_probability,
)

print("FIRST FIXTURE — APPEARANCE PREDICTION")
print("=" * 65)

print("Fixture:", first_fixture["fixture_id"])
print("Date:", fixture_date.date())
print("Match:", fixture_team, "vs", first_fixture["opponent"])

print("\nProbability components")
print("-" * 65)

print(f"CatBoost probability:   {catboost_probability:.4f}")
print(f"Historical probability: {historical_probability:.4f}")
print(f"Final blended probability: {appearance_probability:.4f}")

print("\n✓ Saved production model loaded successfully.")
print("✓ Locked 50/50 blend applied.")
print("✓ Appearance probability is valid.")

FIRST FIXTURE — APPEARANCE PREDICTION
Fixture: future_001
Date: 2026-09-05
Match: Al-Nassr vs Al-Ittihad FC

Probability components
-----------------------------------------------------------------
CatBoost probability:   0.9027
Historical probability: 0.8415
Final blended probability: 0.8721

✓ Saved production model loaded successfully.
✓ Locked 50/50 blend applied.
✓ Appearance probability is valid.


## Create the recursive appearance feature builder

In [78]:
# Build appearance features from a simulation history

def build_appearance_features(fixture, history):
    date = pd.Timestamp(fixture["date"])
    team = fixture["team"]

    past = history.loc[
        history["date"] < date
    ].sort_values(
        ["date", "team", "opponent"]
    )

    team_past = past.loc[past["team"] == team]

    outcomes = past["appeared"].to_numpy()
    team_outcomes = team_past["appeared"].to_numpy()

    streak = 0

    for outcome in outcomes[::-1]:
        if outcome == 0:
            break
        streak += 1

    row = {
        "appeared_previous_pre": int(outcomes[-1]) if len(outcomes) else 0,
        "appearance_rate_last_5_pre": (
            outcomes[-5:].mean() if len(outcomes) else 0.0
        ),
        "appearance_rate_last_10_pre": (
            outcomes[-10:].mean() if len(outcomes) else 0.0
        ),
        "consecutive_appearances_pre": streak,
        "days_since_last_fixture_pre": (
            (date - past["date"].max()).days
            if len(past) else 0
        ),
        "fixtures_last_7_days_pre": int(
            (
                (past["date"] >= date - pd.Timedelta(days=7))
                & (past["date"] < date)
            ).sum()
        ),
        "fixtures_last_14_days_pre": int(
            (
                (past["date"] >= date - pd.Timedelta(days=14))
                & (past["date"] < date)
            ).sum()
        ),
        "age_years_pre": (
            date - pd.Timestamp("1985-02-05")
        ).days / 365.2425,
        "is_national_team": int(team == "Portugal"),
        "fixture_month": date.month,
        "team_appeared_previous_pre": (
            int(team_outcomes[-1]) if len(team_outcomes) else 0
        ),
        "team_appearance_rate_last_5_pre": (
            team_outcomes[-5:].mean()
            if len(team_outcomes)
            else 0.0
        ),
        "team": team,
        "competition": fixture["competition"],
        "venue": fixture["venue"],
    }

    result = pd.DataFrame([row], columns=features)

    for column in categorical:
        result[column] = result[column].astype(str)

    assert result.notna().all().all()

    return result


# Verify that the reusable builder reproduces our first prediction inputs.
reconstructed_X = build_appearance_features(
    first_fixture,
    appearance_df,
)

pd.testing.assert_frame_equal(
    reconstructed_X,
    first_appearance_X,
    check_dtype=False,
)

print("RECURSIVE APPEARANCE FEATURE BUILDER")
print("=" * 65)
print("First fixture:", first_fixture["fixture_id"])
print("Predictors:", len(reconstructed_X.columns))
print("Reference comparison: EXACT MATCH")

print("\n✓ Reusable feature builder validated.")
print("✓ Ready to update appearance history during simulation.")

RECURSIVE APPEARANCE FEATURE BUILDER
First fixture: future_001
Predictors: 15
Reference comparison: EXACT MATCH

✓ Reusable feature builder validated.
✓ Ready to update appearance history during simulation.


In [80]:
# Verify recursive appearance-state updates

second_fixture = calendar.iloc[1]

scenario_results = []

for scenario_name, simulated_appearance in [
    ("Appeared", 1),
    ("Did not appear", 0),
]:
    simulated_fixture = pd.DataFrame(
        [{
            "date": first_fixture["date"],
            "team": first_fixture["team"],
            "opponent": first_fixture["opponent"],
            "appeared": simulated_appearance,
        }]
    )

    scenario_history = pd.concat(
        [
            appearance_df[
                ["date", "team", "opponent", "appeared"]
            ],
            simulated_fixture,
        ],
        ignore_index=True,
    )

    next_X = build_appearance_features(
        second_fixture,
        scenario_history,
    )

    cat_prob = float(
        appearance_model.predict_proba(next_X)[0, 1]
    )

    blended_prob = (
        production_state["blend"]["catboost_weight"] * cat_prob
        + production_state["blend"]["historical_rate_weight"]
        * production_state["historical_appearance_rate"]
    )

    scenario_results.append({
        "scenario": scenario_name,
        "previous_appearance": simulated_appearance,
        "appeared_previous_pre": next_X.iloc[0][
            "appeared_previous_pre"
        ],
        "appearance_rate_last_5_pre": next_X.iloc[0][
            "appearance_rate_last_5_pre"
        ],
        "consecutive_appearances_pre": next_X.iloc[0][
            "consecutive_appearances_pre"
        ],
        "days_since_last_fixture_pre": next_X.iloc[0][
            "days_since_last_fixture_pre"
        ],
        "appearance_probability": blended_prob,
    })

scenario_df = pd.DataFrame(scenario_results)

assert scenario_df["appeared_previous_pre"].tolist() == [1, 0]
assert scenario_df["appearance_probability"].between(0, 1).all()
assert scenario_df["appearance_probability"].notna().all()

print("RECURSIVE APPEARANCE TEST")
print("=" * 75)

print("Previous fixture:", first_fixture["fixture_id"])
print("Next fixture:", second_fixture["fixture_id"])
print("Next fixture date:", second_fixture["date"].date())

print("\nScenario comparison:")
print(
    scenario_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print("\n✓ Simulated appearance outcomes update the next fixture's state.")
print("✓ Both appearance probabilities are valid.")

RECURSIVE APPEARANCE TEST
Previous fixture: future_001
Next fixture: future_002
Next fixture date: 2026-09-09

Scenario comparison:
      scenario  previous_appearance  appeared_previous_pre  appearance_rate_last_5_pre  consecutive_appearances_pre  days_since_last_fixture_pre  appearance_probability
      Appeared                    1                      1                      0.8000                            4                            4                  0.8752
Did not appear                    0                      0                      0.6000                            0                            4                  0.8275

✓ Simulated appearance outcomes update the next fixture's state.
✓ Both appearance probabilities are valid.


## Reference appearance-aware simulation
Retain one detailed reference path to verify that absence produces zero goals and only appearances update the scoring state.

In [82]:
# Appearance-aware Monte Carlo simulation

def simulate_appearance_aware_path(
    rng,
    fixtures,
    starting_state,
    starting_appearance_history,
    target_goal=1000,
):
    goal_state = deepcopy(starting_state)

    appearance_history = starting_appearance_history[
        ["date", "team", "opponent", "appeared"]
    ].copy()

    appearance_history["date"] = pd.to_datetime(
        appearance_history["date"]
    )

    fixtures = fixtures.sort_values(
        ["date", "fixture_id"]
    ).reset_index(drop=True)

    starting_goals = int(goal_state["career_goal_total"])
    starting_apps = int(goal_state["career_appearances"])

    path_rows = []
    milestone_date = pd.NaT
    milestone_fixture_index = np.nan

    for i, fixture in fixtures.iterrows():
        fixture_date = pd.Timestamp(fixture["date"])

        # Appearance prediction
        appearance_X = build_appearance_features(
            fixture,
            appearance_history,
        )

        p_cat = float(
            appearance_model.predict_proba(appearance_X)[0, 1]
        )

        p_appearance = (
            production_state["blend"]["catboost_weight"] * p_cat
            + production_state["blend"]["historical_rate_weight"]
            * production_state["historical_appearance_rate"]
        )

        assert np.isfinite(p_appearance)
        assert 0 <= p_appearance <= 1

        appeared = int(rng.binomial(1, p_appearance))

        career_goals_pre = int(goal_state["career_goal_total"])
        career_apps_pre = int(goal_state["career_appearances"])

        lambda_ensemble = np.nan
        lambda_xgb = np.nan
        lambda_cat = np.nan
        simulated_goals = 0

        # Goals are possible only if Ronaldo appears
        if appeared:
            (
                lambda_ensemble,
                lambda_xgb,
                lambda_cat,
            ) = predict_ensemble_lambda(
                goal_state,
                fixture,
            )

            if (
                not np.isfinite(lambda_ensemble)
                or lambda_ensemble <= 0
            ):
                raise ValueError(
                    f"Invalid goal lambda at fixture {i + 1}"
                )

            simulated_goals = int(
                rng.poisson(lambda_ensemble)
            )

            update_recursive_state(
                state=goal_state,
                date=fixture_date,
                season="2026/27",
                team=fixture["team"],
                opponent=fixture["opponent"],
                competition=fixture["competition"],
                venue=fixture["venue"],
                simulated_goals=simulated_goals,
            )

        # Record every fixture, including missed matches
        appearance_history.loc[
            len(appearance_history)
        ] = {
            "date": fixture_date,
            "team": fixture["team"],
            "opponent": fixture["opponent"],
            "appeared": appeared,
        }

        career_goals_post = int(goal_state["career_goal_total"])
        career_apps_post = int(goal_state["career_appearances"])

        assert career_apps_post - career_apps_pre == appeared
        assert career_goals_post - career_goals_pre == simulated_goals

        path_rows.append({
            "fixture_index": i + 1,
            "fixture_id": fixture["fixture_id"],
            "date": fixture_date,
            "team": fixture["team"],
            "opponent": fixture["opponent"],
            "appearance_probability": p_appearance,
            "appeared": appeared,
            "lambda_xgb": lambda_xgb,
            "lambda_cat": lambda_cat,
            "lambda_ensemble": lambda_ensemble,
            "simulated_goals": simulated_goals,
            "career_appearances_pre": career_apps_pre,
            "career_appearances_post": career_apps_post,
            "career_goals_pre": career_goals_pre,
            "career_goals_post": career_goals_post,
            "goals_remaining_post": max(
                0, target_goal - career_goals_post
            ),
        })

        if career_goals_post >= target_goal:
            milestone_date = fixture_date
            milestone_fixture_index = i + 1
            break

    path_df = pd.DataFrame(path_rows)

    final_goals = int(goal_state["career_goal_total"])
    final_apps = int(goal_state["career_appearances"])

    summary = {
        "starting_goals": starting_goals,
        "final_goals": final_goals,
        "goals_added": final_goals - starting_goals,
        "starting_appearances": starting_apps,
        "final_appearances": final_apps,
        "appearances_added": final_apps - starting_apps,
        "fixtures_used": len(path_df),
        "milestone_reached": final_goals >= target_goal,
        "milestone_date": milestone_date,
        "milestone_fixture_index": milestone_fixture_index,
        "last_simulated_date": (
            path_df["date"].iloc[-1]
            if len(path_df)
            else pd.NaT
        ),
        "goals_remaining": max(0, target_goal - final_goals),
    }

    return path_df, summary


print("✓ Appearance-aware Monte Carlo function created.")
print("✓ Existing goal model and recursive state updater reused.")
print("✓ Missed fixtures update appearance history without adding goals.")

✓ Appearance-aware Monte Carlo function created.
✓ Existing goal model and recursive state updater reused.
✓ Missed fixtures update appearance history without adding goals.


In [84]:
# Test one complete appearance-aware simulation

test_rng = np.random.default_rng(42)

test_path, test_summary = simulate_appearance_aware_path(
    rng=test_rng,
    fixtures=simulation_fixtures,
    starting_state=compact_initial_state,
    starting_appearance_history=appearance_df,
    target_goal=1000,
)

print("APPEARANCE-AWARE SIMULATION AUDIT")
print("=" * 70)

print("\nSummary:")
for key, value in test_summary.items():
    print(f"{key}: {value}")

print("\nFirst five simulated fixtures:")
print(
    test_path[
        [
            "fixture_index",
            "date",
            "opponent",
            "appearance_probability",
            "appeared",
            "simulated_goals",
            "career_goals_post",
        ]
    ].head().to_string(index=False)
)

# Consistency checks
assert len(test_path) <= 45

assert test_path["appeared"].isin([0, 1]).all()

assert (
    test_path.loc[
        test_path["appeared"] == 0,
        "simulated_goals"
    ] == 0
).all()

assert (
    test_path["appeared"].sum()
    == test_summary["appearances_added"]
)

assert (
    test_path["simulated_goals"].sum()
    == test_summary["goals_added"]
)

assert (
    test_summary["final_goals"]
    == test_summary["starting_goals"]
    + test_summary["goals_added"]
)

assert (
    test_summary["final_appearances"]
    == test_summary["starting_appearances"]
    + test_summary["appearances_added"]
)

assert test_path["appearance_probability"].between(0, 1).all()

if test_summary["milestone_reached"]:
    assert test_summary["final_goals"] >= 1000
    assert test_summary["milestone_fixture_index"] == len(test_path)
else:
    assert len(test_path) == 45
    assert test_summary["final_goals"] < 1000

print("\n✓ All simulation consistency checks passed.")

APPEARANCE-AWARE SIMULATION AUDIT

Summary:
starting_goals: 978
final_goals: 1000
goals_added: 22
starting_appearances: 1333
final_appearances: 1356
appearances_added: 23
fixtures_used: 29
milestone_reached: True
milestone_date: 2027-02-11 00:00:00
milestone_fixture_index: 29
last_simulated_date: 2027-02-11 00:00:00
goals_remaining: 0

First five simulated fixtures:
 fixture_index       date          opponent  appearance_probability  appeared  simulated_goals  career_goals_post
             1 2026-09-05     Al-Ittihad FC                0.872108         1                0                978
             2 2026-09-09              Abha                0.875227         1                1                979
             3 2026-09-12 Al Khaleej Saihat                0.869982         0                0                979
             4 2026-09-15            Al Ain                0.672975         0                0                979
             5 2026-09-24             Wales                0.

## Compact appearance history and validated fast simulator

In [86]:
# Compact appearance history


def create_compact_appearance_state(history):
    history = history.sort_values(
        ["date", "team", "opponent"]
    )

    return {
        "outcomes": history["appeared"].astype(int).tolist(),
        "dates": pd.to_datetime(history["date"]).tolist(),
        "team_outcomes": {
            team: group["appeared"].astype(int).tolist()
            for team, group in history.groupby("team")
        },
    }


def build_compact_appearance_features(fixture, state):
    date = pd.Timestamp(fixture["date"])
    team = fixture["team"]

    outcomes = state["outcomes"]
    dates = state["dates"]
    team_outcomes = state["team_outcomes"].get(team, [])

    streak = 0
    for outcome in reversed(outcomes):
        if outcome == 0:
            break
        streak += 1

    row = {
        "appeared_previous_pre": outcomes[-1] if outcomes else 0,
        "appearance_rate_last_5_pre": (
            np.mean(outcomes[-5:]) if outcomes else 0.0
        ),
        "appearance_rate_last_10_pre": (
            np.mean(outcomes[-10:]) if outcomes else 0.0
        ),
        "consecutive_appearances_pre": streak,
        "days_since_last_fixture_pre": (
            (date - dates[-1]).days if dates else 0
        ),
        "fixtures_last_7_days_pre": sum(
            date - pd.Timedelta(days=7) <= d < date
            for d in dates
        ),
        "fixtures_last_14_days_pre": sum(
            date - pd.Timedelta(days=14) <= d < date
            for d in dates
        ),
        "age_years_pre": (
            date - pd.Timestamp("1985-02-05")
        ).days / 365.2425,
        "is_national_team": int(team == "Portugal"),
        "fixture_month": date.month,
        "team_appeared_previous_pre": (
            team_outcomes[-1] if team_outcomes else 0
        ),
        "team_appearance_rate_last_5_pre": (
            np.mean(team_outcomes[-5:])
            if team_outcomes else 0.0
        ),
        "team": team,
        "competition": fixture["competition"],
        "venue": fixture["venue"],
    }

    result = pd.DataFrame([row], columns=features)

    for column in categorical:
        result[column] = result[column].astype(str)

    return result


compact_appearance_state = create_compact_appearance_state(
    appearance_df
)

# Compare against the validated DataFrame-based builder.
reference = build_appearance_features(
    calendar.iloc[0],
    appearance_df,
)

compact = build_compact_appearance_features(
    calendar.iloc[0],
    compact_appearance_state,
)

pd.testing.assert_frame_equal(
    reference,
    compact,
    check_dtype=False,
)

print("COMPACT APPEARANCE STATE")
print("=" * 65)
print("Historical fixtures:", len(compact_appearance_state["dates"]))
print("Teams:", len(compact_appearance_state["team_outcomes"]))
print("Features:", len(compact.columns))
print("Reference comparison: EXACT MATCH")
print("\n✓ Compact appearance feature builder validated.")

COMPACT APPEARANCE STATE
Historical fixtures: 1584
Teams: 6
Features: 15
Reference comparison: EXACT MATCH

✓ Compact appearance feature builder validated.


In [88]:
# Update compact appearance state and verify recursion

def update_compact_appearance_state(state, fixture, appeared):
    date = pd.Timestamp(fixture["date"])
    team = fixture["team"]

    if appeared not in (0, 1):
        raise ValueError("appeared must be 0 or 1")

    state["outcomes"].append(int(appeared))
    state["dates"].append(date)
    state["team_outcomes"].setdefault(team, []).append(int(appeared))


# Compare compact and original builders across both scenarios.
for appeared in (0, 1):
    compact_state = deepcopy(compact_appearance_state)

    update_compact_appearance_state(
        compact_state,
        calendar.iloc[0],
        appeared,
    )

    simulated_record = pd.DataFrame([{
        "date": calendar.iloc[0]["date"],
        "team": calendar.iloc[0]["team"],
        "opponent": calendar.iloc[0]["opponent"],
        "appeared": appeared,
    }])

    reference_history = pd.concat(
        [
            appearance_df[
                ["date", "team", "opponent", "appeared"]
            ],
            simulated_record,
        ],
        ignore_index=True,
    )

    reference_X = build_appearance_features(
        calendar.iloc[1],
        reference_history,
    )

    compact_X = build_compact_appearance_features(
        calendar.iloc[1],
        compact_state,
    )

    pd.testing.assert_frame_equal(
        reference_X,
        compact_X,
        check_dtype=False,
    )

    print(
        f"Previous appearance = {appeared}: "
        "EXACT FEATURE MATCH"
    )

print("\n✓ Compact state updates validated for both scenarios.")
print("✓ Ready to optimize the simulation loop.")

Previous appearance = 0: EXACT FEATURE MATCH
Previous appearance = 1: EXACT FEATURE MATCH

✓ Compact state updates validated for both scenarios.
✓ Ready to optimize the simulation loop.


In [90]:
# Optimize appearance workload calculations

def build_compact_appearance_features(fixture, state):
    date = pd.Timestamp(fixture["date"])
    team = fixture["team"]

    outcomes = state["outcomes"]
    dates = state["dates"]
    team_outcomes = state["team_outcomes"].get(team, [])

    streak = 0
    for outcome in reversed(outcomes):
        if outcome == 0:
            break
        streak += 1

    # Dates are chronological. Stop once we leave the 14-day window.
    fixtures_last_7 = 0
    fixtures_last_14 = 0

    cutoff_7 = date - pd.Timedelta(days=7)
    cutoff_14 = date - pd.Timedelta(days=14)

    for previous_date in reversed(dates):
        if previous_date >= date:
            continue

        if previous_date < cutoff_14:
            break

        fixtures_last_14 += 1

        if previous_date >= cutoff_7:
            fixtures_last_7 += 1

    row = {
        "appeared_previous_pre": outcomes[-1] if outcomes else 0,
        "appearance_rate_last_5_pre": (
            np.mean(outcomes[-5:]) if outcomes else 0.0
        ),
        "appearance_rate_last_10_pre": (
            np.mean(outcomes[-10:]) if outcomes else 0.0
        ),
        "consecutive_appearances_pre": streak,
        "days_since_last_fixture_pre": (
            (date - dates[-1]).days if dates else 0
        ),
        "fixtures_last_7_days_pre": fixtures_last_7,
        "fixtures_last_14_days_pre": fixtures_last_14,
        "age_years_pre": (
            date - pd.Timestamp("1985-02-05")
        ).days / 365.2425,
        "is_national_team": int(team == "Portugal"),
        "fixture_month": date.month,
        "team_appeared_previous_pre": (
            team_outcomes[-1] if team_outcomes else 0
        ),
        "team_appearance_rate_last_5_pre": (
            np.mean(team_outcomes[-5:])
            if team_outcomes else 0.0
        ),
        "team": team,
        "competition": fixture["competition"],
        "venue": fixture["venue"],
    }

    result = pd.DataFrame([row], columns=features)

    for column in categorical:
        result[column] = result[column].astype(str)

    return result


# Validate every future fixture against the original feature builder.
for _, fixture in simulation_fixtures.iterrows():
    reference = build_appearance_features(
        fixture,
        appearance_df,
    )

    optimized = build_compact_appearance_features(
        fixture,
        compact_appearance_state,
    )

    pd.testing.assert_frame_equal(
        reference,
        optimized,
        check_dtype=False,
    )


# Validate recursive updates for both appearance outcomes.
for appeared in (0, 1):
    state = deepcopy(compact_appearance_state)

    update_compact_appearance_state(
        state,
        simulation_fixtures.iloc[0],
        appeared,
    )

    history = pd.concat(
        [
            appearance_df[
                ["date", "team", "opponent", "appeared"]
            ],
            pd.DataFrame([{
                "date": simulation_fixtures.iloc[0]["date"],
                "team": simulation_fixtures.iloc[0]["team"],
                "opponent": simulation_fixtures.iloc[0]["opponent"],
                "appeared": appeared,
            }]),
        ],
        ignore_index=True,
    )

    reference = build_appearance_features(
        simulation_fixtures.iloc[1],
        history,
    )

    optimized = build_compact_appearance_features(
        simulation_fixtures.iloc[1],
        state,
    )

    pd.testing.assert_frame_equal(
        reference,
        optimized,
        check_dtype=False,
    )

print("✓ All 45 fixture feature comparisons passed.")
print("✓ Both recursive scenarios passed.")
print("✓ Historical date scans replaced with a 14-day window.")

✓ All 45 fixture feature comparisons passed.
✓ Both recursive scenarios passed.
✓ Historical date scans replaced with a 14-day window.


In [92]:
# Optimized appearance-aware simulation

def simulate_fast_appearance_path(
    rng,
    fixtures,
    starting_state,
    starting_appearance_state,
    target_goal=1000,
):
    goal_state = deepcopy(starting_state)
    appearance_state = deepcopy(starting_appearance_state)

    starting_goals = int(goal_state["career_goal_total"])
    starting_apps = int(goal_state["career_appearances"])

    milestone_date = pd.NaT
    milestone_fixture_index = np.nan

    fixtures_used = 0
    appearances_added = 0
    goals_added = 0

    for i, fixture in fixtures.iterrows():

        fixture_date = pd.Timestamp(fixture["date"])

        # Predict appearance using compact historical state.
        appearance_X = build_compact_appearance_features(
            fixture,
            appearance_state,
        )

        p_cat = float(
            appearance_model.predict_proba(appearance_X)[0, 1]
        )

        p_appearance = (
            production_state["blend"]["catboost_weight"] * p_cat
            + production_state["blend"]["historical_rate_weight"]
            * production_state["historical_appearance_rate"]
        )

        appeared = int(rng.binomial(1, p_appearance))

        simulated_goals = 0

        if appeared:
            lambda_ensemble, _, _ = predict_ensemble_lambda(
                goal_state,
                fixture,
            )

            simulated_goals = int(
                rng.poisson(lambda_ensemble)
            )

            update_recursive_state(
                state=goal_state,
                date=fixture_date,
                season="2026/27",
                team=fixture["team"],
                opponent=fixture["opponent"],
                competition=fixture["competition"],
                venue=fixture["venue"],
                simulated_goals=simulated_goals,
            )

        # Every fixture updates appearance history.
        update_compact_appearance_state(
            appearance_state,
            fixture,
            appeared,
        )

        fixtures_used += 1
        appearances_added += appeared
        goals_added += simulated_goals

        if goal_state["career_goal_total"] >= target_goal:
            milestone_date = fixture_date
            milestone_fixture_index = i + 1
            break

    final_goals = int(goal_state["career_goal_total"])
    final_apps = int(goal_state["career_appearances"])

    summary = {
        "starting_goals": starting_goals,
        "final_goals": final_goals,
        "goals_added": goals_added,
        "starting_appearances": starting_apps,
        "final_appearances": final_apps,
        "appearances_added": appearances_added,
        "fixtures_used": fixtures_used,
        "milestone_reached": final_goals >= target_goal,
        "milestone_date": milestone_date,
        "milestone_fixture_index": milestone_fixture_index,
        "goals_remaining": max(0, target_goal - final_goals),
    }

    assert final_goals - starting_goals == goals_added
    assert final_apps - starting_apps == appearances_added

    return summary


# Use the same seed to compare against the original simulation.
test_rng = np.random.default_rng(42)

fast_summary = simulate_fast_appearance_path(
    rng=test_rng,
    fixtures=simulation_fixtures,
    starting_state=compact_initial_state,
    starting_appearance_state=compact_appearance_state,
)

print("OPTIMIZED SIMULATION TEST")
print("=" * 65)

for key, value in fast_summary.items():
    print(f"{key}: {value}")

# Compare every shared summary field with Step 12.14.
for key, value in fast_summary.items():
    assert value == test_summary[key], (
        f"Mismatch in {key}: "
        f"optimized={value}, original={test_summary[key]}"
    )

print("\n✓ Optimized simulation matches the original path.")
print("✓ Appearance and goal state updates are consistent.")

OPTIMIZED SIMULATION TEST
starting_goals: 978
final_goals: 1000
goals_added: 22
starting_appearances: 1333
final_appearances: 1356
appearances_added: 23
fixtures_used: 29
milestone_reached: True
milestone_date: 2027-02-11 00:00:00
milestone_fixture_index: 29
goals_remaining: 0

✓ Optimized simulation matches the original path.
✓ Appearance and goal state updates are consistent.


## Direct goal inference (validated against the original predictor)

In [94]:
# Cache static goal features without changing predictions

GOAL_DYNAMIC_FIXTURE_FEATURES = {
    "days_rest",
    "matches_last_7d",
    "matches_last_14d",
    "matches_last_30d",
}

goal_static_cache = {}

for _, fixture in simulation_fixtures.iterrows():
    fixture_date = pd.Timestamp(fixture["date"])

    fixture_features = build_fixture_known_features(
        date=fixture_date,
        season="2026/27",
        team=fixture["team"],
        opponent=fixture["opponent"],
        competition=fixture["competition"],
        venue=fixture["venue"],
        prior_appearance_dates=compact_initial_state["appearance_dates"],
    )

    external_features = build_external_cutoff_features(
        team=fixture["team"],
        opponent=fixture["opponent"],
        season="2026/27",
    )

    static_features = {
        key: value
        for key, value in fixture_features.items()
        if key not in GOAL_DYNAMIC_FIXTURE_FEATURES
    }

    static_features.update(external_features)

    goal_static_cache[fixture["fixture_id"]] = static_features


# Verify that cached values reproduce the original static features.
for _, fixture in simulation_fixtures.iterrows():
    fixture_date = pd.Timestamp(fixture["date"])

    original_fixture = build_fixture_known_features(
        date=fixture_date,
        season="2026/27",
        team=fixture["team"],
        opponent=fixture["opponent"],
        competition=fixture["competition"],
        venue=fixture["venue"],
        prior_appearance_dates=compact_initial_state["appearance_dates"],
    )

    original_external = build_external_cutoff_features(
        team=fixture["team"],
        opponent=fixture["opponent"],
        season="2026/27",
    )

    original_static = {
        key: value
        for key, value in original_fixture.items()
        if key not in GOAL_DYNAMIC_FIXTURE_FEATURES
    }

    original_static.update(original_external)

    assert (
        goal_static_cache[fixture["fixture_id"]]
        == original_static
    )


print("STATIC GOAL FEATURE CACHE")
print("=" * 65)
print("Cached fixtures:", len(goal_static_cache))
print(
    "Static features per fixture:",
    len(next(iter(goal_static_cache.values())))
)
print(
    "Dynamic fixture features excluded:",
    sorted(GOAL_DYNAMIC_FIXTURE_FEATURES)
)

assert len(goal_static_cache) == len(simulation_fixtures)

print("\n✓ Static feature cache validated.")
print("✓ Appearance-dependent features remain dynamic.")

STATIC GOAL FEATURE CACHE
Cached fixtures: 45
Static features per fixture: 45
Dynamic fixture features excluded: ['days_rest', 'matches_last_14d', 'matches_last_30d', 'matches_last_7d']

✓ Static feature cache validated.
✓ Appearance-dependent features remain dynamic.


In [96]:
# Inspect XGBoost preprocessing

print("XGBOOST PREPROCESSOR")
print("=" * 70)

print("Type:", type(xgb_preprocessor).__name__)
print("Input feature count:", len(CONTEXT_FEATURES))
print("Output feature count:", len(
    xgb_preprocessor.get_feature_names_out()
))

print("\nTransformers:")

for name, transformer, columns in xgb_preprocessor.transformers_:
    print(f"\n{name}")
    print("  Type:", type(transformer).__name__)
    print("  Columns:", list(columns) if isinstance(columns, (list, tuple)) else columns)

    if hasattr(transformer, "steps"):
        print("  Pipeline steps:")
        for step_name, step in transformer.steps:
            print(f"    {step_name}: {type(step).__name__}")

print("\nOutput type:")

sample_fixture = simulation_fixtures.iloc[0]

sample_row = pd.DataFrame([{
    **goal_static_cache[sample_fixture["fixture_id"]],
    **{
        "days_rest": calculate_days_rest(
            pd.Timestamp(sample_fixture["date"]),
            compact_initial_state["appearance_dates"],
        ),
        **{
            f"matches_last_{days}d": count_recent_appearances(
                pd.Timestamp(sample_fixture["date"]),
                compact_initial_state["appearance_dates"],
                days,
            )
            for days in (7, 14, 30)
        },
    },
    **build_recursive_features(
        state=compact_initial_state,
        date=pd.Timestamp(sample_fixture["date"]),
        season="2026/27",
        team=sample_fixture["team"],
        opponent=sample_fixture["opponent"],
        competition=sample_fixture["competition"],
        venue=sample_fixture["venue"],
    ),
}])

transformed = xgb_preprocessor.transform(
    sample_row[CONTEXT_FEATURES]
)

print("Type:", type(transformed).__name__)
print("Shape:", transformed.shape)

if hasattr(transformed, "nnz"):
    print("Nonzero values:", transformed.nnz)

print("\n✓ Preprocessor inspection completed.")

XGBOOST PREPROCESSOR
Type: ColumnTransformer
Input feature count: 80
Output feature count: 91

Transformers:

categorical
  Type: OneHotEncoder
  Columns: ['competition_type', 'venue', 'day_of_week']

numeric
  Type: FunctionTransformer
  Columns: ['is_national_team', 'competition_importance', 'age_years', 'career_appearances_pre', 'career_goals_pre', 'career_assists_pre', 'career_goals_per_game_pre', 'days_rest', 'matches_last_7d', 'matches_last_14d', 'matches_last_30d', 'goals_last_3', 'assists_last_5', 'assists_last_10', 'goals_per_game_last_5', 'goals_per_game_last_10', 'goals_per_game_last_20', 'scoring_rate_last_5', 'scoring_rate_last_10', 'multi_goal_rate_last_10', 'goals_ewma_pre', 'assists_ewma_pre', 'consecutive_scoring_games_pre', 'goalless_streak_games_pre', 'days_since_last_goal', 'season_appearances_pre', 'season_goals_pre', 'season_assists_pre', 'season_goals_per_game_pre', 'team_appearances_with_ronaldo_pre', 'team_goals_by_ronaldo_pre', 'team_ronaldo_goals_per_game_pre

In [98]:
# Verify numeric preprocessing before optimizing

numeric_transformer = xgb_preprocessor.named_transformers_["numeric"]

numeric_columns = next(
    columns
    for name, transformer, columns in xgb_preprocessor.transformers_
    if name == "numeric"
)

categorical_transformer = xgb_preprocessor.named_transformers_["categorical"]

categorical_columns = next(
    columns
    for name, transformer, columns in xgb_preprocessor.transformers_
    if name == "categorical"
)

numeric_input = sample_row[numeric_columns].copy()

numeric_output = numeric_transformer.transform(numeric_input)

print("NUMERIC TRANSFORMATION AUDIT")
print("=" * 65)

print("Numeric columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))
print("Numeric transformer:", numeric_transformer)
print("Numeric output type:", type(numeric_output).__name__)
print("Numeric output shape:", numeric_output.shape)

# Check whether the transformation preserves values and ordering.
original_values = numeric_input.to_numpy(dtype=float)
transformed_values = np.asarray(numeric_output, dtype=float)

same_shape = original_values.shape == transformed_values.shape

print("\nShape preserved:", same_shape)

if same_shape:
    np.testing.assert_allclose(
        transformed_values,
        original_values,
        rtol=0,
        atol=0,
        equal_nan=True,
    )

    print("Values and ordering: EXACT MATCH")
    print("✓ Numeric transformation behaves as identity.")
else:
    print("Numeric transformation changes the shape.")
    print("Direct numeric-array construction is not yet validated.")

NUMERIC TRANSFORMATION AUDIT
Numeric columns: 77
Categorical columns: 3
Numeric transformer: FunctionTransformer(accept_sparse=True, check_inverse=False,
                    feature_names_out='one-to-one')
Numeric output type: DataFrame
Numeric output shape: (1, 77)

Shape preserved: True
Values and ordering: EXACT MATCH
✓ Numeric transformation behaves as identity.


In [100]:
# Direct XGBoost preprocessing

encoder = xgb_preprocessor.named_transformers_["categorical"]

numeric_cols = list(numeric_columns)
categorical_cols = list(categorical_columns)

def transform_xgb_direct(feature_row):
    # Use the fitted encoder; preserve its category order and
    # unknown-category handling.
    categorical_values = pd.DataFrame(
        [{col: feature_row[col] for col in categorical_cols}],
        columns=categorical_cols,
    )

    encoded = encoder.transform(categorical_values)

    if hasattr(encoded, "toarray"):
        encoded = encoded.toarray()

    encoded = np.asarray(encoded, dtype=float)

    # Numeric transformer was verified to be an identity operation.
    numeric_values = np.array(
        [[feature_row[col] for col in numeric_cols]],
        dtype=float,
    )

    # ColumnTransformer outputs categorical columns first.
    return np.hstack([encoded, numeric_values])


max_difference = 0.0

for _, fixture in simulation_fixtures.iterrows():
    fixture_date = pd.Timestamp(fixture["date"])

    recursive = build_recursive_features(
        state=compact_initial_state,
        date=fixture_date,
        season="2026/27",
        team=fixture["team"],
        opponent=fixture["opponent"],
        competition=fixture["competition"],
        venue=fixture["venue"],
    )

    appearance_dates = compact_initial_state["appearance_dates"]

    dynamic = {
        "days_rest": calculate_days_rest(
            fixture_date, appearance_dates
        ),
        "matches_last_7d": count_recent_appearances(
            fixture_date, appearance_dates, 7
        ),
        "matches_last_14d": count_recent_appearances(
            fixture_date, appearance_dates, 14
        ),
        "matches_last_30d": count_recent_appearances(
            fixture_date, appearance_dates, 30
        ),
    }

    feature_row = {
        **goal_static_cache[fixture["fixture_id"]],
        **dynamic,
        **recursive,
    }

    original = xgb_preprocessor.transform(
        pd.DataFrame([feature_row])[CONTEXT_FEATURES]
    )

    direct = transform_xgb_direct(feature_row)

    np.testing.assert_allclose(
        direct,
        original,
        rtol=0,
        atol=0,
        equal_nan=True,
    )

    max_difference = max(
        max_difference,
        float(
            np.nanmax(
                np.abs(direct - original)
            )
        ),
    )

print("DIRECT XGBOOST PREPROCESSING VALIDATION")
print("=" * 65)
print("Fixtures compared:", len(simulation_fixtures))
print("Output shape:", direct.shape)
print("Maximum transformation difference:", max_difference)

assert direct.shape[1] == 91

print("\n✓ All 91 transformed features match exactly.")
print("✓ Fitted categorical encoder preserved.")
print("✓ Ready to benchmark direct preprocessing.")

DIRECT XGBOOST PREPROCESSING VALIDATION
Fixtures compared: 45
Output shape: (1, 91)
Maximum transformation difference: 0.0

✓ All 91 transformed features match exactly.
✓ Fitted categorical encoder preserved.
✓ Ready to benchmark direct preprocessing.


In [102]:
# Faster goal predictor with direct preprocessing

def predict_direct_ensemble_lambda(state, fixture):
    fixture_date = pd.Timestamp(fixture["date"])

    recursive = build_recursive_features(
        state=state,
        date=fixture_date,
        season="2026/27",
        team=fixture["team"],
        opponent=fixture["opponent"],
        competition=fixture["competition"],
        venue=fixture["venue"],
    )

    appearance_dates = state["appearance_dates"]

    dynamic = {
        "days_rest": calculate_days_rest(
            fixture_date, appearance_dates
        ),
        "matches_last_7d": count_recent_appearances(
            fixture_date, appearance_dates, 7
        ),
        "matches_last_14d": count_recent_appearances(
            fixture_date, appearance_dates, 14
        ),
        "matches_last_30d": count_recent_appearances(
            fixture_date, appearance_dates, 30
        ),
    }

    feature_row = {
        **goal_static_cache[fixture["fixture_id"]],
        **dynamic,
        **recursive,
    }

    # XGBoost: use our validated direct transformation.
    xgb_input = transform_xgb_direct(feature_row)

    lambda_xgb = float(
        production_xgb.predict(xgb_input)[0]
    )

    # CatBoost: construct only its required columns.
    cat_input = pd.DataFrame(
        [[feature_row[col] for col in FULL_FEATURES]],
        columns=FULL_FEATURES,
    )

    for col in ALL_CATEGORICALS:
        cat_input[col] = cat_input[col].astype(str)

    lambda_cat = float(
        production_catboost.predict(cat_input)[0]
    )

    lambda_ensemble = (
        0.60 * lambda_xgb
        + 0.40 * lambda_cat
    )

    return lambda_ensemble, lambda_xgb, lambda_cat


# Validate against the original predictor.
max_difference = 0.0

for _, fixture in simulation_fixtures.iterrows():
    original = predict_ensemble_lambda(
        compact_initial_state,
        fixture,
    )

    direct = predict_direct_ensemble_lambda(
        compact_initial_state,
        fixture,
    )

    np.testing.assert_allclose(
        direct,
        original,
        rtol=1e-12,
        atol=1e-12,
    )

    max_difference = max(
        max_difference,
        float(
            np.max(
                np.abs(
                    np.asarray(direct)
                    - np.asarray(original)
                )
            )
        ),
    )


print("DIRECT GOAL PREDICTOR VALIDATION")
print("=" * 65)

print("Fixtures compared:", len(simulation_fixtures))
print(f"Maximum prediction difference: {max_difference:.12f}")

print("\n✓ XGBoost predictions match.")
print("✓ CatBoost predictions match.")
print("✓ Ensemble predictions match.")
print("✓ Faster predictor created.")

DIRECT GOAL PREDICTOR VALIDATION
Fixtures compared: 45
Maximum prediction difference: 0.000000000000

✓ XGBoost predictions match.
✓ CatBoost predictions match.
✓ Ensemble predictions match.
✓ Faster predictor created.


In [104]:
# Validate direct predictions throughout a simulation

validation_rng = np.random.default_rng(42)

goal_state = deepcopy(compact_initial_state)
appearance_state = deepcopy(compact_appearance_state)

comparison_rows = []

for _, fixture in simulation_fixtures.iterrows():
    fixture_date = pd.Timestamp(fixture["date"])

    appearance_X = build_compact_appearance_features(
        fixture,
        appearance_state,
    )

    p_cat = float(
        appearance_model.predict_proba(appearance_X)[0, 1]
    )

    p_appearance = (
        production_state["blend"]["catboost_weight"] * p_cat
        + production_state["blend"]["historical_rate_weight"]
        * production_state["historical_appearance_rate"]
    )

    appeared = int(validation_rng.binomial(1, p_appearance))
    simulated_goals = 0

    if appeared:
        original = predict_ensemble_lambda(goal_state, fixture)
        direct = predict_direct_ensemble_lambda(goal_state, fixture)

        np.testing.assert_allclose(
            direct,
            original,
            rtol=1e-12,
            atol=1e-12,
        )

        comparison_rows.append({
            "fixture_id": fixture["fixture_id"],
            "absolute_difference": float(
                np.max(
                    np.abs(np.asarray(direct) - np.asarray(original))
                )
            ),
        })

        simulated_goals = int(validation_rng.poisson(direct[0]))

        update_recursive_state(
            state=goal_state,
            date=fixture_date,
            season="2026/27",
            team=fixture["team"],
            opponent=fixture["opponent"],
            competition=fixture["competition"],
            venue=fixture["venue"],
            simulated_goals=simulated_goals,
        )

    update_compact_appearance_state(
        appearance_state,
        fixture,
        appeared,
    )

    if goal_state["career_goal_total"] >= 1000:
        break

comparison_df = pd.DataFrame(comparison_rows)

assert len(comparison_df) > 0
assert goal_state["career_goal_total"] == test_summary["final_goals"]
assert goal_state["career_appearances"] == test_summary["final_appearances"]

print("DIRECT PREDICTOR RECURSIVE VALIDATION")
print("=" * 65)

print("Appearances compared:", len(comparison_df))
print(
    "Maximum prediction difference:",
    comparison_df["absolute_difference"].max(),
)
print("Final goals:", goal_state["career_goal_total"])
print("Final appearances:", goal_state["career_appearances"])

print("\n✓ All recursive predictions match.")
print("✓ Reference simulation reproduced.")

DIRECT PREDICTOR RECURSIVE VALIDATION
Appearances compared: 23
Maximum prediction difference: 0.0
Final goals: 1000
Final appearances: 1356

✓ All recursive predictions match.
✓ Reference simulation reproduced.


## Forecast-ready engine
Use the validated direct predictor without monkey-patching or multiple competing simulation implementations. The detailed reference remains available for checks.

In [112]:
# The fast simulator looks up this function at runtime.
# Direct predictions were validated against the original for all 45 fixtures
# and through a recursively updated appearance-aware path above.
predict_ensemble_lambda = predict_direct_ensemble_lambda

# Recheck the deterministic reference after selecting the production predictor.
reference_fast = simulate_fast_appearance_path(
    rng=np.random.default_rng(42),
    fixtures=simulation_fixtures,
    starting_state=compact_initial_state,
    starting_appearance_state=compact_appearance_state,
)
print("Reference keys:", sorted(reference_fast.keys()))
print("Test keys:", sorted(test_summary.keys()))

print("\nReference-only keys:")
print(sorted(reference_fast.keys() - test_summary.keys()))

print("\nTest-only keys:")
print(sorted(test_summary.keys() - reference_fast.keys()))
print("Forecast engine ready: appearance-aware, direct goal inference, reference path matched.")

# Validate all fields shared by both simulation engines.
shared_keys = reference_fast.keys() & test_summary.keys()

for key in shared_keys:
    assert reference_fast[key] == test_summary[key], (
        key,
        reference_fast[key],
        test_summary[key],
    )

assert set(test_summary) - set(reference_fast) == {
    "last_simulated_date"
}

print(
    f"Forecast engine VALIDATED: "
    f"{len(shared_keys)} shared fields matched."
)

Reference keys: ['appearances_added', 'final_appearances', 'final_goals', 'fixtures_used', 'goals_added', 'goals_remaining', 'milestone_date', 'milestone_fixture_index', 'milestone_reached', 'starting_appearances', 'starting_goals']
Test keys: ['appearances_added', 'final_appearances', 'final_goals', 'fixtures_used', 'goals_added', 'goals_remaining', 'last_simulated_date', 'milestone_date', 'milestone_fixture_index', 'milestone_reached', 'starting_appearances', 'starting_goals']

Reference-only keys:
[]

Test-only keys:
['last_simulated_date']
Forecast engine ready: appearance-aware, direct goal inference, reference path matched.
Forecast engine VALIDATED: 11 shared fields matched.


## Monte Carlo distribution


In [114]:
from time import perf_counter

N_SIMULATIONS = 100  # Pilot: increase after checking runtime and output.
SEED = 2026
rng = np.random.default_rng(SEED)
start = perf_counter()
forecast_results = [
    simulate_fast_appearance_path(
        rng=rng,
        fixtures=simulation_fixtures,
        starting_state=compact_initial_state,
        starting_appearance_state=compact_appearance_state,
    )
    for _ in range(N_SIMULATIONS)
]
forecast_df = pd.DataFrame(forecast_results)
forecast_df["milestone_date"] = pd.to_datetime(forecast_df["milestone_date"])
completed = forecast_df["milestone_reached"]
assert completed.eq(forecast_df["milestone_date"].notna()).all()
assert (forecast_df["final_goals"] - forecast_df["starting_goals"] == forecast_df["goals_added"]).all()
assert (forecast_df["final_appearances"] - forecast_df["starting_appearances"] == forecast_df["appearances_added"]).all()
print(f"Pilot paths: {len(forecast_df)}; elapsed: {(perf_counter()-start):.1f}s")
print(f"Reached 1000 by horizon: {completed.sum()}/{len(forecast_df)} ({completed.mean():.1%})")
print(f"Unfinished: {(~completed).sum()}")
if completed.any():
    dates = forecast_df.loc[completed, "milestone_date"]
    print("Conditional milestone date quantiles (completed paths only):")
    print(dates.quantile([0.10, 0.50, 0.90]).to_string())
print("Pilot only: increase simulation count and assess appearance sensitivity before reporting a forecast.")

Pilot paths: 100; elapsed: 85.6s
Reached 1000 by horizon: 97/100 (97.0%)
Unfinished: 3
Conditional milestone date quantiles (completed paths only):
0.1   2026-12-03
0.5   2027-02-25
0.9   2027-05-06
Pilot only: increase simulation count and assess appearance sensitivity before reporting a forecast.


In [116]:
from time import perf_counter

N_SIMULATIONS = 1000  # Pilot: increase after checking runtime and output.
SEED = 2026
rng = np.random.default_rng(SEED)
start = perf_counter()
forecast_results = [
    simulate_fast_appearance_path(
        rng=rng,
        fixtures=simulation_fixtures,
        starting_state=compact_initial_state,
        starting_appearance_state=compact_appearance_state,
    )
    for _ in range(N_SIMULATIONS)
]
forecast_df = pd.DataFrame(forecast_results)
forecast_df["milestone_date"] = pd.to_datetime(forecast_df["milestone_date"])
completed = forecast_df["milestone_reached"]
assert completed.eq(forecast_df["milestone_date"].notna()).all()
assert (forecast_df["final_goals"] - forecast_df["starting_goals"] == forecast_df["goals_added"]).all()
assert (forecast_df["final_appearances"] - forecast_df["starting_appearances"] == forecast_df["appearances_added"]).all()
print(f"Pilot paths: {len(forecast_df)}; elapsed: {(perf_counter()-start):.1f}s")
print(f"Reached 1000 by horizon: {completed.sum()}/{len(forecast_df)} ({completed.mean():.1%})")
print(f"Unfinished: {(~completed).sum()}")
if completed.any():
    dates = forecast_df.loc[completed, "milestone_date"]
    print("Conditional milestone date quantiles (completed paths only):")
    print(dates.quantile([0.10, 0.50, 0.90]).to_string())
print("Pilot only: increase simulation count and assess appearance sensitivity before reporting a forecast.")

Pilot paths: 1000; elapsed: 868.4s
Reached 1000 by horizon: 949/1000 (94.9%)
Unfinished: 51
Conditional milestone date quantiles (completed paths only):
0.1   2026-12-03
0.5   2027-03-11
0.9   2027-05-06
Pilot only: increase simulation count and assess appearance sensitivity before reporting a forecast.


## Create the sensitivity simulator

In [119]:
import inspect

# Preserve the validated baseline simulator.
baseline_simulator = simulate_fast_appearance_path

# Create a separate version for sensitivity analysis.
source = inspect.getsource(simulate_fast_appearance_path)

source = source.replace(
    "def simulate_fast_appearance_path(",
    "def simulate_sensitivity_path(",
    1,
)

source = source.replace(
    "    target_goal=1000,\n",
    "    target_goal=1000,\n    appearance_adjustment=0.0,\n",
    1,
)

source = source.replace(
    "        appeared = int(rng.binomial(1, p_appearance))",
    """        p_appearance = float(
            np.clip(p_appearance + appearance_adjustment, 0.0, 1.0)
        )

        appeared = int(rng.binomial(1, p_appearance))""",
    1,
)

assert "def simulate_sensitivity_path(" in source
assert "appearance_adjustment=0.0" in source
assert "np.clip(p_appearance + appearance_adjustment" in source

exec(source)

print("Sensitivity simulator created.")
print("Original baseline simulator preserved.")

Sensitivity simulator created.
Original baseline simulator preserved.


In [121]:
# Verifying the sensitivity simulator
seed = 42

baseline_result = baseline_simulator(
    rng=np.random.default_rng(seed),
    fixtures=simulation_fixtures,
    starting_state=compact_initial_state,
    starting_appearance_state=compact_appearance_state,
)

sensitivity_result = simulate_sensitivity_path(
    rng=np.random.default_rng(seed),
    fixtures=simulation_fixtures,
    starting_state=compact_initial_state,
    starting_appearance_state=compact_appearance_state,
    appearance_adjustment=0.0,
)

assert baseline_result == sensitivity_result, (
    "Zero-adjustment simulation differs from baseline."
)

print("ZERO-ADJUSTMENT VALIDATION: PASSED")
print("Milestone date:", baseline_result["milestone_date"])
print("Final goals:", baseline_result["final_goals"])
print("Appearances added:", baseline_result["appearances_added"])

ZERO-ADJUSTMENT VALIDATION: PASSED
Milestone date: 2027-02-11 00:00:00
Final goals: 1000
Appearances added: 23


In [123]:
# Appearance sensitivity smoke test

scenarios = {
    "Conservative": -0.05,
    "Baseline": 0.0,
    "Optimistic": 0.05,
}

results = []

for name, adjustment in scenarios.items():
    result = simulate_sensitivity_path(
        rng=np.random.default_rng(42),
        fixtures=simulation_fixtures,
        starting_state=compact_initial_state,
        starting_appearance_state=compact_appearance_state,
        appearance_adjustment=adjustment,
    )

    results.append({
        "scenario": name,
        "adjustment": adjustment,
        "appearances": result["appearances_added"],
        "goals": result["goals_added"],
        "milestone_date": result["milestone_date"],
        "reached_1000": result["milestone_reached"],
    })

sensitivity_smoke_test = pd.DataFrame(results)

assert (
    sensitivity_smoke_test.loc[
        sensitivity_smoke_test["scenario"] == "Baseline",
        "milestone_date"
    ].iloc[0] == baseline_result["milestone_date"]
)

print("APPEARANCE SENSITIVITY SMOKE TEST")
print("=" * 80)
print(sensitivity_smoke_test.to_string(index=False))

APPEARANCE SENSITIVITY SMOKE TEST
    scenario  adjustment  appearances  goals milestone_date  reached_1000
Conservative       -0.05           23     22     2027-03-18          True
    Baseline        0.00           23     22     2027-02-11          True
  Optimistic        0.05           23     22     2027-02-11          True


In [125]:
# Run the full appearance-sensitivity experiment

N_SENSITIVITY = 1000

scenarios = {
    "Conservative": -0.05,
    "Baseline": 0.0,
    "Optimistic": 0.05,
}

# Generate the same seeds for all three scenarios.
seeds = np.random.SeedSequence(2027).generate_state(
    N_SENSITIVITY
)

sensitivity_results = []

start = perf_counter()

for scenario, adjustment in scenarios.items():

    scenario_start = perf_counter()

    for seed in seeds:

        result = simulate_sensitivity_path(
            rng=np.random.default_rng(int(seed)),
            fixtures=simulation_fixtures,
            starting_state=compact_initial_state,
            starting_appearance_state=compact_appearance_state,
            appearance_adjustment=adjustment,
        )

        sensitivity_results.append({
            "scenario": scenario,
            "adjustment": adjustment,
            "milestone_reached": result["milestone_reached"],
            "milestone_date": result["milestone_date"],
            "goals_remaining": result["goals_remaining"],
            "appearances_added": result["appearances_added"],
        })

    print(
        f"{scenario}: completed in "
        f"{(perf_counter() - scenario_start) / 60:.2f} minutes"
    )

sensitivity_df = pd.DataFrame(sensitivity_results)

print("\nSENSITIVITY SIMULATIONS COMPLETED")
print("Total paths:", len(sensitivity_df))
print(f"Total runtime: {(perf_counter() - start) / 60:.2f} minutes")

Conservative: completed in 14.72 minutes
Baseline: completed in 10.29 minutes
Optimistic: completed in 8.93 minutes

SENSITIVITY SIMULATIONS COMPLETED
Total paths: 3000
Total runtime: 33.95 minutes


In [127]:
# Summarize appearance sensitivity

sensitivity_df["milestone_date"] = pd.to_datetime(
    sensitivity_df["milestone_date"]
)

summary_rows = []

for scenario, group in sensitivity_df.groupby(
    "scenario",
    sort=False,
):
    completed = group["milestone_reached"].astype(bool)
    milestone_dates = group.loc[completed, "milestone_date"]

    summary_rows.append({
        "scenario": scenario,
        "simulations": len(group),
        "completed": int(completed.sum()),
        "unfinished": int((~completed).sum()),
        "completion_probability": completed.mean(),
        "median_date": milestone_dates.median(),
        "p10_date": milestone_dates.quantile(0.10),
        "p90_date": milestone_dates.quantile(0.90),
        "mean_appearances": group["appearances_added"].mean(),
        "mean_goals_remaining": group["goals_remaining"].mean(),
    })

sensitivity_summary = pd.DataFrame(summary_rows)

assert len(sensitivity_summary) == 3
assert sensitivity_summary["simulations"].eq(1000).all()

print("APPEARANCE SENSITIVITY RESULTS")
print("=" * 110)

print(
    sensitivity_summary.to_string(
        index=False,
        formatters={
            "completion_probability": "{:.1%}".format,
            "mean_appearances": "{:.2f}".format,
            "mean_goals_remaining": "{:.2f}".format,
        },
    )
)

APPEARANCE SENSITIVITY RESULTS
    scenario  simulations  completed  unfinished completion_probability median_date            p10_date   p90_date mean_appearances mean_goals_remaining
Conservative         1000        878         122                  87.8%  2027-04-02 2026-12-10 00:00:00 2027-05-12            26.87                 0.38
    Baseline         1000        942          58                  94.2%  2027-02-25 2026-12-03 12:00:00 2027-05-06            27.07                 0.17
  Optimistic         1000        964          36                  96.4%  2027-02-18 2026-12-01 00:00:00 2027-04-22            27.10                 0.08


## Monthly milestone probabilities

In [130]:
# Monthly cumulative milestone probabilities

monthly_deadlines = pd.to_datetime([
    "2026-09-30",
    "2026-10-31",
    "2026-11-30",
    "2026-12-31",
    "2027-01-31",
    "2027-02-28",
    "2027-03-31",
    "2027-04-30",
    "2027-05-28",
])

monthly_results = []

for scenario, group in sensitivity_df.groupby(
    "scenario",
    sort=False,
):
    milestone_dates = pd.to_datetime(
        group["milestone_date"]
    )

    for deadline in monthly_deadlines:
        reached = (
            milestone_dates.notna()
            & (milestone_dates <= deadline)
        )

        monthly_results.append({
            "scenario": scenario,
            "month": deadline.strftime("%b %Y"),
            "deadline": deadline,
            "completed": int(reached.sum()),
            "probability": reached.mean(),
        })

monthly_df = pd.DataFrame(monthly_results)

monthly_table = monthly_df.pivot(
    index="month",
    columns="scenario",
    values="probability",
)

monthly_table = monthly_table.reindex(
    monthly_deadlines.strftime("%b %Y")
)

monthly_table = monthly_table[
    ["Conservative", "Baseline", "Optimistic"]
]

# Validate cumulative probabilities.
assert monthly_table.notna().all().all()

assert (
    monthly_table.diff().iloc[1:] >= -1e-12
).all().all()

assert np.allclose(
    monthly_table.iloc[-1].to_numpy(),
    sensitivity_summary.set_index("scenario")
    .loc[monthly_table.columns, "completion_probability"]
    .to_numpy(),
)

print("PROBABILITY OF REACHING 1,000 GOALS BY MONTH")
print("=" * 85)

print(
    monthly_table.to_string(
        formatters={
            col: "{:.1%}".format
            for col in monthly_table.columns
        }
    )
)

print("\nMonthly probability analysis: VALIDATED")

PROBABILITY OF REACHING 1,000 GOALS BY MONTH
scenario Conservative Baseline Optimistic
Sep 2026         0.0%     0.0%       0.0%
Oct 2026         0.0%     0.0%       0.1%
Nov 2026         2.8%     6.0%       9.2%
Dec 2026        12.1%    18.9%      30.3%
Jan 2027        12.1%    18.9%      30.3%
Feb 2027        32.1%    47.2%      58.6%
Mar 2027        42.7%    56.6%      69.0%
Apr 2027        74.6%    83.7%      90.6%
May 2027        87.8%    94.2%      96.4%

Monthly probability analysis: VALIDATED


In [132]:
# Analyze simulations that did not reach 1,000

unfinished_df = sensitivity_df.loc[
    ~sensitivity_df["milestone_reached"]
].copy()

unfinished_summary = (
    unfinished_df
    .groupby("scenario")
    .agg(
        unfinished_paths=("goals_remaining", "size"),
        mean_goals_remaining=("goals_remaining", "mean"),
        median_goals_remaining=("goals_remaining", "median"),
        min_goals_remaining=("goals_remaining", "min"),
        max_goals_remaining=("goals_remaining", "max"),
        mean_appearances=("appearances_added", "mean"),
    )
    .reindex(["Conservative", "Baseline", "Optimistic"])
)

print("UNFINISHED SIMULATION ANALYSIS")
print("=" * 100)

print(
    unfinished_summary.to_string(
        formatters={
            "mean_goals_remaining": "{:.2f}".format,
            "median_goals_remaining": "{:.1f}".format,
            "mean_appearances": "{:.2f}".format,
        }
    )
)

print("\nGOALS REMAINING DISTRIBUTION")
print("=" * 100)

remaining_distribution = pd.crosstab(
    unfinished_df["goals_remaining"],
    unfinished_df["scenario"],
).reindex(
    columns=["Conservative", "Baseline", "Optimistic"],
    fill_value=0,
)

print(remaining_distribution.to_string())

assert len(unfinished_df) == 216
assert unfinished_df["goals_remaining"].gt(0).all()

print("\nUnfinished-path analysis: VALIDATED")

UNFINISHED SIMULATION ANALYSIS
              unfinished_paths mean_goals_remaining median_goals_remaining  min_goals_remaining  max_goals_remaining mean_appearances
scenario                                                                                                                             
Conservative               122                 3.14                    2.0                    1                   13            31.93
Baseline                    58                 2.86                    2.5                    1                    9            34.17
Optimistic                  36                 2.25                    2.0                    1                    5            37.75

GOALS REMAINING DISTRIBUTION
scenario         Conservative  Baseline  Optimistic
goals_remaining                                    
1                          37        15          12
2                          26        14          11
3                          20        11           7
4        

## Final forecast results

In [135]:
# Export Monte Carlo forecast results


output_dir = Path("../outputs/forecast")
output_dir.mkdir(parents=True, exist_ok=True)

sensitivity_df.to_csv(
    output_dir / "appearance_sensitivity_paths.csv",
    index=False,
)

sensitivity_summary.to_csv(
    output_dir / "appearance_sensitivity_summary.csv",
    index=False,
)

monthly_df.to_csv(
    output_dir / "monthly_milestone_probabilities.csv",
    index=False,
)

unfinished_summary.to_csv(
    output_dir / "unfinished_paths_summary.csv",
)

remaining_distribution.to_csv(
    output_dir / "unfinished_goals_distribution.csv",
)

print("FORECAST RESULTS EXPORTED")
print("=" * 65)

for path in sorted(output_dir.glob("*.csv")):
    print(f"{path.name}: {path.stat().st_size:,} bytes")

assert len(list(output_dir.glob("*.csv"))) == 5

print("\nAll five forecast files saved successfully.")

FORECAST RESULTS EXPORTED
appearance_sensitivity_paths.csv: 112,147 bytes
appearance_sensitivity_summary.csv: 390 bytes
monthly_milestone_probabilities.csv: 1,154 bytes
unfinished_goals_distribution.csv: 161 bytes
unfinished_paths_summary.csv: 284 bytes

All five forecast files saved successfully.


## Streamlit issues

In [137]:
for name in ["production_xgb", "production_catboost"]:
    model = globals().get(name)

    if model is None:
        print(f"{name}: NOT AVAILABLE IN MEMORY")
    else:
        print(f"{name}: AVAILABLE ({type(model).__name__})")

production_xgb: AVAILABLE (XGBRegressor)
production_catboost: AVAILABLE (CatBoostRegressor)


In [ ]:
artifact_dir = Path("../artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

assert isinstance(production_xgb, XGBRegressor)
assert isinstance(production_catboost, CatBoostRegressor)

xgb_path = artifact_dir / "goal_xgboost_production.json"
catboost_path = artifact_dir / "goal_catboost_production.cbm"

production_xgb.save_model(str(xgb_path))
production_catboost.save_model(str(catboost_path))

# Verify that both saved models can be loaded.
xgb_check = XGBRegressor()
xgb_check.load_model(str(xgb_path))

catboost_check = CatBoostRegressor()
catboost_check.load_model(str(catboost_path))

print("XGBoost:", xgb_path.resolve(), "—", xgb_path.stat().st_size, "bytes")
print("CatBoost:", catboost_path.resolve(), "—", catboost_path.stat().st_size, "bytes")
print("Both production goal models saved and reloaded successfully.")

In [ ]:
import joblib

preprocessor_path = Path("../artifacts/goal_xgboost_preprocessor.joblib")

assert hasattr(xgb_preprocessor, "transformers_")

joblib.dump(xgb_preprocessor, preprocessor_path)

loaded_preprocessor = joblib.load(preprocessor_path)

original = xgb_preprocessor.transform(X_context.iloc[:5])
restored = loaded_preprocessor.transform(X_context.iloc[:5])

np.testing.assert_allclose(original, restored)

assert restored.shape[1] == 91

print("Saved:", preprocessor_path.resolve())
print("Transformed shape:", restored.shape)
print("Preprocessor reload verification: PASSED")

In [144]:
# Validate the prediction service

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from app.services.goal_predictor import GoalPredictor

predictor = GoalPredictor()

result = predictor.predict(first_full_df)

print("Expected goals:", round(result["expected_goals"], 6))
print("P(scores):", f'{result["p_scores"]:.4%}')
print("P(0 goals):", f'{result["p_0"]:.4%}')
print("P(1 goal):", f'{result["p_1"]:.4%}')
print("P(2 goals):", f'{result["p_2"]:.4%}')

assert abs(result["expected_goals"] - 0.761815) < 1e-5
assert abs(result["p_scores"] - 0.533182) < 1e-5

print("Prediction service validation: PASSED")

Expected goals: 0.761815
P(scores): 53.3182%
P(0 goals): 46.6818%
P(1 goal): 35.5629%
P(2 goals): 13.5462%
Prediction service validation: PASSED


In [146]:
print("Future fixture variables:")

for name in [
    "first_future",
    "first_full_df",
    "future_fixtures",
    "fixtures_df",
    "schedule_df",
]:
    value = globals().get(name)

    if value is None:
        print(f"{name}: NOT AVAILABLE")
    else:
        print(f"{name}: {type(value).__name__}, shape={getattr(value, 'shape', 'N/A')}")

print("\nFirst fixture columns:")
print(first_full_df.columns.tolist())

Future fixture variables:
first_future: Series, shape=(11,)
first_full_df: DataFrame, shape=(1, 84)
future_fixtures: DataFrame, shape=(45, 11)
fixtures_df: NOT AVAILABLE
schedule_df: NOT AVAILABLE

First fixture columns:
['season', 'team', 'opponent', 'competition', 'competition_type', 'venue', 'is_national_team', 'competition_importance', 'age_years', 'days_rest', 'matches_last_7d', 'matches_last_14d', 'matches_last_30d', 'month', 'day_of_week', 'season_progress_pct', 'is_season_opening_phase', 'is_season_run_in', 'is_dec_jan_congestion', 'career_appearances_pre', 'career_goals_pre', 'career_goals_per_game_pre', 'no_prior_career_goal_pre', 'first_career_appearance', 'goals_last_3', 'goals_per_game_last_5', 'goals_per_game_last_10', 'goals_per_game_last_20', 'goals_ewma_pre', 'scoring_rate_last_5', 'scoring_rate_last_10', 'multi_goal_rate_last_10', 'consecutive_scoring_games_pre', 'goalless_streak_games_pre', 'days_since_last_goal', 'season_appearances_pre', 'season_goals_pre', 'season

In [ ]:
from pathlib import Path

forecast_dir = Path("../outputs/forecast")
forecast_dir.mkdir(parents=True, exist_ok=True)

schedule_path = forecast_dir / "future_fixtures.csv"

future_fixtures.to_csv(schedule_path, index=False)

print("Saved:", schedule_path.resolve())
print("Fixtures:", len(future_fixtures))
print("Columns:", future_fixtures.columns.tolist())
print("\nFirst three fixtures:")
print(future_fixtures.head(3).to_string(index=False))

In [150]:
# Locate the feature-building function
import inspect

for name, obj in list(globals().items()):
    if not inspect.isfunction(obj):
        continue

    try:
        source = inspect.getsource(obj)
    except (OSError, TypeError):
        continue

    if any(term in source for term in [
        "first_full_df",
        "FULL_FEATURES",
        "CONTEXT_FEATURES",
        "future_fixtures",
    ]):
        print(name)

predict_ensemble_lambda
predict_direct_ensemble_lambda


In [152]:
for name in [
    "predict_ensemble_lambda",
    "predict_direct_ensemble_lambda",
]:
    print(f"\n{'=' * 30} {name} {'=' * 30}")
    print(inspect.getsource(globals()[name]))


============================== predict_ensemble_lambda ==============================
def predict_direct_ensemble_lambda(state, fixture):
    fixture_date = pd.Timestamp(fixture["date"])

    recursive = build_recursive_features(
        state=state,
        date=fixture_date,
        season="2026/27",
        team=fixture["team"],
        opponent=fixture["opponent"],
        competition=fixture["competition"],
        venue=fixture["venue"],
    )

    appearance_dates = state["appearance_dates"]

    dynamic = {
        "days_rest": calculate_days_rest(
            fixture_date, appearance_dates
        ),
        "matches_last_7d": count_recent_appearances(
            fixture_date, appearance_dates, 7
        ),
        "matches_last_14d": count_recent_appearances(
            fixture_date, appearance_dates, 14
        ),
        "matches_last_30d": count_recent_appearances(
            fixture_date, appearance_dates, 30
        ),
    }

    feature_row = {
        **goal_static

In [154]:
# Inspect the recursive feature builder

print(inspect.getsource(build_recursive_features))

def build_recursive_features(
    state,
    date,
    season,
    team,
    opponent,
    competition,
    venue,
):
    """
    Build all 35 Ronaldo-recursive PRE-MATCH features
    from the current simulator state.

    This function does not mutate the state.
    """

    date = pd.Timestamp(date)

    career_history = state[
        "career_goals"
    ]

    career_apps = state[
        "career_appearances"
    ]

    career_goal_total = state[
        "career_goal_total"
    ]


    season_s = state[
        "season_state"
    ][season]

    team_s = state[
        "team_state"
    ][team]

    opponent_s = state[
        "opponent_state"
    ][opponent]

    competition_s = state[
        "competition_state"
    ][competition]

    venue_s = state[
        "venue_state"
    ][venue]


    # -----------------------------------------------------
    # Career
    # -----------------------------------------------------

    features = {

        "career_appearances_pre":
           

In [156]:
# Locate the initial simulation state

for name, obj in list(globals().items()):
    if not inspect.isfunction(obj):
        continue

    try:
        source = inspect.getsource(obj)
    except (OSError, TypeError):
        continue

    if (
        "career_goal_total" in source
        and "season_state" in source
        and "appearance_dates" in source
    ):
        print(name)

update_recursive_state


In [158]:
# find the initialization code

source = inspect.getsource(update_recursive_state)

print(source[:5000])

def update_recursive_state(
    state,
    date,
    season,
    team,
    opponent,
    competition,
    venue,
    simulated_goals,
):
    """
    Advance Ronaldo's recursive state AFTER one simulated
    appearance.

    Only Ronaldo-goal / Ronaldo-appearance dependent state
    is changed.

    External match-result, score, assist and Elo state is
    deliberately NOT changed.
    """

    date = pd.Timestamp(
        date
    )

    goals = int(
        simulated_goals
    )

    if goals < 0:
        raise ValueError(
            "simulated_goals cannot be negative."
        )


    # -----------------------------------------------------
    # Career
    # -----------------------------------------------------

    state[
        "career_appearances"
    ] += 1

    state[
        "career_goal_total"
    ] += goals

    state[
        "career_goals"
    ].append(
        goals
    )


    # -----------------------------------------------------
    # Goal EWMA
    #
    # Historica

In [160]:
# Locate the initial state
for name, obj in list(globals().items()):
    if not inspect.isfunction(obj):
        continue

    try:
        source = inspect.getsource(obj)
    except (OSError, TypeError):
        continue

    if "career_goal_total" in source and "career_goals" in source:
        print(name, "—", len(source), "characters")

build_recursive_features — 7065 characters
update_recursive_state — 3750 characters
simulate_appearance_aware_path — 5038 characters


In [162]:
# Find the initialized state

for name, value in list(globals().items()):
    if not isinstance(value, dict):
        continue

    if {
        "career_goal_total",
        "career_goals",
        "season_state",
        "appearance_dates",
    }.issubset(value.keys()):
        print(
            name,
            "| goals:", value["career_goal_total"],
            "| appearances:", value["career_appearances"],
            "| history length:", len(value["career_goals"]),
        )

initial_recursive_state | goals: 978 | appearances: 1333 | history length: 1333
test_state | goals: 979 | appearances: 1334 | history length: 1334
compact_initial_state | goals: 978 | appearances: 1333 | history length: 20
goal_state | goals: 1000 | appearances: 1356 | history length: 43


In [166]:
import pickle

for key, value in initial_recursive_state.items():
    try:
        pickle.dumps(value)
        print(f"{key}: OK")
    except Exception as error:
        print(f"{key}: FAILED — {type(value).__name__}: {error}")

career_appearances: OK
career_goal_total: OK
career_goals: OK
goal_ewma: OK
consecutive_scoring: OK
goalless_streak: OK
last_goal_date: OK
season_state: FAILED — defaultdict: Can't pickle <function <lambda> at 0x000001A924684D60>: attribute lookup <lambda> on __main__ failed
team_state: FAILED — defaultdict: Can't pickle <function <lambda> at 0x000001A924684AE0>: attribute lookup <lambda> on __main__ failed
opponent_state: FAILED — defaultdict: Can't pickle <function <lambda> at 0x000001A924684E00>: attribute lookup <lambda> on __main__ failed
competition_state: FAILED — defaultdict: Can't pickle <function <lambda> at 0x000001A924684860>: attribute lookup <lambda> on __main__ failed
venue_state: FAILED — defaultdict: Can't pickle <function <lambda> at 0x000001A9246849A0>: attribute lookup <lambda> on __main__ failed
appearance_dates: OK


In [ ]:
from pathlib import Path
from collections import defaultdict

def make_serializable(value):
    if isinstance(value, (dict, defaultdict)):
        return {
            key: make_serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [make_serializable(item) for item in value]

    return value


export_state = make_serializable(initial_recursive_state)

state_path = Path("../artifacts/initial_recursive_state.pkl")

assert export_state["career_goal_total"] == 978
assert export_state["career_appearances"] == 1333
assert len(export_state["career_goals"]) == 1333

with open(state_path, "wb") as file:
    pickle.dump(export_state, file)

with open(state_path, "rb") as file:
    restored_state = pickle.load(file)

assert restored_state["career_goal_total"] == 978
assert restored_state["career_appearances"] == 1333
assert len(restored_state["career_goals"]) == 1333

print("Saved:", state_path.resolve())
print("Career goals:", restored_state["career_goal_total"])
print("Appearances:", restored_state["career_appearances"])
print("Historical records:", len(restored_state["career_goals"]))
print("State export verification: PASSED")

In [170]:


print("Cache type:", type(goal_static_cache).__name__)
print("Cached fixtures:", len(goal_static_cache))

first_id = future_fixtures.iloc[0]["fixture_id"]
first_features = goal_static_cache[first_id]

print("First fixture ID:", first_id)
print("Feature count:", len(first_features))
print("Feature names:", list(first_features.keys()))

missing_ids = (
    set(future_fixtures["fixture_id"])
    - set(goal_static_cache)
)

print("Fixtures missing from cache:", len(missing_ids))

Cache type: dict
Cached fixtures: 45
First fixture ID: future_001
Feature count: 45
Feature names: ['season', 'team', 'opponent', 'competition', 'competition_type', 'venue', 'is_national_team', 'competition_importance', 'age_years', 'month', 'day_of_week', 'season_progress_pct', 'is_season_opening_phase', 'is_season_run_in', 'is_dec_jan_congestion', 'career_assists_pre', 'assists_last_5', 'assists_last_10', 'assists_ewma_pre', 'season_assists_pre', 'ronaldo_assists_vs_opponent_pre', 'team_points_last_5_ronaldo_apps_pre', 'team_points_last_10_ronaldo_apps_pre', 'team_win_rate_last_5_ronaldo_apps_pre', 'team_unbeaten_rate_last_5_ronaldo_apps_pre', 'team_goals_for_last_5_scored_matches_pre', 'team_goals_against_last_5_scored_matches_pre', 'team_score_coverage_last_5_pre', 'no_team_score_history_pre', 'team_h2h_wins_pre', 'team_h2h_draws_pre', 'team_h2h_losses_pre', 'opponent_h2h_points_per_game_pre', 'opponent_h2h_unbeaten_rate_pre', 'opponent_goals_for_per_h2h_scored_match_pre', 'opponen

In [ ]:
# Export the historical cutoff state

cache_path = Path("../artifacts/goal_static_cache.pkl")

assert len(goal_static_cache) == 45
assert set(future_fixtures["fixture_id"]) == set(goal_static_cache)
assert all(len(features) == 45 for features in goal_static_cache.values())

with open(cache_path, "wb") as file:
    pickle.dump(goal_static_cache, file)

with open(cache_path, "rb") as file:
    restored_cache = pickle.load(file)

assert set(restored_cache) == set(goal_static_cache)

print("Saved:", cache_path.resolve())
print("Cached fixtures:", len(restored_cache))
print("Features per fixture:", len(restored_cache["future_001"]))
print("Static cache export verification: PASSED")

In [174]:
# Validate the exported artifacts together

from app.services.goal_predictor import GoalPredictor

artifact_dir = Path("../artifacts")

with open(artifact_dir / "initial_recursive_state.pkl", "rb") as file:
    saved_state = pickle.load(file)

with open(artifact_dir / "goal_static_cache.pkl", "rb") as file:
    saved_cache = pickle.load(file)

fixture = future_fixtures.iloc[0]
fixture_date = pd.Timestamp(fixture["date"])

recursive = build_recursive_features(
    state=saved_state,
    date=fixture_date,
    season="2026/27",
    team=fixture["team"],
    opponent=fixture["opponent"],
    competition=fixture["competition"],
    venue=fixture["venue"],
)

dynamic = {
    "days_rest": calculate_days_rest(
        fixture_date, saved_state["appearance_dates"]
    ),
    "matches_last_7d": count_recent_appearances(
        fixture_date, saved_state["appearance_dates"], 7
    ),
    "matches_last_14d": count_recent_appearances(
        fixture_date, saved_state["appearance_dates"], 14
    ),
    "matches_last_30d": count_recent_appearances(
        fixture_date, saved_state["appearance_dates"], 30
    ),
}

feature_row = {
    **saved_cache[fixture["fixture_id"]],
    **dynamic,
    **recursive,
}

predictor = GoalPredictor()

assert len(feature_row) == 84

fixture_input = pd.DataFrame([feature_row])
result = predictor.predict(fixture_input)

print("Fixture:", fixture["team"], "vs", fixture["opponent"])
print("Feature count:", len(feature_row))
print("Expected goals:", round(result["expected_goals"], 6))
print("P(scores):", f'{result["p_scores"]:.4%}')

assert abs(result["expected_goals"] - 0.761815) < 1e-5
assert abs(result["p_scores"] - 0.533182) < 1e-5

print("Exported artifact integration: PASSED")

Fixture: Al-Nassr vs Al-Ittihad FC
Feature count: 84
Expected goals: 0.761815
P(scores): 53.3182%
Exported artifact integration: PASSED


In [176]:
# Verify the scheduling calculations

from app.services.schedule_features import build_schedule_features

fixture_date = pd.Timestamp(future_fixtures.iloc[0]["date"])
dates = initial_recursive_state["appearance_dates"]

expected = {
    "days_rest": calculate_days_rest(fixture_date, dates),
    "matches_last_7d": count_recent_appearances(fixture_date, dates, 7),
    "matches_last_14d": count_recent_appearances(fixture_date, dates, 14),
    "matches_last_30d": count_recent_appearances(fixture_date, dates, 30),
}

actual = build_schedule_features(fixture_date, dates)

print("Notebook:", expected)
print("App module:", actual)

assert actual == expected, "Scheduling feature mismatch"

print("Scheduling feature validation: PASSED")

Notebook: {'days_rest': 8.0, 'matches_last_7d': 0, 'matches_last_14d': 2, 'matches_last_30d': 3}
App module: {'days_rest': 8.0, 'matches_last_7d': 0, 'matches_last_14d': 2, 'matches_last_30d': 3}
Scheduling feature validation: PASSED


In [178]:

helpers = [
    "safe_rate",
    "rolling_sum",
    "rolling_mean",
    "rolling_scoring_rate",
    "rolling_multi_goal_rate",
]

for name in helpers:
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print(inspect.getsource(globals()[name]))


==================== safe_rate ====================
def safe_rate(numerator, denominator):
    return (
        float(numerator) / float(denominator)
        if denominator > 0
        else 0.0
    )


==================== rolling_sum ====================
def rolling_sum(values, window):
    vals = list(values)[-window:]
    return float(sum(vals))


==================== rolling_mean ====================
def rolling_mean(values, window):
    vals = list(values)[-window:]

    return (
        float(sum(vals)) / len(vals)
        if len(vals) > 0
        else 0.0
    )


==================== rolling_scoring_rate ====================
def rolling_scoring_rate(values, window):
    vals = list(values)[-window:]

    return (
        float(
            sum(
                g > 0
                for g in vals
            )
        )
        / len(vals)
        if len(vals) > 0
        else 0.0
    )


==================== rolling_multi_goal_rate ====================
def rolling_multi_goal_ra

In [180]:
# Validate recursive features

from app.services.recursive_features import (
    build_recursive_features as app_build_recursive_features,
)

fixture = future_fixtures.iloc[0]

kwargs = {
    "state": initial_recursive_state,
    "date": fixture["date"],
    "season": "2026/27",
    "team": fixture["team"],
    "opponent": fixture["opponent"],
    "competition": fixture["competition"],
    "venue": fixture["venue"],
}

expected = build_recursive_features(**kwargs)
actual = app_build_recursive_features(**kwargs)

assert len(expected) == len(actual) == 35
assert set(expected) == set(actual)

for feature in expected:
    assert expected[feature] == actual[feature], (
        f"Mismatch in {feature}: "
        f"notebook={expected[feature]}, app={actual[feature]}"
    )

print("Fixture:", fixture["team"], "vs", fixture["opponent"])
print("Recursive features:", len(actual))
print("Career goals:", actual["career_goals_pre"])
print("Career appearances:", actual["career_appearances_pre"])
print("Recursive feature validation: PASSED")

Fixture: Al-Nassr vs Al-Ittihad FC
Recursive features: 35
Career goals: 978
Career appearances: 1333
Recursive feature validation: PASSED


In [182]:


from app.services.fixture_features import (
    load_fixture_artifacts,
    build_fixture_features,
)
from app.services.goal_predictor import GoalPredictor

# Load the exported historical state and static cache.
state, static_cache = load_fixture_artifacts()

# Use the first fixture only.
fixture = future_fixtures.iloc[0]

# Assemble all 84 features using the app module.
fixture_input = build_fixture_features(
    fixture=fixture,
    state=state,
    static_cache=static_cache,
)

predictor = GoalPredictor()
result = predictor.predict(fixture_input)

print("Fixture:", fixture["team"], "vs", fixture["opponent"])
print("Input shape:", fixture_input.shape)
print("Expected goals:", round(result["expected_goals"], 6))
print("P(scores):", f'{result["p_scores"]:.4%}')

assert fixture_input.shape == (1, 84)
assert fixture_input.columns.tolist() == predictor.full_features

np.testing.assert_allclose(
    result["expected_goals"],
    0.761815,
    atol=1e-5,
)

np.testing.assert_allclose(
    result["p_scores"],
    0.533182,
    atol=1e-5,
)

print("Fixture feature assembler validation: PASSED")

Fixture: Al-Nassr vs Al-Ittihad FC
Input shape: (1, 84)
Expected goals: 0.761815
P(scores): 53.3182%
Fixture feature assembler validation: PASSED


In [184]:
# Inspect the saved simulation paths

forecast_dir = Path("../outputs/forecast")

for filename in [
    "appearance_sensitivity_paths.csv",
    "unfinished_paths_summary.csv",
]:
    path = forecast_dir / filename

    if not path.exists():
        print(f"\n{filename}: NOT FOUND")
        continue

    data = pd.read_csv(path)

    print(f"\n{'=' * 60}")
    print(filename)
    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())
    print("\nFirst two rows:")
    print(data.head(2).to_string(index=False))


appearance_sensitivity_paths.csv
Shape: (3000, 6)
Columns: ['scenario', 'adjustment', 'milestone_reached', 'milestone_date', 'goals_remaining', 'appearances_added']

First two rows:
    scenario  adjustment  milestone_reached milestone_date  goals_remaining  appearances_added
Conservative       -0.05               True     2027-04-29                0                 29
Conservative       -0.05               True     2027-04-02                0                 28

unfinished_paths_summary.csv
Shape: (3, 7)
Columns: ['scenario', 'unfinished_paths', 'mean_goals_remaining', 'median_goals_remaining', 'min_goals_remaining', 'max_goals_remaining', 'mean_appearances']

First two rows:
    scenario  unfinished_paths  mean_goals_remaining  median_goals_remaining  min_goals_remaining  max_goals_remaining  mean_appearances
Conservative               122              3.139344                     2.0                    1                   13         31.926230
    Baseline                58         

In [186]:
# Search for fixture-level results in memory

for name, value in list(globals().items()):
    if not isinstance(value, pd.DataFrame):
        continue

    columns = set(value.columns)

    fixture_columns = {
        "fixture_id", "date", "opponent", "simulated_goals",
        "lambda_ensemble", "expected_goals", "appeared",
    }

    matches = columns & fixture_columns

    if len(matches) >= 2:
        print(
            f"{name}: shape={value.shape}, "
            f"matching columns={sorted(matches)}"
        )

df: shape=(1333, 100), matching columns=['date', 'opponent']
future_fixtures: shape=(45, 11), matching columns=['date', 'fixture_id', 'opponent']
opponent_history: shape=(9, 100), matching columns=['date', 'opponent']
fixture_coverage: shape=(45, 13), matching columns=['date', 'fixture_id', 'opponent']
unseen_fixtures: shape=(27, 12), matching columns=['date', 'fixture_id', 'opponent']
first_meetings: shape=(283, 100), matching columns=['date', 'opponent']
elo_exceptions: shape=(2, 7), matching columns=['date', 'opponent']
team_history: shape=(179, 21), matching columns=['appeared', 'date', 'opponent']
history: shape=(1585, 4), matching columns=['appeared', 'date', 'opponent']
season_rows: shape=(3, 100), matching columns=['date', 'opponent']
hist: shape=(1333, 100), matching columns=['date', 'opponent']
days_mismatch: shape=(5, 10), matching columns=['date', 'opponent']
opening_rows: shape=(15, 10), matching columns=['date', 'opponent']
production_df: shape=(1333, 100), matching colum

In [188]:
# Inspect the simulation records

print("TEST PATH")
print("=" * 60)
print("Shape:", test_path.shape)
print("Columns:", test_path.columns.tolist())
print(test_path.head(5).to_string(index=False))

print("\nPOTENTIAL PATH COLLECTIONS")
print("=" * 60)

for name, value in list(globals().items()):
    if isinstance(value, (list, tuple, dict)):
        if any(term in name.lower() for term in [
            "path", "simulation", "record", "result"
        ]):
            print(
                name,
                "| type:", type(value).__name__,
                "| length:", len(value),
            )

TEST PATH
Shape: (29, 16)
Columns: ['fixture_index', 'fixture_id', 'date', 'team', 'opponent', 'appearance_probability', 'appeared', 'lambda_xgb', 'lambda_cat', 'lambda_ensemble', 'simulated_goals', 'career_appearances_pre', 'career_appearances_post', 'career_goals_pre', 'career_goals_post', 'goals_remaining_post']
 fixture_index fixture_id       date     team          opponent  appearance_probability  appeared  lambda_xgb  lambda_cat  lambda_ensemble  simulated_goals  career_appearances_pre  career_appearances_post  career_goals_pre  career_goals_post  goals_remaining_post
             1 future_001 2026-09-05 Al-Nassr     Al-Ittihad FC                0.872108         1    0.727964    0.812593         0.761815                0                    1333                     1334               978                978                    22
             2 future_002 2026-09-09 Al-Nassr              Abha                0.875227         1    0.791767    1.002590         0.876096                1

In [190]:
# Inspect the stored result structures

for name in ["forecast_results", "sensitivity_results"]:
    collection = globals()[name]

    print(f"\n{'=' * 60}")
    print(name)
    print("Number of entries:", len(collection))

    first = collection[0]
    print("First entry type:", type(first).__name__)

    if isinstance(first, dict):
        for key, value in first.items():
            print(
                f"{key}: {type(value).__name__}, "
                f"shape={getattr(value, 'shape', 'N/A')}, "
                f"length={len(value) if hasattr(value, '__len__') else 'N/A'}"
            )
    else:
        print("First entry:", str(first)[:1000])


forecast_results
Number of entries: 1000
First entry type: dict
starting_goals: int, shape=N/A, length=N/A
final_goals: int, shape=N/A, length=N/A
goals_added: int, shape=N/A, length=N/A
starting_appearances: int, shape=N/A, length=N/A
final_appearances: int, shape=N/A, length=N/A
appearances_added: int, shape=N/A, length=N/A
fixtures_used: int, shape=N/A, length=N/A
milestone_reached: bool, shape=N/A, length=N/A
milestone_date: Timestamp, shape=N/A, length=N/A
milestone_fixture_index: int, shape=N/A, length=N/A
goals_remaining: int, shape=N/A, length=N/A

sensitivity_results
Number of entries: 3000
First entry type: dict
scenario: str, shape=N/A, length=12
adjustment: float, shape=N/A, length=N/A
milestone_reached: bool, shape=N/A, length=N/A
milestone_date: Timestamp, shape=N/A, length=N/A
goals_remaining: int, shape=N/A, length=N/A
appearances_added: int, shape=N/A, length=N/A


In [192]:
# Check the milestone fixture indices

results_df = pd.DataFrame(forecast_results)

print("Total simulations:", len(results_df))
print("Milestone reached:", results_df["milestone_reached"].sum())

print("\nMilestone fixture index:")
print(results_df["milestone_fixture_index"].describe())

print("\nFirst 10 milestone indices:")
print(
    results_df["milestone_fixture_index"]
    .value_counts()
    .sort_index()
    .head(10)
)

print("\nFixture index range:")
print(
    results_df["milestone_fixture_index"].min(),
    results_df["milestone_fixture_index"].max()
)

Total simulations: 1000
Milestone reached: 949

Milestone fixture index:
count    949.000000
mean      32.700738
std        6.195225
min       14.000000
25%       29.000000
50%       33.000000
75%       37.000000
max       45.000000
Name: milestone_fixture_index, dtype: float64

First 10 milestone indices:
milestone_fixture_index
14.0     1
16.0     2
17.0     1
18.0    10
19.0     5
20.0     9
21.0    15
22.0    14
23.0    19
24.0    24
Name: count, dtype: int64

Fixture index range:
14.0 45.0


In [194]:
# Build and save the fixture milestone forecast

fixtures = future_fixtures.copy().reset_index(drop=True)
fixtures["fixture_index"] = range(1, len(fixtures) + 1)

results = pd.DataFrame(forecast_results)
reached = results.loc[
    results["milestone_reached"],
    "milestone_fixture_index"
].astype(int)

assert len(fixtures) == 45
assert reached.between(1, len(fixtures)).all()

counts = reached.value_counts()

fixtures["milestone_count"] = (
    fixtures["fixture_index"].map(counts).fillna(0).astype(int)
)

fixtures["milestone_probability"] = (
    fixtures["milestone_count"] / len(results)
)

fixtures["cumulative_milestone_probability"] = (
    fixtures["milestone_probability"].cumsum()
)

output = Path("../outputs/forecast/fixture_milestone_probabilities.csv")
fixtures.to_csv(output, index=False)

print("Saved:", output)
print("Rows:", len(fixtures))
print("Total milestone count:", fixtures["milestone_count"].sum())
print("Final cumulative probability:",
      fixtures["cumulative_milestone_probability"].iloc[-1])

print(
    fixtures[
        [
            "fixture_index",
            "date",
            "opponent",
            "milestone_count",
            "milestone_probability",
            "cumulative_milestone_probability",
        ]
    ].head(5).to_string(index=False)
)

Saved: ..\outputs\forecast\fixture_milestone_probabilities.csv
Rows: 45
Total milestone count: 949
Final cumulative probability: 0.9490000000000003
 fixture_index       date          opponent  milestone_count  milestone_probability  cumulative_milestone_probability
             1 2026-09-05     Al-Ittihad FC                0                    0.0                               0.0
             2 2026-09-09              Abha                0                    0.0                               0.0
             3 2026-09-12 Al Khaleej Saihat                0                    0.0                               0.0
             4 2026-09-15            Al Ain                0                    0.0                               0.0
             5 2026-09-24             Wales                0                    0.0                               0.0


In [196]:
#Checking two baseline sources
summary = pd.read_csv(
    "../outputs/forecast/appearance_sensitivity_summary.csv"
)

baseline_summary = summary.loc[
    summary["scenario"] == "Baseline"
].iloc[0]

print("Appearance sensitivity summary")
print("Completed:", baseline_summary["completed"])
print("Simulations:", baseline_summary["simulations"])
print(
    "Completion probability:",
    baseline_summary["completion_probability"]
)

print("\nCurrent forecast_results")
print(
    "Completed:",
    sum(r["milestone_reached"] for r in forecast_results)
)
print("Simulations:", len(forecast_results))
print(
    "Completion probability:",
    sum(r["milestone_reached"] for r in forecast_results)
    / len(forecast_results)
)

Appearance sensitivity summary
Completed: 942
Simulations: 1000
Completion probability: 0.942

Current forecast_results
Completed: 949
Simulations: 1000
Completion probability: 0.949


In [198]:
# Inspect how both result sets were generated


print("forecast_results related variables")
print("=" * 60)

for name in [
    "N_SIMULATIONS",
    "RANDOM_SEED",
    "BASE_SEED",
    "forecast_results",
]:
    if name in globals():
        value = globals()[name]

        if name == "forecast_results":
            print(name, "length:", len(value))
        else:
            print(name, "=", value)


print("\nSensitivity baseline records")
print("=" * 60)

baseline_sensitivity = [
    r for r in sensitivity_results
    if r["scenario"] == "Baseline"
]

print("Records:", len(baseline_sensitivity))
print(
    "Completed:",
    sum(r["milestone_reached"] for r in baseline_sensitivity)
)


print("\nSimulation-related functions")
print("=" * 60)

for name in [
    "simulate_appearance_aware_path",
    "run_forecast",
    "run_appearance_sensitivity",
]:
    if name in globals() and callable(globals()[name]):
        print(f"\n--- {name} ---")
        try:
            print(inspect.getsource(globals()[name]))
        except Exception as e:
            print("Could not inspect source:", e)

forecast_results related variables
N_SIMULATIONS = 1000
forecast_results length: 1000

Sensitivity baseline records
Records: 1000
Completed: 942

Simulation-related functions

--- simulate_appearance_aware_path ---
def simulate_appearance_aware_path(
    rng,
    fixtures,
    starting_state,
    starting_appearance_history,
    target_goal=1000,
):
    goal_state = deepcopy(starting_state)

    appearance_history = starting_appearance_history[
        ["date", "team", "opponent", "appeared"]
    ].copy()

    appearance_history["date"] = pd.to_datetime(
        appearance_history["date"]
    )

    fixtures = fixtures.sort_values(
        ["date", "fixture_id"]
    ).reset_index(drop=True)

    starting_goals = int(goal_state["career_goal_total"])
    starting_apps = int(goal_state["career_appearances"])

    path_rows = []
    milestone_date = pd.NaT
    milestone_fixture_index = np.nan

    for i, fixture in fixtures.iterrows():
        fixture_date = pd.Timestamp(fixture["date"])

